In [ ]:

# ==========================================================
# CELL 01 | KHOI TAO DU AN NIDS TREN GOOGLE COLAB
# ==========================================================

from google.colab import drive
from pathlib import Path
import torch
import shutil

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/NIDS_Project")

DATA_DIR = ROOT / "datasets"
MODEL_DIR = ROOT / "models"
RESULT_DIR = ROOT / "results"

IDS17_DIR = DATA_DIR / "CIC-IDS2017"
DDOS19_DIR = DATA_DIR / "CIC-DDoS2019"

for folder in [
    IDS17_DIR,
    DDOS19_DIR,
    MODEL_DIR / "CIC-IDS2017",
    MODEL_DIR / "CIC-DDoS2019",
    RESULT_DIR / "CIC-IDS2017",
    RESULT_DIR / "CIC-DDoS2019"
]:
    folder.mkdir(parents=True, exist_ok=True)

# Thu muc tam tren o dia Colab
WORK_DIR = Path("/content/NIDS_Work")
WORK_DIR.mkdir(exist_ok=True)

print("Project:", ROOT)
print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

disk = shutil.disk_usage("/content")
print(f"Disk free: {disk.free / (1024**3):.2f} GB")

print("CELL 01 COMPLETED")


Mounted at /content/drive
Project: /content/drive/MyDrive/NIDS_Project
GPU: False
Disk free: 107.61 GB
CELL 01 COMPLETED


In [ ]:

# ==========================================================
# CELL 02 | DOC CIC-IDS2017 TU GOOGLE DRIVE
# ==========================================================

import zipfile
import pandas as pd

zip_path = IDS17_DIR / "CIC-IDS2017.zip"
extract_dir = WORK_DIR / "CIC-IDS2017"

assert zip_path.exists(), (
    f"Khong tim thay file: {zip_path}"
)

extract_dir.mkdir(parents=True, exist_ok=True)

# Chi giai nen cac file CSV
with zipfile.ZipFile(zip_path, "r") as zf:
    members = [
        m for m in zf.infolist()
        if m.filename.lower().endswith(".csv")
        and not m.is_dir()
    ]
    print("So file CSV trong ZIP:", len(members))
    zf.extractall(extract_dir, members=members)

csv_files_ids17 = sorted(
    extract_dir.rglob("*.csv")
)

print("\nDANH SACH FILE:")
for path in csv_files_ids17:
    print(
        path.name,
        f"({path.stat().st_size / 1024**2:.1f} MB)"
    )

assert csv_files_ids17, "Khong tim thay CSV!"

# Doc header, khong load toan bo dataset
df = pd.read_csv(
    csv_files_ids17[0],
    nrows=5,
    low_memory=False
)

df.columns = df.columns.str.strip()

print("\nTong so cot:", len(df.columns))
print("\nDANH SACH COT:")

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nKIEM TRA METADATA:")

for col in [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Source Port",
    "Destination Port",
    "Label"
]:
    print(
        f"{col}:",
        "CO" if col in df.columns else "KHONG CO"
    )

display(df.head())

print("\nCELL 02 COMPLETED")


So file CSV trong ZIP: 8

DANH SACH FILE:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (91.6 MB)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (97.2 MB)
Friday-WorkingHours-Morning.pcap_ISCX.csv (71.9 MB)
Monday-WorkingHours.pcap_ISCX.csv (256.2 MB)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (103.7 MB)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (87.8 MB)
Tuesday-WorkingHours.pcap_ISCX.csv (166.6 MB)
Wednesday-workingHours.pcap_ISCX.csv (272.4 MB)

Tong so cot: 85

DANH SACH COT:
1. Flow ID
2. Source IP
3. Source Port
4. Destination IP
5. Destination Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Total Fwd Packets
10. Total Backward Packets
11. Total Length of Fwd Packets
12. Total Length of Bwd Packets
13. Fwd Packet Length Max
14. Fwd Packet Length Min
15. Fwd Packet Length Mean
16. Fwd Packet Length Std
17. Bwd Packet Length Max
18. Bwd Packet Length Min
19. Bwd Packet Length Mean
20. Bwd Packet Length Std
21. Flow Bytes/s
22. Flow Packets/s

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3,2,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443,192.168.10.16,46236,6,7/7/2017 3:30,34,1,1,...,20,0,0,0,0,0,0,0,0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443,192.168.10.5,54863,6,7/7/2017 3:30,3,2,0,...,20,0,0,0,0,0,0,0,0,BENIGN



CELL 02 COMPLETED


In [ ]:

!pip -q install -U kaggle

import os
from getpass import getpass

token = getpass("Nhap Kaggle API Token: ").strip()

assert token, "Token khong duoc de trong"

os.environ["KAGGLE_API_TOKEN"] = token
del token

print("Da cau hinh Kaggle API Token!")


Nhap Kaggle API Token: ··········
Da cau hinh Kaggle API Token!


In [ ]:

# CELL 04 | KIEM TRA KET NOI KAGGLE

!kaggle datasets list -s "CIC-DDoS2019" -p 1


ref                                                              title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dhoogla/cicddos2019                                              CIC-DDoS2019                                     30141235  2022-08-11 22:21:08.757000          20718         82                1  
aymenabb/ddos-evaluation-dataset-cic-ddos2019                    DDoS Evaluation Dataset (CIC-DDoS2019)           20591120  2023-07-28 17:22:48.960000           5869         37        0.9411765  
rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files        CIC-DDoS2019 30GB (Full Dataset CSV Files)     3326303131  2023-11-04 04:08:10.477000           9384         51        0.7058824  
dhoogla/cicidscollec

In [ ]:

# ============================================================
# CELL 04 | LIET KE FILE CIC-DDOS2019 TREN KAGGLE
# ============================================================

import subprocess

DATASET_SLUG = (
    "rodrigorosasilva/"
    "cic-ddos2019-30gb-full-dataset-csv-files"
)

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "files",
        DATASET_SLUG,
        "--page-size",
        "200"
    ],
    capture_output=True,
    text=True
)

print("=== DANH SACH FILE CIC-DDOS2019 ===")
print(result.stdout)

if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("\nCELL 04 COMPLETED")


=== DANH SACH FILE CIC-DDOS2019 ===
name                           size  creationDate                
-----------------------  ----------  --------------------------  
01-12/DrDoS_DNS.csv      2133307948  2023-11-04 04:09:01.671000  
01-12/DrDoS_LDAP.csv      917302277  2023-11-04 04:08:49.116000  
01-12/DrDoS_MSSQL.csv    1889181756  2023-11-04 04:09:02.089000  
01-12/DrDoS_NTP.csv       645008606  2023-11-04 04:08:46.805000  
01-12/DrDoS_NetBIOS.csv  1697479829  2023-11-04 04:08:55.375000  
01-12/DrDoS_SNMP.csv     2172579815  2023-11-04 04:09:05.409000  
01-12/DrDoS_SSDP.csv     1252686219  2023-11-04 04:08:53.641000  
01-12/DrDoS_UDP.csv      1506039371  2023-11-04 04:08:59.551000  
01-12/Syn.csv             637312127  2023-11-04 04:08:47.721000  
01-12/TFTP.csv           9302011115  2023-11-04 04:12:34.339000  
01-12/UDPLag.csv          157968770  2023-11-04 04:10:34.656000  
03-11/LDAP.csv            871398918  2023-11-04 04:10:45.687000  
03-11/MSSQL.csv          2386225228  202

In [ ]:

# ====================================================
# CELL 05 | KIEM TRA SCHEMA CIC-DDOS2019
# ====================================================

import subprocess
from pathlib import Path
import pandas as pd

DATASET_SLUG = (
    "rodrigorosasilva/"
    "cic-ddos2019-30gb-full-dataset-csv-files"
)

FILE_NAME = "03-11/Portmap.csv"

DOWNLOAD_DIR = Path("/content/NIDS_Work/CIC-DDoS2019")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# 1. Chi tai 1 file tu Kaggle
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d", DATASET_SLUG,
        "-f", FILE_NAME,
        "-p", str(DOWNLOAD_DIR),
        "--unzip"
    ],
    check=True
)

# 2. Tim CSV vua tai
csv_files = sorted(DOWNLOAD_DIR.rglob("*.csv"))

assert csv_files, "Khong tim thay CSV sau khi tai!"

csv_path = next(
    (p for p in csv_files if p.name == "Portmap.csv"),
    csv_files[0]
)

print("File:", csv_path)
print("Size:", round(csv_path.stat().st_size / 1024**2, 2), "MB")

# 3. Chi doc 10 dong dau
df_sample = pd.read_csv(
    csv_path,
    nrows=10,
    low_memory=False
)

df_sample.columns = df_sample.columns.str.strip()

print("\n=== TONG SO COT ===")
print(len(df_sample.columns))

print("\n=== DANH SACH COT ===")
for i, col in enumerate(df_sample.columns, 1):
    print(f"{i}. {col}")

# 4. Kiem tra metadata cho Giai doan 3 va 4
print("\n=== KIEM TRA CAC COT QUAN TRONG ===")

required_columns = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Source Port",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Label"
]

for col in required_columns:
    status = "CO" if col in df_sample.columns else "THIEU"
    print(f"{col}: {status}")

# 5. Xem cac nhan xuat hien trong 10 dong mau
if "Label" in df_sample.columns:
    print("\n=== NHAN TRONG MAU ===")
    print(df_sample["Label"].value_counts())

print("\n=== 5 DONG DAU ===")
display(df_sample.head())

print("\nCELL 05 COMPLETED")


File: /content/NIDS_Work/CIC-DDoS2019/03-11/Portmap.csv
Size: 74.97 MB

=== TONG SO COT ===
88

=== DANH SACH COT ===
1. Unnamed: 0
2. Flow ID
3. Source IP
4. Source Port
5. Destination IP
6. Destination Port
7. Protocol
8. Timestamp
9. Flow Duration
10. Total Fwd Packets
11. Total Backward Packets
12. Total Length of Fwd Packets
13. Total Length of Bwd Packets
14. Fwd Packet Length Max
15. Fwd Packet Length Min
16. Fwd Packet Length Mean
17. Fwd Packet Length Std
18. Bwd Packet Length Max
19. Bwd Packet Length Min
20. Bwd Packet Length Mean
21. Bwd Packet Length Std
22. Flow Bytes/s
23. Flow Packets/s
24. Flow IAT Mean
25. Flow IAT Std
26. Flow IAT Max
27. Flow IAT Min
28. Fwd IAT Total
29. Fwd IAT Mean
30. Fwd IAT Std
31. Fwd IAT Max
32. Fwd IAT Min
33. Bwd IAT Total
34. Bwd IAT Mean
35. Bwd IAT Std
36. Bwd IAT Max
37. Bwd IAT Min
38. Fwd PSH Flags
39. Bwd PSH Flags
40. Fwd URG Flags
41. Bwd URG Flags
42. Fwd Header Length
43. Bwd Header Length
44. Fwd Packets/s
45. Bwd Packets/s
46.

,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,24,192.168.50.254-224.0.0.5-0-0-0,192.168.50.254,0,224.0.0.5,0,0,2018-11-03 09:18:16.964447,114456999,45,...,28337.112288,98168.0,3.0,9529897.25,351582.631269,10001143.0,9048097.0,0,0,BENIGN
1,26,192.168.50.253-224.0.0.5-0-0-0,192.168.50.253,0,224.0.0.5,0,0,2018-11-03 09:18:18.506537,114347504,56,...,121314.911865,420255.0,4.0,9493929.75,351541.079539,9978130.0,8820294.0,0,0,BENIGN
2,176563,172.217.10.98-192.168.50.6-443-54799-6,192.168.50.6,54799,172.217.10.98,443,6,2018-11-03 09:18:18.610576,36435473,6,...,0.000000,62416.0,62416.0,36373056.00,0.000000,36373056.0,36373056.0,0,0,BENIGN
3,50762,172.217.7.2-192.168.50.6-443-54800-6,192.168.50.6,54800,172.217.7.2,443,6,2018-11-03 09:18:18.610579,36434705,6,...,0.000000,62413.0,62413.0,36372291.00,0.000000,36372291.0,36372291.0,0,0,BENIGN
4,87149,172.217.10.98-192.168.50.6-443-54801-6,192.168.50.6,54801,172.217.10.98,443,6,2018-11-03 09:18:18.610581,36434626,6,...,0.000000,62409.0,62409.0,36372216.00,0.000000,36372216.0,36372216.0,0,0,BENIGN



CELL 05 COMPLETED


In [ ]:

# ============================================================
# CELL 06 | CHUAN BI CIC-IDS2017
# ============================================================

from pathlib import Path
import zipfile
import shutil

ROOT = Path("/content/drive/MyDrive/NIDS_Project")
WORK = Path("/content/NIDS_Work")

IDS17_DRIVE = ROOT / "datasets/CIC-IDS2017"
IDS17_LOCAL = WORK / "CIC-IDS2017"
DDOS19_LOCAL = WORK / "CIC-DDoS2019"

IDS17_LOCAL.mkdir(parents=True, exist_ok=True)
DDOS19_LOCAL.mkdir(parents=True, exist_ok=True)

# Uu tien cac CSV da giai nen tu Cell 02
ids17_files = sorted(IDS17_LOCAL.rglob("*.csv"))

if not ids17_files:
    zip_path = IDS17_DRIVE / "CIC-IDS2017.zip"

    if zip_path.exists():
        print("Dang giai nen CIC-IDS2017...")

        with zipfile.ZipFile(zip_path) as zf:
            for member in zf.infolist():
                if not member.filename.lower().endswith(".csv"):
                    continue

                # Chi lay ten file, tranh duong dan khong an toan
                filename = Path(member.filename).name
                if not filename:
                    continue

                target = IDS17_LOCAL / filename
                with zf.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

    else:
        # Truong hop CSV duoc upload truc tiep len Drive
        ids17_files = sorted(IDS17_DRIVE.rglob("*.csv"))

# Quet lai sau giai nen
if not ids17_files:
    ids17_files = sorted(IDS17_LOCAL.rglob("*.csv"))

if not ids17_files:
    raise FileNotFoundError(
        "Chua tim thay CIC-IDS2017. "
        "Hay upload CIC-IDS2017.zip len Google Drive."
    )

print("=== CIC-IDS2017 ===")
print("So file:", len(ids17_files))

for i, path in enumerate(ids17_files, 1):
    mb = path.stat().st_size / (1024 ** 2)
    print(f"{i:02d}. {path.name} - {mb:.1f} MB")

print("\n=== CIC-DDOS2019 ===")
print("Portmap da tai:", bool(list(
    DDOS19_LOCAL.rglob("Portmap.csv")
)))

print("\nCELL 06 COMPLETED")


=== CIC-IDS2017 ===
So file: 8
01. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv - 91.6 MB
02. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv - 97.2 MB
03. Friday-WorkingHours-Morning.pcap_ISCX.csv - 71.9 MB
04. Monday-WorkingHours.pcap_ISCX.csv - 256.2 MB
05. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv - 103.7 MB
06. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv - 87.8 MB
07. Tuesday-WorkingHours.pcap_ISCX.csv - 166.6 MB
08. Wednesday-workingHours.pcap_ISCX.csv - 272.4 MB

=== CIC-DDOS2019 ===
Portmap da tai: True

CELL 06 COMPLETED


In [ ]:

# ============================================================
# CELL 07 | HAM EDA VA LAY MAU CHUNG CHO HAI DATASET
# ============================================================

import numpy as np
import pandas as pd
import json
import gc
from collections import Counter

CHUNK_SIZE = 25000
SAMPLE_PER_LABEL_FILE = 3000
SEED = 42

RESULT_ROOT = ROOT / "results" / "EDA"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# Danh sach dac trung can kiem tra Infinity
NUMERIC_CHECK = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
]

def analyze_csv(path, dataset_name, file_id):
    """
    Doc CSV theo chunk:
    1. Thong ke tong so flow va phan bo Label
    2. Thong ke missing va Infinity
    3. Lay mau ngau nhien theo tung Label
    4. Luu JSON va Parquet len Google Drive
    """

    output_dir = RESULT_ROOT / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    report_path = output_dir / f"{file_id}_eda.json"
    sample_path = output_dir / f"{file_id}_sample.parquet"

    # Neu da xu ly thanh cong, khong xu ly lai
    if report_path.exists() and sample_path.exists():
        print(f"[SKIP] {dataset_name} / {file_id}")
        return

    print(f"\nDang xu ly: {dataset_name} / {path.name}")

    rng = np.random.default_rng(SEED)

    total_rows = 0
    label_counts = Counter()
    missing_counts = Counter()
    infinity_counts = Counter()

    reservoirs = {}

    reader = pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        dtype=str,
        encoding="utf-8-sig",
        encoding_errors="replace",
        on_bad_lines="skip"
    )

    for chunk in reader:

        chunk.columns = chunk.columns.str.strip()

        if "Label" not in chunk.columns:
            raise ValueError(
                f"File {path.name} thieu cot Label"
            )

        chunk["Label"] = (
            chunk["Label"]
            .fillna("__MISSING__")
            .str.strip()
        )

        total_rows += len(chunk)

        label_counts.update(
            chunk["Label"].value_counts().to_dict()
        )

        missing_counts.update(
            chunk.isna().sum().to_dict()
        )

        # Kiem tra Infinity
        for col in NUMERIC_CHECK:
            if col in chunk.columns:
                numeric = pd.to_numeric(
                    chunk[col], errors="coerce"
                )
                infinity_counts[col] += int(
                    np.isinf(numeric.to_numpy()).sum()
                )

        # Reservoir sampling bang random priority:
        # moi flow co co hoi duoc chon nhu nhau
        # trong cung file va cung nhan.
        for label, group in chunk.groupby(
            "Label", sort=False
        ):
            priorities = rng.random(len(group))

            k = min(SAMPLE_PER_LABEL_FILE, len(group))
            idx = np.argpartition(
                priorities, k - 1
            )[:k]

            selected = group.iloc[idx].copy()
            selected["__sample_priority"] = priorities[idx]

            if label in reservoirs:
                selected = pd.concat(
                    [reservoirs[label], selected],
                    ignore_index=True
                )

            reservoirs[label] = selected.nsmallest(
                SAMPLE_PER_LABEL_FILE,
                "__sample_priority"
            )

    if total_rows == 0:
        raise ValueError(f"File rong: {path}")

    # Luu sample co ca IP, Timestamp va cac flow features
    samples = pd.concat(
        list(reservoirs.values()),
        ignore_index=True
    )

    samples = samples.drop(
        columns=["__sample_priority"]
    )

    samples["__source_file"] = file_id
    samples["__dataset"] = dataset_name

    # Luu bao cao truoc vao bien, sau do ghi file
    report = {
        "dataset": dataset_name,
        "file": path.name,
        "rows_read": total_rows,
        "sample_rows": len(samples),
        "label_counts": dict(label_counts),
        "missing_counts": dict(missing_counts),
        "infinity_counts": dict(infinity_counts),
        "columns": samples.columns.tolist(),
    }

    # Parquet giup su dung lai nhanh hon CSV
    samples.to_parquet(sample_path, index=False)

    report_path.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print("Tong flow:", f"{total_rows:,}")
    print("Labels:", dict(label_counts))
    print("So flow mau:", f"{len(samples):,}")
    print("Da luu:", sample_path.name)

    del samples, reservoirs
    gc.collect()

print("CELL 07 COMPLETED - HAM EDA DA SAN SANG")


CELL 07 COMPLETED - HAM EDA DA SAN SANG


In [ ]:

# ============================================================
# CELL 08 | EDA TOAN BO CIC-IDS2017
# ============================================================

for i, path in enumerate(ids17_files, 1):

    print(f"\n[{i}/{len(ids17_files)}]")

    file_id = path.stem.replace(
        " ", "_"
    ).replace(".", "_")

    analyze_csv(
        path=path,
        dataset_name="CIC-IDS2017",
        file_id=file_id
    )

print("\nEDA CIC-IDS2017 HOAN THANH!")



[1/8]

Dang xu ly: CIC-IDS2017 / Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Tong flow: 225,745
Labels: {'BENIGN': 97718, 'DDoS': 128027}
So flow mau: 6,000
Da luu: Friday-WorkingHours-Afternoon-DDos_pcap_ISCX_sample.parquet

[2/8]

Dang xu ly: CIC-IDS2017 / Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Tong flow: 286,467
Labels: {'BENIGN': 127537, 'PortScan': 158930}
So flow mau: 6,000
Da luu: Friday-WorkingHours-Afternoon-PortScan_pcap_ISCX_sample.parquet

[3/8]

Dang xu ly: CIC-IDS2017 / Friday-WorkingHours-Morning.pcap_ISCX.csv
Tong flow: 191,033
Labels: {'BENIGN': 189067, 'Bot': 1966}
So flow mau: 4,966
Da luu: Friday-WorkingHours-Morning_pcap_ISCX_sample.parquet

[4/8]

Dang xu ly: CIC-IDS2017 / Monday-WorkingHours.pcap_ISCX.csv
Tong flow: 529,918
Labels: {'BENIGN': 529918}
So flow mau: 3,000
Da luu: Monday-WorkingHours_pcap_ISCX_sample.parquet

[5/8]

Dang xu ly: CIC-IDS2017 / Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Tong flow: 288,602
Labels: {'B

In [ ]:

# ============================================================
# CELL 09 | EDA CIC-DDOS2019 TU KAGGLE
# ============================================================

import subprocess

DATASET_SLUG = (
    "rodrigorosasilva/"
    "cic-ddos2019-30gb-full-dataset-csv-files"
)

DDOS_FILES = [
    "03-11/Portmap.csv",
    "01-12/UDPLag.csv",
    "03-11/UDPLag.csv",
    "01-12/Syn.csv",
    "01-12/DrDoS_NTP.csv",
    "03-11/LDAP.csv",
    "01-12/DrDoS_LDAP.csv",
    "01-12/TFTP.csv",
    "01-12/DrDoS_SSDP.csv",
    "03-11/NetBIOS.csv",
    "01-12/DrDoS_UDP.csv",
    "01-12/DrDoS_NetBIOS.csv",
    "03-11/UDP.csv",
    "03-11/Syn.csv",
    "01-12/DrDoS_MSSQL.csv",
    "01-12/DrDoS_DNS.csv",
    "01-12/DrDoS_SNMP.csv",
    "03-11/MSSQL.csv",
]

MAX_NEW_FILES = 6
processed = 0

for file_name in DDOS_FILES:

    file_id = file_name.replace(
        "/", "_"
    ).replace(".csv", "")

    output_dir = RESULT_ROOT / "CIC-DDoS2019"
    report_file = output_dir / f"{file_id}_eda.json"
    sample_file = output_dir / f"{file_id}_sample.parquet"

    if report_file.exists() and sample_file.exists():
        print("[DA XU LY]", file_name)
        continue

    if processed >= MAX_NEW_FILES:
        break

    # Kiem tra file da tai tu cac cell truoc
    local_path = DDOS19_LOCAL / file_name

    if not local_path.exists():
        print("\nDang tai:", file_name)

        result = subprocess.run(
            [
                "kaggle", "datasets", "download",
                "-d", DATASET_SLUG,
                "-f", file_name,
                "-p", str(DDOS19_LOCAL),
                "--unzip"
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            print("Loi:", result.stderr)
            raise RuntimeError(
                f"Khong tai duoc {file_name}"
            )

    # Mot so phien ban CLI giai nen ra thu muc khac
    if not local_path.exists():
        candidates = list(
            DDOS19_LOCAL.rglob(Path(file_name).name)
        )

        if not candidates:
            raise FileNotFoundError(file_name)

        local_path = candidates[0]

    print("Dang EDA:", local_path)

    analyze_csv(
        path=local_path,
        dataset_name="CIC-DDoS2019",
        file_id=file_id
    )

    # Chi xoa CSV tam cua DDoS2019 sau khi
    # bao cao va sample da duoc luu thanh cong
    local_path.unlink()
    print("Da xoa CSV tam:", file_name)

    processed += 1

completed = sum(
    (
        (RESULT_ROOT / "CIC-DDoS2019" /
         f"{f.replace('/', '_').replace('.csv', '')}_eda.json"
        ).exists()
        and
        (RESULT_ROOT / "CIC-DDoS2019" /
         f"{f.replace('/', '_').replace('.csv', '')}_sample.parquet"
        ).exists()
    )
    for f in DDOS_FILES
)

print(f"\nDA HOAN THANH: {completed}/18 FILE DDOS2019")

if completed < 18:
    print("Chay lai Cell 09 de xu ly cac file tiep theo.")
else:
    print("EDA CIC-DDOS2019 HOAN THANH!")


[DA XU LY] 03-11/Portmap.csv
[DA XU LY] 01-12/UDPLag.csv
[DA XU LY] 03-11/UDPLag.csv
[DA XU LY] 01-12/Syn.csv
[DA XU LY] 01-12/DrDoS_NTP.csv
[DA XU LY] 03-11/LDAP.csv
[DA XU LY] 01-12/DrDoS_LDAP.csv
[DA XU LY] 01-12/TFTP.csv
[DA XU LY] 01-12/DrDoS_SSDP.csv
[DA XU LY] 03-11/NetBIOS.csv
[DA XU LY] 01-12/DrDoS_UDP.csv
[DA XU LY] 01-12/DrDoS_NetBIOS.csv

Dang tai: 03-11/UDP.csv
Dang EDA: /content/NIDS_Work/CIC-DDoS2019/03-11/UDP.csv

Dang xu ly: CIC-DDoS2019 / UDP.csv
Tong flow: 3,782,206
Labels: {'MSSQL': 24392, 'BENIGN': 3134, 'UDP': 3754680}
So flow mau: 9,000
Da luu: 03-11_UDP_sample.parquet
Da xoa CSV tam: 03-11/UDP.csv

Dang tai: 03-11/Syn.csv
Dang EDA: /content/NIDS_Work/CIC-DDoS2019/03-11/Syn.csv

Dang xu ly: CIC-DDoS2019 / Syn.csv
Tong flow: 4,320,541
Labels: {'Syn': 4284751, 'BENIGN': 35790}
So flow mau: 6,000
Da luu: 03-11_Syn_sample.parquet
Da xoa CSV tam: 03-11/Syn.csv

Dang tai: 01-12/DrDoS_MSSQL.csv
Dang EDA: /content/NIDS_Work/CIC-DDoS2019/01-12/DrDoS_MSSQL.csv

Dang xu ly:

In [ ]:

# ============================================================
# CELL 10 | TONG HOP EDA - CA HAI DATASET
# ============================================================

# Su dung lai:
# ROOT, pd, json, Counter tu cac cell truoc
# KHONG import lai thu vien

EDA_ROOT = ROOT / "results" / "EDA"

DATASET_FILES = {
    "CIC-IDS2017": 8,
    "CIC-DDoS2019": 18
}

summary_rows = []
label_rows = []
quality_rows = []

for dataset_name, expected_count in DATASET_FILES.items():

    print("\n" + "=" * 65)
    print("DATASET:", dataset_name)
    print("=" * 65)

    folder = EDA_ROOT / dataset_name

    reports = sorted(folder.glob("*_eda.json"))
    samples = sorted(folder.glob("*_sample.parquet"))

    print("EDA reports:", len(reports))
    print("Sample files:", len(samples))
    print("Expected:", expected_count)

    # Kiem tra du file hay chua
    if len(reports) != expected_count:
        raise RuntimeError(
            f"{dataset_name}: Chua du {expected_count} EDA reports!"
        )

    if len(samples) != expected_count:
        raise RuntimeError(
            f"{dataset_name}: Chua du {expected_count} sample files!"
        )

    # Xac nhan moi report deu co sample tuong ung
    for report_path in reports:
        sample_name = report_path.name.replace(
            "_eda.json", "_sample.parquet"
        )

        if not (folder / sample_name).exists():
            raise FileNotFoundError(
                f"Thieu sample tuong ung: {sample_name}"
            )

    total_rows = 0
    total_samples = 0

    label_counts = Counter()
    missing_counts = Counter()
    infinity_counts = Counter()

    # Doc cac bao cao da luu
    for report_path in reports:

        report = json.loads(
            report_path.read_text(encoding="utf-8")
        )

        rows = int(report["rows_read"])
        sampled = int(report["sample_rows"])

        total_rows += rows
        total_samples += sampled

        label_counts.update(report["label_counts"])
        missing_counts.update(report["missing_counts"])
        infinity_counts.update(report["infinity_counts"])

        summary_rows.append({
            "Dataset": dataset_name,
            "File": report["file"],
            "Total_Flows": rows,
            "Sample_Flows": sampled,
            "Num_Classes": len(report["label_counts"])
        })

    # Bao cao tong quan
    print("\n=== THONG KE TONG QUAN ===")
    print(f"Total flows : {total_rows:,}")
    print(f"Sample flows: {total_samples:,}")
    print(f"Num classes : {len(label_counts)}")

    print("\n=== PHAN BO NHAN ===")

    for label, count in label_counts.most_common():

        percentage = (
            100 * count / total_rows
            if total_rows else 0
        )

        print(
            f"{str(label):25s}"
            f"{count:>12,}"
            f" ({percentage:.2f}%)"
        )

        label_rows.append({
            "Dataset": dataset_name,
            "Label": label,
            "Count": count,
            "Percentage": round(percentage, 4)
        })

    # Thong ke missing
    print("\n=== TOP 10 MISSING VALUES ===")

    for col, count in missing_counts.most_common(10):
        if count > 0:
            print(f"{col}: {count:,}")

    # Thong ke Infinity
    print("\n=== INFINITY VALUES ===")

    for col, count in infinity_counts.items():
        if count > 0:
            print(f"{col}: {count:,}")

    quality_rows.append({
        "Dataset": dataset_name,
        "Files": len(reports),
        "Total_Flows": total_rows,
        "Sample_Flows": total_samples,
        "Num_Classes": len(label_counts),
        "Missing_Cells": sum(missing_counts.values()),
        "Infinity_Checked": sum(infinity_counts.values())
    })

# ============================================================
# LUU BAO CAO TONG HOP
# ============================================================

files_df = pd.DataFrame(summary_rows)
labels_df = pd.DataFrame(label_rows)
quality_df = pd.DataFrame(quality_rows)

files_df.to_csv(
    EDA_ROOT / "all_files_summary.csv",
    index=False
)

labels_df.to_csv(
    EDA_ROOT / "all_labels_distribution.csv",
    index=False
)

quality_df.to_csv(
    EDA_ROOT / "all_datasets_quality.csv",
    index=False
)

print("\n" + "=" * 65)
print("TONG HOP HAI DATASET")
print("=" * 65)

display(quality_df)

print("\n=== LABEL DISTRIBUTION ===")
display(labels_df)

print("\nCELL 10 COMPLETED")



DATASET: CIC-IDS2017
EDA reports: 8
Sample files: 8
Expected: 8

=== THONG KE TONG QUAN ===
Total flows : 3,119,345
Sample flows: 55,193
Num classes : 16

=== PHAN BO NHAN ===
BENIGN                      2,273,097 (72.87%)
__MISSING__                   288,602 (9.25%)
DoS Hulk                      231,073 (7.41%)
PortScan                      158,930 (5.09%)
DDoS                          128,027 (4.10%)
DoS GoldenEye                  10,293 (0.33%)
FTP-Patator                     7,938 (0.25%)
SSH-Patator                     5,897 (0.19%)
DoS slowloris                   5,796 (0.19%)
DoS Slowhttptest                5,499 (0.18%)
Bot                             1,966 (0.06%)
Web Attack � Brute Force        1,507 (0.05%)
Web Attack � XSS                  652 (0.02%)
Infiltration                       36 (0.00%)
Web Attack � Sql Injection          21 (0.00%)
Heartbleed                         11 (0.00%)

=== TOP 10 MISSING VALUES ===
Flow Bytes/s: 289,960
Flow ID: 288,602
Source IP: 288,

,Dataset,Files,Total_Flows,Sample_Flows,Num_Classes,Missing_Cells,Infinity_Checked
0,CIC-IDS2017,8,3119345,55193,16,24243926,4376
1,CIC-DDoS2019,18,70427637,109571,19,249152,4092348



=== LABEL DISTRIBUTION ===


,Dataset,Label,Count,Percentage
0,CIC-IDS2017,BENIGN,2273097,72.8710
1,CIC-IDS2017,__MISSING__,288602,9.2520
2,CIC-IDS2017,DoS Hulk,231073,7.4077
3,CIC-IDS2017,PortScan,158930,5.0950
4,CIC-IDS2017,DDoS,128027,4.1043
5,CIC-IDS2017,DoS GoldenEye,10293,0.3300
6,CIC-IDS2017,FTP-Patator,7938,0.2545
7,CIC-IDS2017,SSH-Patator,5897,0.1890
8,CIC-IDS2017,DoS slowloris,5796,0.1858
9,CIC-IDS2017,DoS Slowhttptest,5499,0.1763



CELL 10 COMPLETED


In [ ]:

# ============================================================
# CELL 11 | TONG HOP SAMPLE - CA HAI DATASET
# ============================================================

# Su dung lai pd, ROOT, DATASET_FILES, EDA_ROOT
# KHONG import lai thu vien

PREP_DIR = ROOT / "datasets" / "prepared"
PREP_DIR.mkdir(parents=True, exist_ok=True)

prepared_summary = []

for dataset_name, expected_count in DATASET_FILES.items():

    print("\n" + "=" * 65)
    print("DATASET:", dataset_name)
    print("=" * 65)

    source_dir = EDA_ROOT / dataset_name

    parquet_files = sorted(
        source_dir.glob("*_sample.parquet")
    )

    if len(parquet_files) != expected_count:
        raise RuntimeError(
            f"{dataset_name}: Chua du {expected_count} Parquet!"
        )

    # --------------------------------------------------------
    # 1. DOC CAC FILE SAMPLE
    # --------------------------------------------------------

    parts = []
    reference_columns = None

    for path in parquet_files:

        part = pd.read_parquet(path)
        part.columns = part.columns.str.strip()

        if part.columns.duplicated().any():
            raise ValueError(
                f"Trung ten cot trong {path.name}"
            )

        if "Label" not in part.columns:
            raise ValueError(
                f"Thieu Label trong {path.name}"
            )

        # Kiem tra su khac nhau cua schema
        columns = set(part.columns)

        if reference_columns is None:
            reference_columns = columns

        elif columns != reference_columns:
            print(
                "[WARNING] Schema khac:",
                path.name
            )

            print(
                "Cot thieu:",
                sorted(reference_columns - columns)
            )

            print(
                "Cot them:",
                sorted(columns - reference_columns)
            )

        parts.append(part)

    # --------------------------------------------------------
    # 2. GHEP SAMPLE CUA CUNG MOT DATASET
    # --------------------------------------------------------

    df = pd.concat(
        parts,
        ignore_index=True,
        sort=False
    )

    df["Label"] = (
        df["Label"]
        .fillna("__MISSING__")
        .astype(str)
        .str.strip()
    )

    # Kiem tra khong tron lan dataset
    if "__dataset" in df.columns:
        if not df["__dataset"].eq(dataset_name).all():
            raise ValueError(
                "Phat hien sample tu dataset khac!"
            )

    print("\n=== SHAPE ===")
    print(df.shape)

    print("\n=== SO LUONG LOP ===")
    print(df["Label"].nunique())

    print("\n=== PHAN BO NHAN SAMPLE ===")
    print(df["Label"].value_counts())

    # --------------------------------------------------------
    # 3. KIEM TRA METADATA QUAN TRONG
    # --------------------------------------------------------

    metadata_cols = [
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
        "Protocol",
        "Timestamp",
        "Label",
        "__source_file"
    ]

    print("\n=== METADATA CHECK ===")

    for col in metadata_cols:
        status = (
            "OK" if col in df.columns
            else "MISSING"
        )

        print(f"{col:25s}: {status}")

    # --------------------------------------------------------
    # 4. KIEM TRA MISSING
    # --------------------------------------------------------

    print("\n=== TOP 10 COT MISSING ===")

    missing = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    display(missing.head(10).to_frame("Missing_Count"))

    # --------------------------------------------------------
    # 5. KIEM TRA FLOW TRUNG LAP
    # --------------------------------------------------------

    duplicate_keys = [
        "Flow ID",
        "Timestamp",
        "Label"
    ]

    if all(c in df.columns for c in duplicate_keys):

        duplicate_count = int(
            df.duplicated(
                subset=duplicate_keys
            ).sum()
        )

        print(
            "\nDuplicate Flow ID + Timestamp + Label:",
            f"{duplicate_count:,}"
        )

    else:
        duplicate_count = None

    # Chua xoa duplicate tai buoc nay
    # De danh gia va quyet dinh chong leakage sau

    # --------------------------------------------------------
    # 6. LUU DATASET DA TONG HOP
    # --------------------------------------------------------

    output_path = PREP_DIR / (
        f"{dataset_name}_sample_combined.parquet"
    )

    # Ghi file tam de tranh file dich bi hong neu loi
    temp_path = PREP_DIR / (
        f"{dataset_name}_sample_combined.tmp.parquet"
    )

    df.to_parquet(
        temp_path,
        index=False
    )

    temp_path.replace(output_path)

    size_mb = (
        output_path.stat().st_size / (1024 ** 2)
    )

    prepared_summary.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Classes": df["Label"].nunique(),
        "Duplicate_Flows": duplicate_count,
        "Size_MB": round(size_mb, 2)
    })

    print("\n=== DA LUU ===")
    print(output_path)
    print(f"Size: {size_mb:.2f} MB")

    del df, parts
    gc.collect()

# ------------------------------------------------------------
# 7. BAO CAO TONG HOP
# ------------------------------------------------------------

prepared_df = pd.DataFrame(prepared_summary)

display(prepared_df)

prepared_df.to_csv(
    PREP_DIR / "prepared_summary.csv",
    index=False
)

print("\nCELL 11 COMPLETED")
print("DA CHUAN BI XONG CA HAI DATASET")



DATASET: CIC-IDS2017

=== SHAPE ===
(55193, 87)

=== SO LUONG LOP ===
16

=== PHAN BO NHAN SAMPLE ===
Label
BENIGN                        24000
DDoS                           3000
PortScan                       3000
DoS Slowhttptest               3000
DoS slowloris                  3000
SSH-Patator                    3000
FTP-Patator                    3000
__MISSING__                    3000
DoS GoldenEye                  3000
DoS Hulk                       3000
Bot                            1966
Web Attack � Brute Force       1507
Web Attack � XSS                652
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64

=== METADATA CHECK ===
Flow ID                  : OK
Source IP                : OK
Source Port              : OK
Destination IP           : OK
Destination Port         : OK
Protocol                 : OK
Timestamp                : OK
Label                    : OK
__source_file            

,Missing_Count
Flow Bytes/s,3021
Source IP,3000
Flow ID,3000
Destination IP,3000
Destination Port,3000
Protocol,3000
Source Port,3000
Timestamp,3000
Flow Duration,3000
Total Backward Packets,3000



Duplicate Flow ID + Timestamp + Label: 5,344

=== DA LUU ===
/content/drive/MyDrive/NIDS_Project/datasets/prepared/CIC-IDS2017_sample_combined.parquet
Size: 8.59 MB

DATASET: CIC-DDoS2019

=== SHAPE ===
(109571, 90)

=== SO LUONG LOP ===
19

=== PHAN BO NHAN SAMPLE ===
Label
BENIGN           41259
Syn               9000
UDP               6000
MSSQL             6000
LDAP              6000
NetBIOS           6000
DrDoS_DNS         3000
DrDoS_LDAP        3000
DrDoS_MSSQL       3000
DrDoS_NetBIOS     3000
DrDoS_NTP         3000
TFTP              3000
DrDoS_UDP         3000
DrDoS_SSDP        3000
DrDoS_SNMP        3000
Portmap           3000
UDP-lag           3000
UDPLag            1873
WebDDoS            439
Name: count, dtype: int64

=== METADATA CHECK ===
Flow ID                  : OK
Source IP                : OK
Source Port              : OK
Destination IP           : OK
Destination Port         : OK
Protocol                 : OK
Timestamp                : OK
Label                    :

,Missing_Count
Flow Bytes/s,795
Unnamed: 0,0
Source IP,0
Source Port,0
Destination IP,0
Destination Port,0
Protocol,0
Timestamp,0
Flow Duration,0
Flow ID,0



Duplicate Flow ID + Timestamp + Label: 27

=== DA LUU ===
/content/drive/MyDrive/NIDS_Project/datasets/prepared/CIC-DDoS2019_sample_combined.parquet
Size: 13.50 MB


,Dataset,Rows,Columns,Classes,Duplicate_Flows,Size_MB
0,CIC-IDS2017,55193,87,16,5344,8.59
1,CIC-DDoS2019,109571,90,19,27,13.50



CELL 11 COMPLETED
DA CHUAN BI XONG CA HAI DATASET


In [ ]:

# ============================================================
# CELL 12 | CLEANING - CIC-IDS2017 & CIC-DDOS2019
# ============================================================

PREP_DIR = ROOT / "datasets" / "prepared"
CLEAN_DIR = PREP_DIR / "cleaned"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CIC-IDS2017", "CIC-DDoS2019"]
MIN_CLASS_SAMPLES = 100

def normalize_label(label, dataset):
    s = str(label).strip()

    if s.upper() == "BENIGN":
        return "BENIGN"

    if dataset == "CIC-DDoS2019":
        if s.lower().replace("-", "").replace("_", "") == "udplag":
            return "UDPLag"

    if dataset == "CIC-IDS2017":
        lower = s.lower()

        if lower.startswith("web attack"):
            if "brute force" in lower:
                return "WebAttack-BruteForce"
            if "sql injection" in lower:
                return "WebAttack-SQLInjection"
            if "xss" in lower:
                return "WebAttack-XSS"

    return s


cleaning_summary = []

for dataset in DATASETS:
    print("\n" + "=" * 65)
    print("CLEANING:", dataset)
    print("=" * 65)

    source = PREP_DIR / f"{dataset}_sample_combined.parquet"
    df = pd.read_parquet(source)
    original_rows = len(df)

    # 1. Chuan hoa nhan
    df["Label"] = df["Label"].map(
        lambda x: normalize_label(x, dataset)
    )

    # 2. Xac dinh flow khong hop le
    required_meta = [
        "Flow ID", "Source IP",
        "Destination IP", "Timestamp"
    ]

    missing_label = df["Label"].isin([
        "__MISSING__", "", "nan", "None"
    ])

    missing_meta = df[required_meta].isna().any(axis=1)

    for col in required_meta:
        missing_meta |= (
            df[col].astype(str).str.strip().eq("")
        )

    invalid = missing_label | missing_meta

    print("\nFlow thieu Label:", int(missing_label.sum()))
    print("Flow thieu metadata:", int(missing_meta.sum()))
    print("Tong flow khong hop le:", int(invalid.sum()))

    if invalid.any():
        print("\nFILE NGUON CO DU LIEU LOI:")
        print(
            df.loc[invalid, "__source_file"]
            .value_counts()
            .to_string()
        )

    df = df.loc[~invalid].copy()

    # 3. Phat hien flow cung khoa nhung khac nhan
    flow_key = [
        "Flow ID", "Timestamp",
        "Source IP", "Source Port",
        "Destination IP", "Destination Port",
        "Protocol"
    ]

    # Chuan hoa khoa de tranh sai khac vi khoang trang
    for col in flow_key:
        df[col] = df[col].astype(str).str.strip()

    conflict_mask = (
        df.groupby(flow_key, dropna=False)["Label"]
        .transform("nunique") > 1
    )

    conflicts = int(conflict_mask.sum())

    print("\nFlow co nhan mau thuan:", conflicts)

    # Khong tu chon mot nhan cho flow mau thuan
    df = df.loc[~conflict_mask].copy()

    # 4. Loai flow trung khoa
    duplicate_count = int(
        df.duplicated(subset=flow_key).sum()
    )

    df = df.drop_duplicates(
        subset=flow_key,
        keep="first"
    ).copy()

    print("Flow trung da loai:", duplicate_count)

    # 5. Xu ly lop qua it mau
    counts = df["Label"].value_counts()

    rare_labels = counts[
        counts < MIN_CLASS_SAMPLES
    ]

    print("\nCAC LOP HIEM:")
    print(rare_labels.to_string() if len(rare_labels)
          else "Khong co")

    # Luu danh sach lop hiem
    rare_labels.rename("Count").to_csv(
        CLEAN_DIR / f"{dataset}_rare_classes.csv"
    )

    # Loai lop hiem khoi tap train thu nghiem
    df = df[
        df["Label"].isin(
            counts[counts >= MIN_CLASS_SAMPLES].index
        )
    ].copy()

    # 6. Luu du lieu sach
    output = CLEAN_DIR / f"{dataset}_clean.parquet"
    df.to_parquet(output, index=False)

    print("\nSHAPE SAU CLEAN:", df.shape)
    print("\nPHAN BO NHAN SAU CLEAN:")
    print(df["Label"].value_counts())

    print("\nDA LUU:", output)

    cleaning_summary.append({
        "Dataset": dataset,
        "Original": original_rows,
        "Invalid": int(invalid.sum()),
        "Conflicting": conflicts,
        "Duplicates": duplicate_count,
        "Cleaned": len(df),
        "Classes": df["Label"].nunique()
    })

    del df
    gc.collect()

cleaning_df = pd.DataFrame(cleaning_summary)

print("\n========== CLEANING SUMMARY ==========")
display(cleaning_df)

cleaning_df.to_csv(
    CLEAN_DIR / "cleaning_summary.csv",
    index=False
)

print("\nCELL 12 COMPLETED")



CLEANING: CIC-IDS2017

Flow thieu Label: 3000
Flow thieu metadata: 3000
Tong flow khong hop le: 3000

FILE NGUON CO DU LIEU LOI:
__source_file
Thursday-WorkingHours-Morning-WebAttacks_pcap_ISCX    3000

Flow co nhan mau thuan: 0
Flow trung da loai: 1604

CAC LOP HIEM:
Label
Infiltration              35
WebAttack-SQLInjection    12
Heartbleed                11

SHAPE SAU CLEAN: (50531, 87)

PHAN BO NHAN SAU CLEAN:
Label
BENIGN                  23977
PortScan                 3000
DoS slowloris            2989
DoS Hulk                 2988
DDoS                     2981
DoS Slowhttptest         2868
DoS GoldenEye            2822
FTP-Patator              2514
SSH-Patator              2392
Bot                      1936
WebAttack-BruteForce     1418
WebAttack-XSS             646
Name: count, dtype: int64

DA LUU: /content/drive/MyDrive/NIDS_Project/datasets/prepared/cleaned/CIC-IDS2017_clean.parquet

CLEANING: CIC-DDoS2019

Flow thieu Label: 0
Flow thieu metadata: 0
Tong flow khong hop le: 0

,Dataset,Original,Invalid,Conflicting,Duplicates,Cleaned,Classes
0,CIC-IDS2017,55193,3000,0,1604,50531,12
1,CIC-DDoS2019,109571,0,0,27,109544,18



CELL 12 COMPLETED


In [ ]:

# ============================================================
# CELL 13 | TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

SPLIT_DIR = PREP_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
split_summary = []

for dataset in DATASETS:
    print("\n" + "=" * 65)
    print("SPLITTING:", dataset)
    print("=" * 65)

    source = CLEAN_DIR / f"{dataset}_clean.parquet"
    df = pd.read_parquet(source)

    if df["Label"].value_counts().min() < 3:
        raise ValueError(
            f"{dataset}: Co lop khong du de chia 3 tap"
        )

    # 70% Train - 30% phan con lai
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df["Label"],
        random_state=SEED
    )

    # 15% Validation - 15% Test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["Label"],
        random_state=SEED
    )

    # Kiem tra khong trung khoa flow
    flow_key = [
        "Flow ID", "Timestamp",
        "Source IP", "Source Port",
        "Destination IP", "Destination Port",
        "Protocol"
    ]

    def flow_ids(frame):
        return set(
            pd.util.hash_pandas_object(
                frame[flow_key], index=False
            ).tolist()
        )

    train_ids = flow_ids(train_df)
    val_ids = flow_ids(val_df)
    test_ids = flow_ids(test_df)

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    # Luu rieng cho tung dataset
    output_dir = SPLIT_DIR / dataset
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, frame in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        frame.to_parquet(
            output_dir / f"{name}.parquet",
            index=False
        )

        print(
            f"{name.upper():5s}: "
            f"{len(frame):>8,} flows | "
            f"{frame['Label'].nunique()} classes"
        )

    # Thong ke phan bo nhan theo tung tap
    distribution = pd.DataFrame({
        "Train": train_df["Label"].value_counts(),
        "Validation": val_df["Label"].value_counts(),
        "Test": test_df["Label"].value_counts()
    }).fillna(0).astype(int)

    display(distribution)

    split_summary.append({
        "Dataset": dataset,
        "Train": len(train_df),
        "Validation": len(val_df),
        "Test": len(test_df),
        "Classes": df["Label"].nunique()
    })

    del df, train_df, val_df, test_df, temp_df
    gc.collect()

split_summary_df = pd.DataFrame(split_summary)

split_summary_df.to_csv(
    SPLIT_DIR / "split_summary.csv",
    index=False
)

print("\n========== SPLIT SUMMARY ==========")
display(split_summary_df)

print("\nCELL 13 COMPLETED")



SPLITTING: CIC-IDS2017
TRAIN:   35,371 flows | 12 classes
VAL  :    7,580 flows | 12 classes
TEST :    7,580 flows | 12 classes


,Train,Validation,Test
Label,,,
BENIGN,16784,3597,3596
PortScan,2100,450,450
DoS slowloris,2092,448,449
DoS Hulk,2091,448,449
DDoS,2087,447,447
DoS Slowhttptest,2008,430,430
DoS GoldenEye,1975,423,424
FTP-Patator,1760,377,377
SSH-Patator,1674,359,359



SPLITTING: CIC-DDoS2019
TRAIN:   76,680 flows | 18 classes
VAL  :   16,432 flows | 18 classes
TEST :   16,432 flows | 18 classes


,Train,Validation,Test
Label,,,
BENIGN,28862,6185,6185
DrDoS_DNS,2100,450,450
DrDoS_LDAP,2100,450,450
DrDoS_MSSQL,2100,450,450
DrDoS_NTP,2100,450,450
DrDoS_NetBIOS,2100,450,450
DrDoS_SNMP,2100,450,450
DrDoS_SSDP,2100,450,450
DrDoS_UDP,2100,450,450



========== SPLIT SUMMARY ==========


,Dataset,Train,Validation,Test,Classes
0,CIC-IDS2017,35371,7580,7580,12
1,CIC-DDoS2019,76680,16432,16432,18



CELL 13 COMPLETED


In [ ]:

# ============================================================
# CELL 14 | PREPROCESSING CHO ML VA DEEP LEARNING
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib

READY_DIR = PREP_DIR / "ready"
READY_DIR.mkdir(parents=True, exist_ok=True)

# Khong dua metadata, target va cac cot nghi ngo leakage
# vao mo hinh phan loai
EXCLUDE_FEATURES = {
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Timestamp",
    "Label",
    "Inbound",
    "SimillarHTTP",
    "Fwd Header Length.1",
    "__source_file",
    "__dataset"
}

preprocessing_summary = []

for dataset in DATASETS:
    print("\n" + "=" * 65)
    print("PREPROCESSING:", dataset)
    print("=" * 65)

    input_dir = SPLIT_DIR / dataset
    output_dir = READY_DIR / dataset
    output_dir.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_parquet(
        input_dir / "train.parquet"
    )
    val_df = pd.read_parquet(
        input_dir / "val.parquet"
    )
    test_df = pd.read_parquet(
        input_dir / "test.parquet"
    )

    # --------------------------------------------------------
    # 1. CHON CAC DAC TRUNG SO CHI DUA TREN TRAIN
    # --------------------------------------------------------

    candidate_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_FEATURES
        and not col.startswith("__")
    ]

    feature_cols = []

    for col in candidate_cols:
        numeric = pd.to_numeric(
            train_df[col],
            errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

        valid_ratio = numeric.notna().mean()

        if valid_ratio >= 0.95 and numeric.nunique() > 1:
            feature_cols.append(col)

    if not feature_cols:
        raise ValueError(
            f"{dataset}: Khong tim thay numeric features!"
        )

    print("So dac trung dau vao:", len(feature_cols))

    # --------------------------------------------------------
    # 2. CHUYEN THANH MA TRAN SO
    # --------------------------------------------------------

    def make_numeric_matrix(frame):
        numeric_df = frame[feature_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        numeric_df = numeric_df.replace(
            [np.inf, -np.inf], np.nan
        )

        # Han che gia tri cuc lon, giam nguy co loi so
        numeric_df = numeric_df.clip(
            lower=-1e12,
            upper=1e12
        )

        return numeric_df.to_numpy(dtype=np.float32)

    X_train_raw = make_numeric_matrix(train_df)
    X_val_raw = make_numeric_matrix(val_df)
    X_test_raw = make_numeric_matrix(test_df)

    # --------------------------------------------------------
    # 3. IMPUTER - CHI FIT TREN TRAIN
    # --------------------------------------------------------

    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(X_train_raw)
    X_val = imputer.transform(X_val_raw)
    X_test = imputer.transform(X_test_raw)

    # --------------------------------------------------------
    # 4. SCALER - CHI FIT TREN TRAIN
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    X_train = X_train.astype(np.float32)
    X_val = X_val.astype(np.float32)
    X_test = X_test.astype(np.float32)

    # --------------------------------------------------------
    # 5. MA HOA NHAN
    # --------------------------------------------------------

    encoder = LabelEncoder()

    y_train = encoder.fit_transform(
        train_df["Label"]
    )
    y_val = encoder.transform(
        val_df["Label"]
    )
    y_test = encoder.transform(
        test_df["Label"]
    )

    # --------------------------------------------------------
    # 6. KIEM TRA DU LIEU
    # --------------------------------------------------------

    for name, X, y in [
        ("train", X_train, y_train),
        ("val", X_val, y_val),
        ("test", X_test, y_test)
    ]:

        if not np.isfinite(X).all():
            raise ValueError(
                f"{dataset}/{name}: Con NaN hoac Inf!"
            )

        if len(X) != len(y):
            raise ValueError(
                f"{dataset}/{name}: X/y khong khop!"
            )

        # Luu array cho ML va PyTorch
        np.savez_compressed(
            output_dir / f"{name}.npz",
            X=X,
            y=y.astype(np.int64)
        )

        print(
            f"{name.upper():5s} | "
            f"X={X.shape} | y={y.shape}"
        )

    # --------------------------------------------------------
    # 7. LUU PREPROCESSOR VA LABEL ENCODER
    # --------------------------------------------------------

    preprocessing_objects = {
        "feature_names": feature_cols,
        "imputer": imputer,
        "scaler": scaler,
        "label_encoder": encoder,
        "dataset": dataset
    }

    joblib.dump(
        preprocessing_objects,
        output_dir / "preprocessor.joblib"
    )

    label_mapping = {
        str(i): str(label)
        for i, label in enumerate(encoder.classes_)
    }

    (output_dir / "label_mapping.json").write_text(
        json.dumps(
            label_mapping,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("\nLABEL MAPPING:")
    print(label_mapping)

    preprocessing_summary.append({
        "Dataset": dataset,
        "Features": len(feature_cols),
        "Classes": len(encoder.classes_),
        "Train": len(y_train),
        "Validation": len(y_val),
        "Test": len(y_test)
    })

    del (
        train_df, val_df, test_df,
        X_train_raw, X_val_raw, X_test_raw,
        X_train, X_val, X_test,
        y_train, y_val, y_test
    )
    gc.collect()

preprocessing_df = pd.DataFrame(preprocessing_summary)

print("\n========== PREPROCESSING SUMMARY ==========")
display(preprocessing_df)

preprocessing_df.to_csv(
    READY_DIR / "preprocessing_summary.csv",
    index=False
)

print("\nCELL 14 COMPLETED")



PREPROCESSING: CIC-IDS2017
So dac trung dau vao: 70
TRAIN | X=(35371, 70) | y=(35371,)
VAL   | X=(7580, 70) | y=(7580,)
TEST  | X=(7580, 70) | y=(7580,)

LABEL MAPPING:
{'0': 'BENIGN', '1': 'Bot', '2': 'DDoS', '3': 'DoS GoldenEye', '4': 'DoS Hulk', '5': 'DoS Slowhttptest', '6': 'DoS slowloris', '7': 'FTP-Patator', '8': 'PortScan', '9': 'SSH-Patator', '10': 'WebAttack-BruteForce', '11': 'WebAttack-XSS'}

PREPROCESSING: CIC-DDoS2019
So dac trung dau vao: 66
TRAIN | X=(76680, 66) | y=(76680,)
VAL   | X=(16432, 66) | y=(16432,)
TEST  | X=(16432, 66) | y=(16432,)

LABEL MAPPING:
{'0': 'BENIGN', '1': 'DrDoS_DNS', '2': 'DrDoS_LDAP', '3': 'DrDoS_MSSQL', '4': 'DrDoS_NTP', '5': 'DrDoS_NetBIOS', '6': 'DrDoS_SNMP', '7': 'DrDoS_SSDP', '8': 'DrDoS_UDP', '9': 'LDAP', '10': 'MSSQL', '11': 'NetBIOS', '12': 'Portmap', '13': 'Syn', '14': 'TFTP', '15': 'UDP', '16': 'UDPLag', '17': 'WebDDoS'}

========== PREPROCESSING SUMMARY ==========


,Dataset,Features,Classes,Train,Validation,Test
0,CIC-IDS2017,70,12,35371,7580,7580
1,CIC-DDoS2019,66,18,76680,16432,16432



CELL 14 COMPLETED


In [ ]:

# ============================================================
# CELL 15 | LOAD DATA FOR TRAINING
# ============================================================

READY_DIR = ROOT / "datasets" / "prepared" / "ready"

DATASETS = ["CIC-IDS2017", "CIC-DDoS2019"]
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def load_training_data(dataset):
    folder = READY_DIR / dataset

    result = {}

    for split in ["train", "val", "test"]:
        with np.load(
            folder / f"{split}.npz",
            allow_pickle=False
        ) as data:
            result[f"X_{split}"] = data["X"].astype(
                np.float32
            )
            result[f"y_{split}"] = data["y"].astype(
                np.int64
            )

    mapping = json.loads(
        (folder / "label_mapping.json").read_text(
            encoding="utf-8"
        )
    )

    result["class_names"] = [
        mapping[str(i)] for i in range(len(mapping))
    ]

    result["n_features"] = result["X_train"].shape[1]
    result["n_classes"] = len(result["class_names"])

    for split in ["train", "val", "test"]:
        X = result[f"X_{split}"]
        y = result[f"y_{split}"]

        assert X.ndim == 2
        assert len(X) == len(y)
        assert X.shape[1] == result["n_features"]
        assert np.isfinite(X).all()
        assert y.min() >= 0
        assert y.max() < result["n_classes"]

    assert set(np.unique(result["y_train"])) == set(
        range(result["n_classes"])
    )

    return result


for dataset in DATASETS:
    data = load_training_data(dataset)

    print(f"\n=== {dataset} ===")
    print("Features:", data["n_features"])
    print("Classes :", data["n_classes"])
    print("Train   :", data["X_train"].shape)
    print("Val     :", data["X_val"].shape)
    print("Test    :", data["X_test"].shape)

    del data

print("\nCELL 15 COMPLETED")


Device: cpu

=== CIC-IDS2017 ===
Features: 70
Classes : 12
Train   : (35371, 70)
Val     : (7580, 70)
Test    : (7580, 70)

=== CIC-DDoS2019 ===
Features: 66
Classes : 18
Train   : (76680, 66)
Val     : (16432, 66)
Test    : (16432, 66)

CELL 15 COMPLETED


In [ ]:

# ============================================================
# CELL 16 | TRAIN 4 ML BASELINES
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score,
    balanced_accuracy_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier

ML_RESULTS = []

for dataset in DATASETS:

    print("\n" + "=" * 60)
    print("ML TRAINING:", dataset)
    print("=" * 60)

    data = load_training_data(dataset)

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_val = data["X_val"]
    y_val = data["y_val"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    n_classes = data["n_classes"]
    class_names = data["class_names"]

    model_dir = ROOT / "models" / dataset / "ML"
    result_dir = ROOT / "results" / dataset / "ML"

    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED
        ),

        "XGBoost": XGBClassifier(
            n_estimators=220,
            max_depth=6,
            learning_rate=0.06,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="multi:softprob",
            num_class=n_classes,
            tree_method="hist",
            device="cpu",
            eval_metric="mlogloss",
            n_jobs=2,
            random_state=SEED
        ),

        "LinearSVM": LinearSVC(
            class_weight="balanced",
            max_iter=6000,
            random_state=SEED
        ),

        "LogisticRegression": LogisticRegression(
            class_weight="balanced",
            max_iter=1500,
            random_state=SEED
        )
    }

    for name, model in models.items():

        print(f"\nDang huan luyen: {name}")

        model_path = model_dir / f"{name}.joblib"

        # Neu da train va luu, khong train lai
        if model_path.exists():
            print("Load model da luu...")
            model = joblib.load(model_path)
        else:
            model.fit(X_train, y_train)
            joblib.dump(model, model_path)

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        val_f1 = f1_score(
            y_val, val_pred,
            average="macro",
            zero_division=0
        )

        test_f1 = f1_score(
            y_test, test_pred,
            average="macro",
            zero_division=0
        )

        test_acc = accuracy_score(y_test, test_pred)
        test_bal_acc = balanced_accuracy_score(
            y_test, test_pred
        )

        print(f"Val Macro-F1  : {val_f1:.4f}")
        print(f"Test Macro-F1 : {test_f1:.4f}")
        print(f"Test Accuracy : {test_acc:.4f}")

        ML_RESULTS.append({
            "Dataset": dataset,
            "Model": name,
            "Val_Macro_F1": val_f1,
            "Test_Macro_F1": test_f1,
            "Test_Accuracy": test_acc,
            "Test_Balanced_Accuracy": test_bal_acc
        })

        labels_idx = list(range(n_classes))

        report = classification_report(
            y_test,
            test_pred,
            labels=labels_idx,
            target_names=class_names,
            zero_division=0,
            output_dict=True
        )

        pd.DataFrame(report).T.to_csv(
            result_dir / f"{name}_classification_report.csv"
        )

        cm = confusion_matrix(
            y_test, test_pred,
            labels=labels_idx
        )

        pd.DataFrame(
            cm,
            index=class_names,
            columns=class_names
        ).to_csv(
            result_dir / f"{name}_confusion_matrix.csv"
        )

        # Luu ngay sau moi model de tranh mat ket qua
        pd.DataFrame(ML_RESULTS).to_csv(
            ROOT / "results" / "ml_baselines_summary.csv",
            index=False
        )

        del model
        gc.collect()

    del data, models
    gc.collect()

ml_results_df = pd.DataFrame(ML_RESULTS)

print("\n=== ML BASELINE RESULTS ===")
display(
    ml_results_df.sort_values(
        ["Dataset", "Val_Macro_F1"],
        ascending=[True, False]
    )
)

print("\nCELL 16 COMPLETED")



ML TRAINING: CIC-IDS2017

Dang huan luyen: RandomForest
Val Macro-F1  : 0.9277
Test Macro-F1 : 0.9233
Test Accuracy : 0.9822

Dang huan luyen: XGBoost
Val Macro-F1  : 0.9269
Test Macro-F1 : 0.9283
Test Accuracy : 0.9846

Dang huan luyen: LinearSVM
Val Macro-F1  : 0.8373
Test Macro-F1 : 0.8415
Test Accuracy : 0.9157

Dang huan luyen: LogisticRegression
Val Macro-F1  : 0.8436
Test Macro-F1 : 0.8483
Test Accuracy : 0.8978

ML TRAINING: CIC-DDoS2019

Dang huan luyen: RandomForest
Val Macro-F1  : 0.5772
Test Macro-F1 : 0.5769
Test Accuracy : 0.7331

Dang huan luyen: XGBoost
Val Macro-F1  : 0.5830
Test Macro-F1 : 0.5838
Test Accuracy : 0.7699

Dang huan luyen: LinearSVM


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Val Macro-F1  : 0.3918
Test Macro-F1 : 0.3834
Test Accuracy : 0.6715

Dang huan luyen: LogisticRegression
Val Macro-F1  : 0.3988
Test Macro-F1 : 0.3925
Test Accuracy : 0.6158

=== ML BASELINE RESULTS ===


,Dataset,Model,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy
5,CIC-DDoS2019,XGBoost,0.583038,0.583816,0.769900,0.597789
4,CIC-DDoS2019,RandomForest,0.577157,0.576932,0.733082,0.575143
7,CIC-DDoS2019,LogisticRegression,0.398751,0.392472,0.615811,0.479955
6,CIC-DDoS2019,LinearSVM,0.391828,0.383418,0.671495,0.436783
0,CIC-IDS2017,RandomForest,0.927691,0.923304,0.982190,0.924234
1,CIC-IDS2017,XGBoost,0.926919,0.928306,0.984565,0.928735
3,CIC-IDS2017,LogisticRegression,0.843625,0.848268,0.897757,0.916751
2,CIC-IDS2017,LinearSVM,0.837291,0.841525,0.915699,0.893731



CELL 16 COMPLETED


In [ ]:

# ============================================================
# CELL 17 | 5 DEEP LEARNING ARCHITECTURES
# ============================================================

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DL_ARCHITECTURES = [
    "CNN1D",
    "LSTM",
    "BiLSTM",
    "CNN-LSTM",
    "CNN-BiLSTM"
]


class FlowFeatureNet(nn.Module):

    def __init__(self, architecture, n_classes):
        super().__init__()

        self.architecture = architecture

        if architecture == "CNN1D":

            self.cnn = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=5, padding=2),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.MaxPool1d(2),

                nn.Conv1d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm1d(64),
                nn.ReLU(),

                nn.AdaptiveAvgPool1d(1)
            )

            output_size = 64

        elif architecture in ["LSTM", "BiLSTM"]:

            bidirectional = architecture == "BiLSTM"

            self.lstm = nn.LSTM(
                input_size=1,
                hidden_size=64,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            )

            output_size = 128 if bidirectional else 64

        elif architecture in ["CNN-LSTM", "CNN-BiLSTM"]:

            self.cnn = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=5, padding=2),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.MaxPool1d(2),

                nn.Conv1d(32, 48, kernel_size=3, padding=1),
                nn.BatchNorm1d(48),
                nn.ReLU()
            )

            bidirectional = architecture == "CNN-BiLSTM"

            self.lstm = nn.LSTM(
                input_size=48,
                hidden_size=64,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            )

            output_size = 128 if bidirectional else 64

        else:
            raise ValueError(
                f"Architecture khong ho tro: {architecture}"
            )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(output_size, n_classes)
        )

    def forward(self, x):

        if self.architecture == "CNN1D":
            # (Batch, Features) -> (Batch, 1, Features)
            z = self.cnn(x.unsqueeze(1))
            z = z.squeeze(-1)

        elif self.architecture in ["LSTM", "BiLSTM"]:
            # Xem cac feature la cac vi tri trong chuoi
            z = x.unsqueeze(-1)
            z, _ = self.lstm(z)
            z = z[:, -1, :]

        else:
            # CNN trich xuat dac trung cuc bo
            z = self.cnn(x.unsqueeze(1))

            # (Batch, Channels, Length)
            # -> (Batch, Length, Channels)
            z = z.transpose(1, 2)

            # LSTM xu ly chuoi feature da trich xuat
            z, _ = self.lstm(z)
            z = z[:, -1, :]

        return self.classifier(z)


# Kiem tra forward pass truoc khi train
for dataset in DATASETS:

    data = load_training_data(dataset)

    dummy = torch.zeros(
        4, data["n_features"],
        device=DEVICE
    )

    for architecture in DL_ARCHITECTURES:
        model = FlowFeatureNet(
            architecture,
            data["n_classes"]
        ).to(DEVICE)

        with torch.inference_mode():
            output = model(dummy)

        assert output.shape == (
            4, data["n_classes"]
        )

        print(
            dataset, architecture,
            "Output:", tuple(output.shape)
        )

        del model

    del data

print("\nCELL 17 COMPLETED")


CIC-IDS2017 CNN1D Output: (4, 12)
CIC-IDS2017 LSTM Output: (4, 12)
CIC-IDS2017 BiLSTM Output: (4, 12)
CIC-IDS2017 CNN-LSTM Output: (4, 12)
CIC-IDS2017 CNN-BiLSTM Output: (4, 12)
CIC-DDoS2019 CNN1D Output: (4, 18)
CIC-DDoS2019 LSTM Output: (4, 18)
CIC-DDoS2019 BiLSTM Output: (4, 18)
CIC-DDoS2019 CNN-LSTM Output: (4, 18)
CIC-DDoS2019 CNN-BiLSTM Output: (4, 18)

CELL 17 COMPLETED


In [ ]:

# ============================================================
# CELL 18 | TRAIN DEEP LEARNING - BOTH DATASETS
# ============================================================

EPOCHS = 18
PATIENCE = 4
BATCH_SIZE = 512 if DEVICE.type == "cuda" else 128
LEARNING_RATE = 1e-3

# Lan dau: uu tien mo hinh chinh
RUN_DL_MODELS = DL_ARCHITECTURES

# Sau khi CNN-LSTM thanh cong, doi thanh:
# RUN_DL_MODELS = DL_ARCHITECTURES

USE_AMP = DEVICE.type == "cuda"

DL_RESULT_ROOT = ROOT / "results"
all_dl_records = []


def get_predictions(model, loader):
    model.eval()
    predictions = []
    targets = []

    with torch.inference_mode():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)

            predictions.append(
                logits.argmax(dim=1).cpu().numpy()
            )
            targets.append(yb.numpy())

    return (
        np.concatenate(targets),
        np.concatenate(predictions)
    )


for dataset in DATASETS:

    print("\n" + "=" * 65)
    print("DEEP LEARNING:", dataset)
    print("=" * 65)

    data = load_training_data(dataset)
    n_classes = data["n_classes"]
    n_features = data["n_features"]
    class_names = data["class_names"]

    model_dir = ROOT / "models" / dataset / "DL"
    result_dir = DL_RESULT_ROOT / dataset / "DL"

    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    def make_loader(split, shuffle=False):
        x = torch.from_numpy(data[f"X_{split}"])
        y = torch.from_numpy(data[f"y_{split}"])

        return DataLoader(
            TensorDataset(x, y),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=0,
            pin_memory=USE_AMP
        )

    train_loader = make_loader("train", shuffle=True)
    val_loader = make_loader("val")
    test_loader = make_loader("test")

    # Class weights giam nhe tac dong mat can bang
    counts = np.bincount(
        data["y_train"], minlength=n_classes
    )

    weights = np.sqrt(
        len(data["y_train"]) /
        (n_classes * np.maximum(counts, 1))
    )

    weights = np.clip(weights, 0.5, 6.0)

    class_weights = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )

    for architecture in RUN_DL_MODELS:

        print(f"\n=== TRAIN {architecture} ===")

        checkpoint_path = (
            model_dir / f"{architecture}_best.pt"
        )

        metrics_path = (
            result_dir / f"{architecture}_metrics.json"
        )

        if checkpoint_path.exists() and metrics_path.exists():
            print("Model da hoan thanh - bo qua.")

            previous = json.loads(
                metrics_path.read_text(encoding="utf-8")
            )
            all_dl_records.append(previous)
            continue

        model = FlowFeatureNet(
            architecture, n_classes
        ).to(DEVICE)

        criterion = nn.CrossEntropyLoss(
            weight=class_weights
        )

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=1e-4
        )

        grad_scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP
        )

        best_val_f1 = -1.0
        best_epoch = 0
        patience_counter = 0
        history = []

        for epoch in range(1, EPOCHS + 1):

            model.train()
            total_loss = 0.0
            total_samples = 0

            for xb, yb in train_loader:

                xb = xb.to(
                    DEVICE, non_blocking=True
                )
                yb = yb.to(
                    DEVICE, non_blocking=True
                )

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(
                    "cuda", enabled=USE_AMP
                ):
                    logits = model(xb)
                    loss = criterion(logits, yb)

                grad_scaler.scale(loss).backward()

                grad_scaler.unscale_(optimizer)

                nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )

                grad_scaler.step(optimizer)
                grad_scaler.update()

                total_loss += (
                    loss.item() * len(xb)
                )
                total_samples += len(xb)

            train_loss = total_loss / total_samples

            # Validation Macro-F1
            y_true, y_pred = get_predictions(
                model, val_loader
            )

            val_f1 = f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

            history.append({
                "Epoch": epoch,
                "Train_Loss": train_loss,
                "Val_Macro_F1": val_f1
            })

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"Loss={train_loss:.4f} | "
                f"Val Macro-F1={val_f1:.4f}"
            )

            # Luu lich su sau moi epoch
            pd.DataFrame(history).to_csv(
                result_dir /
                f"{architecture}_history.csv",
                index=False
            )

            # Luu checkpoint tot nhat
            if val_f1 > best_val_f1 + 1e-4:

                best_val_f1 = val_f1
                best_epoch = epoch
                patience_counter = 0

                torch.save({
                    "model_state": model.state_dict(),
                    "architecture": architecture,
                    "dataset": dataset,
                    "n_features": n_features,
                    "n_classes": n_classes,
                    "class_names": class_names,
                    "best_epoch": epoch,
                    "best_val_macro_f1": float(val_f1)
                }, checkpoint_path)

                print("  -> Saved best checkpoint")

            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                print("Early stopping.")
                break

        # Tai checkpoint tot nhat de danh gia Test
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=True
        )

        model.load_state_dict(
            checkpoint["model_state"]
        )

        y_true, y_pred = get_predictions(
            model, test_loader
        )

        test_f1 = f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )

        test_acc = accuracy_score(
            y_true, y_pred
        )

        test_bal_acc = balanced_accuracy_score(
            y_true, y_pred
        )

        print("\n=== FINAL TEST RESULT ===")
        print("Best epoch:", best_epoch)
        print(f"Validation Macro-F1: {best_val_f1:.4f}")
        print(f"Test Macro-F1      : {test_f1:.4f}")
        print(f"Test Accuracy      : {test_acc:.4f}")

        record = {
            "Dataset": dataset,
            "Model": architecture,
            "Val_Macro_F1": float(best_val_f1),
            "Test_Macro_F1": float(test_f1),
            "Test_Accuracy": float(test_acc),
            "Test_Balanced_Accuracy": float(test_bal_acc),
            "Best_Epoch": best_epoch
        }

        # Classification report
        report = classification_report(
            y_true, y_pred,
            labels=list(range(n_classes)),
            target_names=class_names,
            zero_division=0,
            output_dict=True
        )

        pd.DataFrame(report).T.to_csv(
            result_dir /
            f"{architecture}_classification_report.csv"
        )

        # Confusion matrix
        cm = confusion_matrix(
            y_true, y_pred,
            labels=list(range(n_classes))
        )

        pd.DataFrame(
            cm,
            index=class_names,
            columns=class_names
        ).to_csv(
            result_dir /
            f"{architecture}_confusion_matrix.csv"
        )

        metrics_path.write_text(
            json.dumps(
                record, indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        all_dl_records.append(record)

        # Luu tong hop sau moi model
        pd.DataFrame(all_dl_records).to_csv(
            DL_RESULT_ROOT / "dl_models_summary.csv",
            index=False
        )

        del model, optimizer, grad_scaler
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del data, train_loader, val_loader, test_loader
    gc.collect()

print("\n=== DEEP LEARNING SUMMARY ===")
display(pd.DataFrame(all_dl_records))

print("\nCELL 18 COMPLETED")



DEEP LEARNING: CIC-IDS2017

=== TRAIN CNN1D ===
Epoch 01/18 | Loss=1.8738 | Val Macro-F1=0.4909
  -> Saved best checkpoint
Epoch 02/18 | Loss=1.2083 | Val Macro-F1=0.6943
  -> Saved best checkpoint
Epoch 03/18 | Loss=0.9048 | Val Macro-F1=0.7355
  -> Saved best checkpoint
Epoch 04/18 | Loss=0.7482 | Val Macro-F1=0.7461
  -> Saved best checkpoint
Epoch 05/18 | Loss=0.6522 | Val Macro-F1=0.7341
Epoch 06/18 | Loss=0.5874 | Val Macro-F1=0.7670
  -> Saved best checkpoint
Epoch 07/18 | Loss=0.5463 | Val Macro-F1=0.7710
  -> Saved best checkpoint
Epoch 08/18 | Loss=0.5087 | Val Macro-F1=0.7965
  -> Saved best checkpoint
Epoch 09/18 | Loss=0.4795 | Val Macro-F1=0.7951
Epoch 10/18 | Loss=0.4556 | Val Macro-F1=0.8090
  -> Saved best checkpoint
Epoch 11/18 | Loss=0.4369 | Val Macro-F1=0.7973
Epoch 12/18 | Loss=0.4188 | Val Macro-F1=0.8491
  -> Saved best checkpoint
Epoch 13/18 | Loss=0.4101 | Val Macro-F1=0.7639
Epoch 14/18 | Loss=0.3986 | Val Macro-F1=0.7944
Epoch 15/18 | Loss=0.3831 | Val Macr

,Dataset,Model,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy,Best_Epoch
0,CIC-IDS2017,CNN1D,0.849138,0.855036,0.909894,0.887052,12
1,CIC-IDS2017,LSTM,0.837153,0.840775,0.923087,0.887809,17
2,CIC-IDS2017,BiLSTM,0.810069,0.814100,0.897889,0.858528,15
3,CIC-IDS2017,CNN-LSTM,0.862683,0.860558,0.944855,0.893863,7
4,CIC-IDS2017,CNN-BiLSTM,0.867516,0.872032,0.954749,0.905239,15
5,CIC-DDoS2019,CNN1D,0.421669,0.418515,0.717989,0.462041,7
6,CIC-DDoS2019,LSTM,0.425810,0.423225,0.715372,0.465197,17
7,CIC-DDoS2019,BiLSTM,0.433861,0.426553,0.722797,0.476992,15
8,CIC-DDoS2019,CNN-LSTM,0.481173,0.482639,0.735029,0.528511,15
9,CIC-DDoS2019,CNN-BiLSTM,0.493094,0.493523,0.740263,0.529573,14



CELL 18 COMPLETED


In [ ]:

# ============================================================
# CELL 19 | FINAL MODEL COMPARISON - BOTH DATASETS
# ============================================================

# Su dung lai:
# ROOT, pd, DATASETS tu cac cell truoc
# Khong import lai thu vien

RESULT_DIR = ROOT / "results"

ML_PATH = RESULT_DIR / "ml_baselines_summary.csv"
DL_PATH = RESULT_DIR / "dl_models_summary.csv"

# ============================================================
# 1. KIEM TRA FILE KET QUA
# ============================================================

assert ML_PATH.exists(), (
    "Chua tim thay ket qua Machine Learning!"
)

assert DL_PATH.exists(), (
    "Chua tim thay ket qua Deep Learning!"
)

ml_df = pd.read_csv(ML_PATH)
dl_df = pd.read_csv(DL_PATH)

print("=== KIEM TRA KET QUA ===")
print("ML records:", len(ml_df))
print("DL records:", len(dl_df))

# ============================================================
# 2. CHUAN HOA VA GHEP KET QUA
# ============================================================

metrics = [
    "Dataset",
    "Model",
    "Val_Macro_F1",
    "Test_Macro_F1",
    "Test_Accuracy",
    "Test_Balanced_Accuracy"
]

for name, df in [("ML", ml_df), ("DL", dl_df)]:
    missing = set(metrics) - set(df.columns)
    if missing:
        raise ValueError(
            f"{name} thieu cot: {missing}"
        )

ml_df = ml_df[metrics].copy()
dl_df = dl_df[metrics].copy()

ml_df["Model_Type"] = "Machine Learning"
dl_df["Model_Type"] = "Deep Learning"

comparison = pd.concat(
    [ml_df, dl_df],
    ignore_index=True
)

# Loai ban ghi trung neu notebook da chay nhieu lan
comparison = comparison.drop_duplicates(
    subset=["Dataset", "Model"],
    keep="last"
)

# Xep hang theo Validation Macro-F1
# Khong su dung Test de chon model
comparison = comparison.sort_values(
    ["Dataset", "Val_Macro_F1"],
    ascending=[True, False]
).reset_index(drop=True)

# ============================================================
# 3. BAO CAO RIENG CHO TUNG DATASET
# ============================================================

best_models = []

for dataset in DATASETS:

    print("\n" + "=" * 70)
    print("DATASET:", dataset)
    print("=" * 70)

    subset = comparison[
        comparison["Dataset"] == dataset
    ].copy()

    subset.insert(
        0, "Rank",
        range(1, len(subset) + 1)
    )

    print("So mo hinh:", len(subset), "/9")

    if len(subset) != 9:
        print(
            "WARNING: Chua du 9 mo hinh!"
        )

    print("\n=== MODEL COMPARISON ===")

    display(
        subset[
            [
                "Rank",
                "Model",
                "Model_Type",
                "Val_Macro_F1",
                "Test_Macro_F1",
                "Test_Accuracy",
                "Test_Balanced_Accuracy"
            ]
        ].round(4)
    )

    # ========================================================
    # 4. TIM MO HINH TOT NHAT THEO VALIDATION
    # ========================================================

    if subset.empty:
        continue

    best_overall = subset.iloc[0]

    dl_only = subset[
        subset["Model_Type"] == "Deep Learning"
    ]

    ml_only = subset[
        subset["Model_Type"] == "Machine Learning"
    ]

    print("\n=== BEST OVERALL MODEL ===")
    print("Model:", best_overall["Model"])
    print(
        "Validation Macro-F1:",
        f"{best_overall['Val_Macro_F1']:.4f}"
    )
    print(
        "Test Macro-F1:",
        f"{best_overall['Test_Macro_F1']:.4f}"
    )

    best_dl_name = None
    best_ml_name = None

    if not dl_only.empty:
        best_dl = dl_only.iloc[0]
        best_dl_name = best_dl["Model"]

        print("\n=== BEST DEEP LEARNING ===")
        print("Model:", best_dl_name)
        print(
            "Test Macro-F1:",
            f"{best_dl['Test_Macro_F1']:.4f}"
        )

    if not ml_only.empty:
        best_ml = ml_only.iloc[0]
        best_ml_name = best_ml["Model"]

        print("\n=== BEST MACHINE LEARNING ===")
        print("Model:", best_ml_name)
        print(
            "Test Macro-F1:",
            f"{best_ml['Test_Macro_F1']:.4f}"
        )

    best_models.append({
        "Dataset": dataset,
        "Best_Overall": best_overall["Model"],
        "Best_Overall_Val_F1": best_overall["Val_Macro_F1"],
        "Best_Overall_Test_F1": best_overall["Test_Macro_F1"],
        "Best_ML": best_ml_name,
        "Best_DL": best_dl_name
    })

    # Luu bang rieng cua dataset
    output_path = (
        RESULT_DIR / dataset / "model_comparison.csv"
    )

    output_path.parent.mkdir(
        parents=True, exist_ok=True
    )

    subset.to_csv(
        output_path,
        index=False
    )

    print("\nSaved:", output_path)

# ============================================================
# 5. TONG HOP HAI DATASET
# ============================================================

best_df = pd.DataFrame(best_models)

print("\n" + "=" * 70)
print("FINAL BEST MODELS SUMMARY")
print("=" * 70)

display(best_df.round(4))

# ============================================================
# 6. BANG SO SANH TEST MACRO-F1
# ============================================================

print("\n=== TEST MACRO-F1 COMPARISON ===")

pivot_df = comparison.pivot(
    index="Model",
    columns="Dataset",
    values="Test_Macro_F1"
)

display(
    pivot_df.sort_index().round(4)
)

# ============================================================
# 7. LUU TAT CA KET QUA
# ============================================================

comparison.to_csv(
    RESULT_DIR / "model_comparison.csv",
    index=False
)

best_df.to_csv(
    RESULT_DIR / "best_models_summary.csv",
    index=False
)

pivot_df.to_csv(
    RESULT_DIR / "model_comparison_pivot.csv"
)

print("\n========================================")
print("CELL 19 COMPLETED")
print("========================================")

print("Total evaluated models:", len(comparison))
print("Total datasets:", comparison["Dataset"].nunique())

print("\nSaved:")
print("1. model_comparison.csv")
print("2. best_models_summary.csv")
print("3. model_comparison_pivot.csv")


=== KIEM TRA KET QUA ===
ML records: 8
DL records: 10

DATASET: CIC-IDS2017
So mo hinh: 9 /9

=== MODEL COMPARISON ===


,Rank,Model,Model_Type,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy
9,1,RandomForest,Machine Learning,0.9277,0.9233,0.9822,0.9242
10,2,XGBoost,Machine Learning,0.9269,0.9283,0.9846,0.9287
11,3,CNN-BiLSTM,Deep Learning,0.8675,0.8720,0.9547,0.9052
12,4,CNN-LSTM,Deep Learning,0.8627,0.8606,0.9449,0.8939
13,5,CNN1D,Deep Learning,0.8491,0.8550,0.9099,0.8871
14,6,LogisticRegression,Machine Learning,0.8436,0.8483,0.8978,0.9168
15,7,LinearSVM,Machine Learning,0.8373,0.8415,0.9157,0.8937
16,8,LSTM,Deep Learning,0.8372,0.8408,0.9231,0.8878
17,9,BiLSTM,Deep Learning,0.8101,0.8141,0.8979,0.8585



=== BEST OVERALL MODEL ===
Model: RandomForest
Validation Macro-F1: 0.9277
Test Macro-F1: 0.9233

=== BEST DEEP LEARNING ===
Model: CNN-BiLSTM
Test Macro-F1: 0.8720

=== BEST MACHINE LEARNING ===
Model: RandomForest
Test Macro-F1: 0.9233

Saved: /content/drive/MyDrive/NIDS_Project/results/CIC-IDS2017/model_comparison.csv

DATASET: CIC-DDoS2019
So mo hinh: 9 /9

=== MODEL COMPARISON ===


,Rank,Model,Model_Type,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy
0,1,XGBoost,Machine Learning,0.5830,0.5838,0.7699,0.5978
1,2,RandomForest,Machine Learning,0.5772,0.5769,0.7331,0.5751
2,3,CNN-BiLSTM,Deep Learning,0.4931,0.4935,0.7403,0.5296
3,4,CNN-LSTM,Deep Learning,0.4812,0.4826,0.7350,0.5285
4,5,BiLSTM,Deep Learning,0.4339,0.4266,0.7228,0.4770
5,6,LSTM,Deep Learning,0.4258,0.4232,0.7154,0.4652
6,7,CNN1D,Deep Learning,0.4217,0.4185,0.7180,0.4620
7,8,LogisticRegression,Machine Learning,0.3988,0.3925,0.6158,0.4800
8,9,LinearSVM,Machine Learning,0.3918,0.3834,0.6715,0.4368



=== BEST OVERALL MODEL ===
Model: XGBoost
Validation Macro-F1: 0.5830
Test Macro-F1: 0.5838

=== BEST DEEP LEARNING ===
Model: CNN-BiLSTM
Test Macro-F1: 0.4935

=== BEST MACHINE LEARNING ===
Model: XGBoost
Test Macro-F1: 0.5838

Saved: /content/drive/MyDrive/NIDS_Project/results/CIC-DDoS2019/model_comparison.csv

FINAL BEST MODELS SUMMARY


,Dataset,Best_Overall,Best_Overall_Val_F1,Best_Overall_Test_F1,Best_ML,Best_DL
0,CIC-IDS2017,RandomForest,0.9277,0.9233,RandomForest,CNN-BiLSTM
1,CIC-DDoS2019,XGBoost,0.5830,0.5838,XGBoost,CNN-BiLSTM



=== TEST MACRO-F1 COMPARISON ===


Dataset,CIC-DDoS2019,CIC-IDS2017
Model,,
BiLSTM,0.4266,0.8141
CNN-BiLSTM,0.4935,0.8720
CNN-LSTM,0.4826,0.8606
CNN1D,0.4185,0.8550
LSTM,0.4232,0.8408
LinearSVM,0.3834,0.8415
LogisticRegression,0.3925,0.8483
RandomForest,0.5769,0.9233
XGBoost,0.5838,0.9283



CELL 19 COMPLETED
Total evaluated models: 18
Total datasets: 2

Saved:
1. model_comparison.csv
2. best_models_summary.csv
3. model_comparison_pivot.csv


In [ ]:

# ============================================================
# CELL 19.1 | KHOI PHUC COLAB SAU KHI MAT KET NOI
# ============================================================

from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import json
import joblib
import gc
import torch

# 1. Ket noi lai Google Drive
drive.mount("/content/drive")

# 2. Khoi phuc cac duong dan
ROOT = Path("/content/drive/MyDrive/NIDS_Project")

READY_DIR = ROOT / "datasets" / "prepared" / "ready"

DATASETS = [
    "CIC-IDS2017",
    "CIC-DDoS2019"
]

# 3. Khoi phuc ham load_training_data
def load_training_data(dataset):

    folder = READY_DIR / dataset

    result = {}

    for split in ["train", "val", "test"]:

        with np.load(
            folder / f"{split}.npz",
            allow_pickle=False
        ) as data:

            result[f"X_{split}"] = data["X"].astype(
                np.float32
            )

            result[f"y_{split}"] = data["y"].astype(
                np.int64
            )

    mapping = json.loads(
        (folder / "label_mapping.json").read_text(
            encoding="utf-8"
        )
    )

    result["class_names"] = [
        mapping[str(i)] for i in range(len(mapping))
    ]

    result["n_features"] = result["X_train"].shape[1]
    result["n_classes"] = len(result["class_names"])

    return result


# 4. Kiem tra du lieu da luu
print("=== RECOVERY STATUS ===")

for dataset in DATASETS:

    folder = READY_DIR / dataset

    print(f"\n{dataset}")

    for filename in [
        "train.npz",
        "val.npz",
        "test.npz",
        "preprocessor.joblib",
        "label_mapping.json"
    ]:

        path = folder / filename

        status = "OK" if path.exists() else "MISSING"

        print(f"  {filename}: {status}")

# 5. Kiem tra checkpoint CNN-LSTM
for dataset in DATASETS:

    checkpoint = (
        ROOT / "models" / dataset /
        "DL" / "CNN-LSTM_best.pt"
    )

    print(
        f"\n{dataset} CNN-LSTM checkpoint:",
        "OK" if checkpoint.exists() else "MISSING"
    )

print("\nRECOVERY COMPLETED")


Mounted at /content/drive
=== RECOVERY STATUS ===

CIC-IDS2017
  train.npz: OK
  val.npz: OK
  test.npz: OK
  preprocessor.joblib: OK
  label_mapping.json: OK

CIC-DDoS2019
  train.npz: OK
  val.npz: OK
  test.npz: OK
  preprocessor.joblib: OK
  label_mapping.json: OK

CIC-IDS2017 CNN-LSTM checkpoint: OK

CIC-DDoS2019 CNN-LSTM checkpoint: OK

RECOVERY COMPLETED


In [ ]:

# ============================================================
# CELL 20 | GAUSSIAN ANOMALY DETECTION
# CIC-IDS2017 + CIC-DDOS2019
# ============================================================

# Thu vien moi can dung trong Cell 20
from sklearn.covariance import LedoitWolf
from sklearn.metrics import precision_recall_fscore_support

# Tai su dung:
# ROOT, DATASETS, pd, np, json, joblib
# load_training_data(), gc

GAUSSIAN_RESULTS = []

# Dat nguong theo phan vi 99% score cua
# BENIGN o tap Validation
THRESHOLD_PERCENTILE = 99.0

for dataset in DATASETS:

    print("\n" + "=" * 65)
    print("GAUSSIAN ANOMALY:", dataset)
    print("=" * 65)

    data = load_training_data(dataset)

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_val = data["X_val"]
    y_val = data["y_val"]

    X_test = data["X_test"]
    y_test = data["y_test"]

    class_names = data["class_names"]

    benign_id = class_names.index("BENIGN")

    # --------------------------------------------------------
    # 1. CHI LAY BENIGN DE FIT GAUSSIAN
    # --------------------------------------------------------

    X_train_normal = X_train[
        y_train == benign_id
    ]

    X_val_normal = X_val[
        y_val == benign_id
    ]

    print(
        "Train BENIGN:",
        len(X_train_normal)
    )

    print(
        "Validation BENIGN:",
        len(X_val_normal)
    )

    if len(X_train_normal) < 100:
        raise ValueError(
            "Khong du BENIGN de fit Gaussian!"
        )

    if len(X_val_normal) < 100:
        raise ValueError(
            "Khong du BENIGN de calibrate threshold!"
        )

    # --------------------------------------------------------
    # 2. FIT GAUSSIAN
    # --------------------------------------------------------

    # LedoitWolf uoc luong covariance
    # co shrinkage, phu hop khi feature tuong quan

    gaussian = LedoitWolf(
        assume_centered=False
    )

    gaussian.fit(X_train_normal)

    print("Gaussian fitted successfully!")

    # --------------------------------------------------------
    # 3. TINH ANOMALY SCORE
    # --------------------------------------------------------

    # Score cao -> cang bat thuong
    # Dung squared Mahalanobis distance

    val_normal_scores = gaussian.mahalanobis(
        X_val_normal
    )

    test_scores = gaussian.mahalanobis(
        X_test
    )

    # Kiem tra diem so hop le
    assert np.isfinite(val_normal_scores).all()
    assert np.isfinite(test_scores).all()

    # --------------------------------------------------------
    # 4. CHON THRESHOLD TREN VALIDATION BENIGN
    # --------------------------------------------------------

    threshold = float(
        np.percentile(
            val_normal_scores,
            THRESHOLD_PERCENTILE
        )
    )

    print(
        f"Threshold P{THRESHOLD_PERCENTILE}:",
        threshold
    )

    # --------------------------------------------------------
    # 5. DU DOAN BINARY TREN TEST
    # --------------------------------------------------------

    # 0 = BENIGN
    # 1 = Attack / Anomaly

    y_test_binary = (
        y_test != benign_id
    ).astype(int)

    y_pred_binary = (
        test_scores > threshold
    ).astype(int)

    # --------------------------------------------------------
    # 6. DANH GIA
    # --------------------------------------------------------

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test_binary,
            y_pred_binary,
            average="binary",
            zero_division=0
        )
    )

    normal_mask = y_test_binary == 0
    attack_mask = y_test_binary == 1

    false_positives = int(
        y_pred_binary[normal_mask].sum()
    )

    normal_count = int(normal_mask.sum())

    fpr = (
        false_positives / normal_count
        if normal_count else 0
    )

    attack_recall = float(
        y_pred_binary[attack_mask].mean()
    ) if attack_mask.any() else 0.0

    print("\n=== TEST RESULTS ===")
    print(f"Precision     : {precision:.4f}")
    print(f"Attack Recall : {attack_recall:.4f}")
    print(f"Binary F1     : {f1:.4f}")
    print(f"FPR           : {fpr:.4f}")

    # --------------------------------------------------------
    # 7. PHAT HIEN TUNG LOP TAN CONG
    # --------------------------------------------------------

    class_results = []

    for class_id, class_name in enumerate(class_names):

        mask = y_test == class_id

        detection_rate = float(
            y_pred_binary[mask].mean()
        ) if mask.any() else 0.0

        class_results.append({
            "Class": class_name,
            "Test_Samples": int(mask.sum()),
            "Alert_Rate": detection_rate
        })

    class_df = pd.DataFrame(class_results)

    print("\n=== DETECTION RATE PER CLASS ===")
    display(
        class_df.sort_values(
            "Alert_Rate",
            ascending=True
        ).round(4)
    )

    # --------------------------------------------------------
    # 8. LUU GAUSSIAN MODEL
    # --------------------------------------------------------

    model_dir = (
        ROOT / "models" / dataset / "Gaussian"
    )

    result_dir = (
        ROOT / "results" / dataset / "Gaussian"
    )

    model_dir.mkdir(
        parents=True, exist_ok=True
    )

    result_dir.mkdir(
        parents=True, exist_ok=True
    )

    model_path = (
        model_dir / "gaussian_ledoitwolf.joblib"
    )

    joblib.dump(
        {
            "model": gaussian,
            "threshold": threshold,
            "percentile": THRESHOLD_PERCENTILE,
            "benign_class_id": benign_id,
            "dataset": dataset
        },
        model_path
    )

    class_df.to_csv(
        result_dir / "per_class_detection.csv",
        index=False
    )

    record = {
        "Dataset": dataset,
        "Model": "Gaussian-LedoitWolf",
        "Threshold": threshold,
        "Precision": float(precision),
        "Attack_Recall": float(attack_recall),
        "Binary_F1": float(f1),
        "FPR": float(fpr)
    }

    (
        result_dir / "metrics.json"
    ).write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    GAUSSIAN_RESULTS.append(record)

    print("\nSaved Gaussian:", model_path)

    del data, gaussian
    gc.collect()

# ------------------------------------------------------------
# 9. TONG HOP KET QUA
# ------------------------------------------------------------

gaussian_df = pd.DataFrame(
    GAUSSIAN_RESULTS
)

print("\n" + "=" * 65)
print("GAUSSIAN SUMMARY - BOTH DATASETS")
print("=" * 65)

display(gaussian_df.round(4))

gaussian_df.to_csv(
    ROOT / "results" / "gaussian_summary.csv",
    index=False
)

print("\nCELL 20 COMPLETED")



GAUSSIAN ANOMALY: CIC-IDS2017
Train BENIGN: 16784
Validation BENIGN: 3597
Gaussian fitted successfully!
Threshold P99.0: 263.76352576728857

=== TEST RESULTS ===
Precision     : 0.9634
Attack Recall : 0.1915
Binary F1     : 0.3195
FPR           : 0.0081

=== DETECTION RATE PER CLASS ===


,Class,Test_Samples,Alert_Rate
7,FTP-Patator,377,0.0000
9,SSH-Patator,359,0.0000
8,PortScan,450,0.0022
0,BENIGN,3596,0.0081
1,Bot,290,0.0172
11,WebAttack-XSS,97,0.0206
10,WebAttack-BruteForce,212,0.0519
3,DoS GoldenEye,424,0.2500
5,DoS Slowhttptest,430,0.3000
6,DoS slowloris,449,0.3185



Saved Gaussian: /content/drive/MyDrive/NIDS_Project/models/CIC-IDS2017/Gaussian/gaussian_ledoitwolf.joblib

GAUSSIAN ANOMALY: CIC-DDoS2019
Train BENIGN: 28862
Validation BENIGN: 6185
Gaussian fitted successfully!
Threshold P99.0: 317.4508865320874

=== TEST RESULTS ===
Precision     : 0.9740
Attack Recall : 0.2190
Binary F1     : 0.3576
FPR           : 0.0097

=== DETECTION RATE PER CLASS ===


,Class,Test_Samples,Alert_Rate
5,DrDoS_NetBIOS,450,0.0000
12,Portmap,450,0.0000
15,UDP,900,0.0000
14,TFTP,450,0.0000
17,WebDDoS,66,0.0000
11,NetBIOS,900,0.0011
8,DrDoS_UDP,450,0.0022
13,Syn,1350,0.0030
4,DrDoS_NTP,450,0.0044
0,BENIGN,6185,0.0097



Saved Gaussian: /content/drive/MyDrive/NIDS_Project/models/CIC-DDoS2019/Gaussian/gaussian_ledoitwolf.joblib

GAUSSIAN SUMMARY - BOTH DATASETS


,Dataset,Model,Threshold,Precision,Attack_Recall,Binary_F1,FPR
0,CIC-IDS2017,Gaussian-LedoitWolf,263.7635,0.9634,0.1915,0.3195,0.0081
1,CIC-DDoS2019,Gaussian-LedoitWolf,317.4509,0.9740,0.2190,0.3576,0.0097



CELL 20 COMPLETED


In [ ]:

# ============================================================
# CELL 21 | GAUSSIAN + CNN-LSTM PARALLEL EVALUATION
# CIC-IDS2017 & CIC-DDOS2019
# ============================================================

# Su dung lai torch, np, pd, joblib, ROOT, DATASETS,
# load_training_data va precision_recall_fscore_support
# tu cac Cell truoc.
# Khong can train lai.

# 1. Khoi phuc dung kien truc CNN-LSTM da train
class CNNLSTMReload(torch.nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.cnn = torch.nn.Sequential(
            torch.nn.Conv1d(1, 32, kernel_size=5, padding=2),
            torch.nn.BatchNorm1d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(2),
            torch.nn.Conv1d(32, 48, kernel_size=3, padding=1),
            torch.nn.BatchNorm1d(48),
            torch.nn.ReLU()
        )

        self.lstm = torch.nn.LSTM(
            input_size=48,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(64, n_classes)
        )

    def forward(self, x):
        z = self.cnn(x.unsqueeze(1))
        z = z.transpose(1, 2)
        z, _ = self.lstm(z)
        z = z[:, -1, :]
        return self.classifier(z)


# 2. Cau hinh
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 1024
results = []

print("Device:", DEVICE)


def binary_metrics(y_true, y_pred):
    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )
    )

    normal = y_true == 0
    attack = y_true == 1

    fp = int(y_pred[normal].sum())
    tp = int(y_pred[attack].sum())

    fpr = fp / max(int(normal.sum()), 1)

    return {
        "Precision": float(precision),
        "Attack_Recall": float(recall),
        "Binary_F1": float(f1),
        "FPR": float(fpr),
        "True_Positives": tp,
        "False_Positives": fp
    }


# 3. Chay rieng tung dataset
for dataset in DATASETS:

    print("\n" + "=" * 65)
    print("DATASET:", dataset)
    print("=" * 65)

    data = load_training_data(dataset)

    X_test = data["X_test"]
    y_test = data["y_test"]

    n_classes = data["n_classes"]
    benign_id = data["class_names"].index("BENIGN")

    y_binary = (y_test != benign_id).astype(int)

    # --------------------------------------------------------
    # A. LOAD CNN-LSTM CHECKPOINT
    # --------------------------------------------------------

    cnn_path = (
        ROOT / "models" / dataset /
        "DL" / "CNN-LSTM_best.pt"
    )

    checkpoint = torch.load(
        cnn_path,
        map_location=DEVICE,
        weights_only=True
    )

    assert checkpoint["n_classes"] == n_classes
    assert checkpoint["n_features"] == X_test.shape[1]

    cnn = CNNLSTMReload(n_classes).to(DEVICE)
    cnn.load_state_dict(checkpoint["model_state"])
    cnn.eval()

    predictions = []

    with torch.inference_mode():

        for start in range(0, len(X_test), BATCH_SIZE):
            batch = torch.from_numpy(
                X_test[start:start + BATCH_SIZE]
            ).to(DEVICE)

            logits = cnn(batch)

            predictions.append(
                logits.argmax(dim=1).cpu().numpy()
            )

    cnn_multiclass = np.concatenate(predictions)

    # Nhan BENIGN = 0, con lai = 1
    cnn_alert = (
        cnn_multiclass != benign_id
    ).astype(int)

    # --------------------------------------------------------
    # B. LOAD GAUSSIAN
    # --------------------------------------------------------

    gaussian_path = (
        ROOT / "models" / dataset /
        "Gaussian" / "gaussian_ledoitwolf.joblib"
    )

    gaussian_package = joblib.load(gaussian_path)

    gaussian_model = gaussian_package["model"]
    threshold = gaussian_package["threshold"]

    gaussian_scores = gaussian_model.mahalanobis(
        X_test
    )

    gaussian_alert = (
        gaussian_scores > threshold
    ).astype(int)

    # --------------------------------------------------------
    # C. KET HOP SONG SONG
    # --------------------------------------------------------

    or_alert = (
        cnn_alert | gaussian_alert
    ).astype(int)

    and_alert = (
        cnn_alert & gaussian_alert
    ).astype(int)

    methods = {
        "CNN-LSTM": cnn_alert,
        "Gaussian": gaussian_alert,
        "Parallel_OR": or_alert,
        "Parallel_AND": and_alert
    }

    dataset_results = []

    for method, y_pred in methods.items():
        metrics = binary_metrics(y_binary, y_pred)

        record = {
            "Dataset": dataset,
            "Method": method,
            **metrics
        }

        dataset_results.append(record)
        results.append(record)

    print("\n=== BINARY ATTACK DETECTION ===")

    display(
        pd.DataFrame(dataset_results).round(4)
    )

    # --------------------------------------------------------
    # D. GAUSSIAN CO PHAT HIEN THEM TAN CONG KHONG?
    # --------------------------------------------------------

    rescued_attacks = int(
        (
            (y_binary == 1)
            & (cnn_alert == 0)
            & (gaussian_alert == 1)
        ).sum()
    )

    extra_false_alarms = int(
        (
            (y_binary == 0)
            & (cnn_alert == 0)
            & (gaussian_alert == 1)
        ).sum()
    )

    print("\n=== GAUSSIAN CONTRIBUTION ===")
    print(
        "Attack CNN bo sot nhung Gaussian phat hien:",
        rescued_attacks
    )
    print(
        "Canh bao nham BENIGN bo sung boi Gaussian:",
        extra_false_alarms
    )

    # --------------------------------------------------------
    # E. LUU KET QUA CHI TIET
    # --------------------------------------------------------

    output_dir = (
        ROOT / "results" / dataset / "Ensemble"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    prediction_df = pd.DataFrame({
        "True_Class": y_test,
        "True_Attack": y_binary,
        "CNN_Predicted_Class": cnn_multiclass,
        "CNN_Alert": cnn_alert,
        "Gaussian_Score": gaussian_scores,
        "Gaussian_Alert": gaussian_alert,
        "Parallel_OR": or_alert,
        "Parallel_AND": and_alert
    })

    prediction_df.to_parquet(
        output_dir / "test_predictions.parquet",
        index=False
    )

    pd.DataFrame(dataset_results).to_csv(
        output_dir / "binary_comparison.csv",
        index=False
    )

    print("\nSaved:", output_dir)

    del (
        cnn, checkpoint, data,
        gaussian_model, gaussian_package,
        X_test, y_test
    )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# 4. Luu tong hop 2 dataset
final_df = pd.DataFrame(results)

final_df.to_csv(
    ROOT / "results" /
    "gaussian_cnnlstm_parallel_summary.csv",
    index=False
)

print("\n=== FINAL PARALLEL COMPARISON ===")
display(final_df.round(4))

print("\nCELL 21 COMPLETED")


NameError: name 'torch' is not defined

In [ ]:

# ============================================================
# CELL 22 | TEMPORAL FLOW & BEACONING EXPLORATION
# CIC-IDS2017 + CIC-DDOS2019
# ============================================================

# Chi import thu vien chua chac da co trong runtime
import zipfile
import subprocess

# Su dung lai ROOT, pd, np, Path da co
BEACON_DIR = ROOT / "results" / "Beaconing"
BEACON_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = 150_000
CHUNK_SIZE = 25_000
MIN_CONNECTIONS = 8
MAX_CV = 0.20

REQUIRED = [
    "Source IP",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]

def load_contiguous_flows(reader, max_rows=MAX_ROWS):
    parts = []
    total = 0

    for chunk in reader:
        chunk.columns = chunk.columns.str.strip()

        missing = set(REQUIRED) - set(chunk.columns)
        if missing:
            raise ValueError(f"Thieu cot: {missing}")

        remaining = max_rows - total
        if remaining <= 0:
            break

        chunk = chunk[REQUIRED].iloc[:remaining].copy()
        parts.append(chunk)
        total += len(chunk)

    if not parts:
        raise ValueError("Khong doc duoc flow!")

    return pd.concat(parts, ignore_index=True)


# ============================================================
# A. CIC-IDS2017 - DOC CSV GOC TU ZIP TREN DRIVE
# ============================================================

ids_zip = (
    ROOT / "datasets" / "CIC-IDS2017" /
    "CIC-IDS2017.zip"
)

if not ids_zip.exists():
    raise FileNotFoundError(
        f"Khong tim thay ZIP IDS2017: {ids_zip}"
    )

with zipfile.ZipFile(ids_zip) as zf:
    members = [
        x for x in zf.namelist()
        if x.lower().endswith(".csv")
    ]

    # Uu tien file Friday Morning (co luu luong Bot)
    selected = next(
        (
            x for x in members
            if "friday" in x.lower()
            and "morning" in x.lower()
        ),
        members[0]
    )

    print("IDS2017 source:", selected)

    with zf.open(selected) as stream:
        reader = pd.read_csv(
            stream,
            chunksize=CHUNK_SIZE,
            dtype=str,
            encoding="utf-8-sig",
            encoding_errors="replace",
            on_bad_lines="skip"
        )

        ids_flows = load_contiguous_flows(reader)



# ============================================================
# B. CIC-DDOS2019 | TAI VA DOC PORTMAP.CSV
# ============================================================

# Su dung lai: Path, subprocess, zipfile, pd
# Khong can cai them thu vien

DDOS_DIR = Path("/content/NIDS_Work/CIC-DDoS2019")
DDOS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = (
    "rodrigorosasilva/"
    "cic-ddos2019-30gb-full-dataset-csv-files"
)

FILE_NAME = "03-11/Portmap.csv"

# 1. Tim file da co truoc
candidates = sorted(
    DDOS_DIR.rglob("Portmap.csv")
)

# 2. Neu chua co, tai dung 1 file tu Kaggle
if not candidates:

    print("Dang tai Portmap.csv tu Kaggle...")

    result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d", DATASET_SLUG,
            "-f", FILE_NAME,
            "-p", str(DDOS_DIR),
            "--unzip"
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Kaggle download failed")

    # Tim lai sau khi tai
    candidates = sorted(
        DDOS_DIR.rglob("Portmap.csv")
    )

# 3. Xu ly truong hop Kaggle de lai file ZIP
if not candidates:

    zip_files = sorted(
        DDOS_DIR.rglob("*.zip")
    )

    for zip_path in zip_files:

        if not zipfile.is_zipfile(zip_path):
            continue

        with zipfile.ZipFile(zip_path) as zf:

            members = [
                m for m in zf.infolist()
                if Path(m.filename).name == "Portmap.csv"
                and not m.is_dir()
            ]

            for member in members:

                target = DDOS_DIR / "Portmap.csv"

                with zf.open(member) as src:
                    with open(target, "wb") as dst:
                        while True:
                            block = src.read(1024 * 1024)
                            if not block:
                                break
                            dst.write(block)

                print("Da giai nen:", target)

    candidates = sorted(
        DDOS_DIR.rglob("Portmap.csv")
    )

# 4. Kiem tra ket qua
if not candidates:

    print("\nCac file hien co:")

    for p in sorted(DDOS_DIR.rglob("*"))[:30]:
        if p.is_file():
            print(p, p.stat().st_size)

    raise FileNotFoundError(
        "Khong tim thay Portmap.csv sau khi tai. "
        "Hay kiem tra output Kaggle phia tren."
    )

ddos_path = candidates[0]

print("\n=== DOWNLOAD SUCCESS ===")
print("File:", ddos_path)
print(
    "Size:",
    round(ddos_path.stat().st_size / 1024**2, 2),
    "MB"
)

# 5. Doc 150.000 flow lien tiep
reader = pd.read_csv(
    ddos_path,
    chunksize=CHUNK_SIZE,
    dtype=str,
    encoding_errors="replace",
    on_bad_lines="skip"
)

ddos_flows = load_contiguous_flows(reader)

print("\nDDoS2019 loaded:", ddos_flows.shape)
print("Columns:", ddos_flows.columns.tolist())



# ============================================================
# C. PHAN TICH CHU KY KET NOI
# ============================================================

def analyze_beaconing(df, dataset):

    df = df.copy()

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        format="mixed",
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "Source IP",
            "Destination IP",
            "Timestamp"
        ]
    )

    group_cols = [
        "Source IP",
        "Destination IP",
        "Destination Port",
        "Protocol"
    ]

    df = df.sort_values(
        group_cols + ["Timestamp"]
    )

    # Khong tinh hai flow trung timestamp thanh hai chu ky
    df = df.drop_duplicates(
        subset=group_cols + ["Timestamp"]
    )

    candidates = []

    for key, group in df.groupby(group_cols):

        if len(group) < MIN_CONNECTIONS:
            continue

        intervals = (
            group["Timestamp"]
            .diff()
            .dt.total_seconds()
            .dropna()
        )

        intervals = intervals[
            intervals > 0
        ]

        if len(intervals) < MIN_CONNECTIONS - 1:
            continue

        mean_interval = intervals.mean()
        std_interval = intervals.std(ddof=0)

        if mean_interval <= 0:
            continue

        cv = std_interval / mean_interval

        # Chi xem cac chu ky tu 1 giay den 1 gio
        if not (1 <= mean_interval <= 3600):
            continue

        if cv <= MAX_CV:
            candidates.append({
                "Source_IP": key[0],
                "Destination_IP": key[1],
                "Destination_Port": key[2],
                "Protocol": key[3],
                "Connections": len(group),
                "Mean_Interval_Seconds": mean_interval,
                "Std_Interval_Seconds": std_interval,
                "CV": cv,
                "Most_Common_Label":
                    group["Label"].mode().iloc[0]
            })

    result = pd.DataFrame(candidates)

    print(f"\n=== {dataset} ===")
    print("Flows examined:", len(df))
    print("Beaconing candidates:", len(result))

    if not result.empty:
        result = result.sort_values(
            ["CV", "Connections"],
            ascending=[True, False]
        )

        display(result.head(15))

    output_path = (
        BEACON_DIR /
        f"{dataset}_beacon_candidates.csv"
    )

    result.to_csv(output_path, index=False)

    print("Saved:", output_path)

    return result


ids_candidates = analyze_beaconing(
    ids_flows, "CIC-IDS2017"
)

ddos_candidates = analyze_beaconing(
    ddos_flows, "CIC-DDoS2019"
)

print("\nCELL 22 COMPLETED")


IDS2017 source: Friday-WorkingHours-Morning.pcap_ISCX.csv
Dang tai Portmap.csv tu Kaggle...
Dataset URL: https://www.kaggle.com/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files
License(s): other
Portmap.csv.zip: Skipping, found more recently modified local copy (use --force to force download)

Da giai nen: /content/NIDS_Work/CIC-DDoS2019/Portmap.csv

=== DOWNLOAD SUCCESS ===
File: /content/NIDS_Work/CIC-DDoS2019/Portmap.csv
Size: 74.97 MB

DDoS2019 loaded: (150000, 6)
Columns: ['Source IP', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Label']

=== CIC-IDS2017 ===
Flows examined: 47244
Beaconing candidates: 53


,Source_IP,Destination_IP,Destination_Port,Protocol,Connections,Mean_Interval_Seconds,Std_Interval_Seconds,CV,Most_Common_Label
29,192.168.10.3,192.168.10.1,53,17,149,60.0,0.0,0.0,BENIGN
47,192.168.10.51,162.213.33.50,443,6,90,60.0,0.0,0.0,BENIGN
28,192.168.10.25,192.168.10.50,139,6,50,180.0,0.0,0.0,BENIGN
44,192.168.10.50,192.168.10.25,137,17,29,300.0,0.0,0.0,BENIGN
33,192.168.10.5,205.185.216.42,443,6,28,60.0,0.0,0.0,BENIGN
35,192.168.10.5,34.199.20.76,443,6,28,60.0,0.0,0.0,BENIGN
8,192.168.10.15,200.147.68.17,80,6,15,60.0,0.0,0.0,BENIGN
9,192.168.10.15,23.21.34.180,8053,6,13,120.0,0.0,0.0,BENIGN
13,192.168.10.15,54.210.10.124,443,6,13,120.0,0.0,0.0,BENIGN
14,192.168.10.16,146.148.100.4,443,6,13,120.0,0.0,0.0,BENIGN


Saved: /content/drive/MyDrive/NIDS_Project/results/Beaconing/CIC-IDS2017_beacon_candidates.csv

=== CIC-DDoS2019 ===
Flows examined: 149999
Beaconing candidates: 4


,Source_IP,Destination_IP,Destination_Port,Protocol,Connections,Mean_Interval_Seconds,Std_Interval_Seconds,CV,Most_Common_Label
1,192.168.50.253,224.0.0.5,0,0,22,123.436844,0.936026,0.007583,BENIGN
2,192.168.50.254,224.0.0.5,0,0,22,123.868684,1.112370,0.008980,BENIGN
0,172.16.0.5,192.168.50.4,0,0,21,126.312880,6.982731,0.055281,Portmap
3,8.6.0.1,8.0.6.4,0,0,18,143.982151,18.125931,0.125890,BENIGN


Saved: /content/drive/MyDrive/NIDS_Project/results/Beaconing/CIC-DDoS2019_beacon_candidates.csv

CELL 22 COMPLETED


In [ ]:

# ============================================================
# CELL 23 | GRAPH-BASED NETWORK ANALYSIS
# CIC-IDS2017 + CIC-DDOS2019
# ============================================================

from ipaddress import ip_address

GRAPH_DIR = ROOT / "results" / "Graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# Nguong tham do ban dau, chua duoc hieu chinh
MIN_HUB_DEGREE = 5
MIN_P2P_PEERS = 3
MIN_RECIPROCAL_RATIO = 0.5

GRAPH_COLUMNS = [
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label"
]

def is_valid_endpoint(value):
    """Loai IP khong hop le va dia chi multicast."""
    try:
        ip = ip_address(str(value).strip())
        return not (
            ip.is_multicast
            or ip.is_unspecified
            or ip.is_loopback
            or str(ip) == "255.255.255.255"
        )
    except ValueError:
        return False


def analyze_graph(input_df, dataset):

    print("\n" + "=" * 65)
    print("GRAPH ANALYSIS:", dataset)
    print("=" * 65)

    df = input_df[GRAPH_COLUMNS].copy()

    # 1. Lam sach metadata
    df = df.dropna(subset=GRAPH_COLUMNS[:3])

    df["Source IP"] = df["Source IP"].astype(str).str.strip()
    df["Destination IP"] = (
        df["Destination IP"].astype(str).str.strip()
    )

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        format="mixed",
        errors="coerce"
    )

    df = df.dropna(subset=["Timestamp"])

    # Khong xem tu ket noi voi chinh IP la mot edge
    df = df[
        df["Source IP"] != df["Destination IP"]
    ].copy()

    # 2. Loc cac dia chi khong hop le / multicast
    unique_ips = pd.unique(
        pd.concat([
            df["Source IP"],
            df["Destination IP"]
        ], ignore_index=True)
    )

    valid_map = {
        ip: is_valid_endpoint(ip)
        for ip in unique_ips
    }

    df = df[
        df["Source IP"].map(valid_map)
        & df["Destination IP"].map(valid_map)
    ].copy()

    if df.empty:
        print("Khong co flow hop le.")
        return None

    # 3. Tao graph co huong:
    # Source IP -> Destination IP
    edges = (
        df.groupby(
            ["Source IP", "Destination IP"]
        )
        .agg(
            Flow_Count=("Label", "size"),
            First_Seen=("Timestamp", "min"),
            Last_Seen=("Timestamp", "max")
        )
        .reset_index()
    )

    # 4. Tinh Out-degree va In-degree
    out_stats = (
        edges.groupby("Source IP")
        .agg(
            Out_Degree=("Destination IP", "nunique"),
            Out_Flows=("Flow_Count", "sum")
        )
        .rename_axis("IP")
        .reset_index()
    )

    in_stats = (
        edges.groupby("Destination IP")
        .agg(
            In_Degree=("Source IP", "nunique"),
            In_Flows=("Flow_Count", "sum")
        )
        .rename_axis("IP")
        .reset_index()
    )

    nodes = pd.DataFrame({
        "IP": sorted(
            set(edges["Source IP"])
            | set(edges["Destination IP"])
        )
    })

    nodes = nodes.merge(out_stats, on="IP", how="left")
    nodes = nodes.merge(in_stats, on="IP", how="left")

    for col in [
        "Out_Degree", "In_Degree",
        "Out_Flows", "In_Flows"
    ]:
        nodes[col] = nodes[col].fillna(0).astype(int)

    # 5. Tinh reciprocal connections
    edge_pairs = set(zip(
        edges["Source IP"],
        edges["Destination IP"]
    ))

    reciprocal_counts = {}

    reciprocal_edges = 0

    for src, dst in edge_pairs:
        if (dst, src) in edge_pairs:
            reciprocal_edges += 1
            reciprocal_counts[src] = (
                reciprocal_counts.get(src, 0) + 1
            )

    nodes["Mutual_Peers"] = (
        nodes["IP"]
        .map(reciprocal_counts)
        .fillna(0)
        .astype(int)
    )

    nodes["Reciprocal_Ratio"] = (
        nodes["Mutual_Peers"]
        / nodes["Out_Degree"].clip(lower=1)
    )

    # 6. Tim central hubs (nhieu nguon -> 1 dich)
    hubs = nodes[
        nodes["In_Degree"] >= MIN_HUB_DEGREE
    ].sort_values(
        ["In_Degree", "In_Flows"],
        ascending=False
    )

    # 7. Tim P2P-like nodes
    p2p = nodes[
        (nodes["Out_Degree"] >= MIN_P2P_PEERS)
        & (nodes["Mutual_Peers"] >= MIN_P2P_PEERS)
        & (
            nodes["Reciprocal_Ratio"]
            >= MIN_RECIPROCAL_RATIO
        )
    ].sort_values(
        ["Mutual_Peers", "Out_Degree"],
        ascending=False
    )

    # 8. Thong ke nhan trong raw excerpt
    print("\n=== LABEL DISTRIBUTION ===")
    print(df["Label"].value_counts().head(15))

    print("\n=== GRAPH SUMMARY ===")
    print("Flows:", f"{len(df):,}")
    print("Nodes:", f"{len(nodes):,}")
    print("Directed edges:", f"{len(edges):,}")

    reciprocity = (
        reciprocal_edges / len(edge_pairs)
        if edge_pairs else 0
    )

    print("Edge reciprocity:", round(reciprocity, 4))
    print("Hub candidates:", len(hubs))
    print("P2P-like nodes:", len(p2p))

    print("\n=== TOP CENTRAL HUB CANDIDATES ===")
    display(
        hubs[
            ["IP", "In_Degree", "In_Flows"]
        ].head(15)
    )

    print("\n=== TOP P2P-LIKE CANDIDATES ===")
    display(
        p2p[
            [
                "IP",
                "Out_Degree",
                "In_Degree",
                "Mutual_Peers",
                "Reciprocal_Ratio"
            ]
        ].head(15)
    )

    # 9. Luu ket qua
    output_dir = GRAPH_DIR / dataset
    output_dir.mkdir(parents=True, exist_ok=True)

    nodes.to_csv(
        output_dir / "nodes.csv",
        index=False
    )

    edges.to_parquet(
        output_dir / "edges.parquet",
        index=False
    )

    hubs.to_csv(
        output_dir / "hub_candidates.csv",
        index=False
    )

    p2p.to_csv(
        output_dir / "p2p_candidates.csv",
        index=False
    )

    summary = {
        "Dataset": dataset,
        "Flows": len(df),
        "Nodes": len(nodes),
        "Directed_Edges": len(edges),
        "Reciprocity": reciprocity,
        "Hub_Candidates": len(hubs),
        "P2P_Candidates": len(p2p)
    }

    print("\nSaved:", output_dir)

    return summary


# ============================================================
# CHAY CHO CA HAI DATASET
# ============================================================

# Luu lai 2 excerpt goc de khong phai tai lai
# neu Colab khoi dong lai o cac buoc sau.

for name, variable in [
    ("CIC-IDS2017", "ids_flows"),
    ("CIC-DDoS2019", "ddos_flows")
]:
    backup_path = GRAPH_DIR / f"{name}_raw_excerpt.parquet"

    if variable in globals():
        globals()[variable].to_parquet(
            backup_path, index=False
        )
    elif backup_path.exists():
        globals()[variable] = pd.read_parquet(
            backup_path
        )
    else:
        raise RuntimeError(
            f"Thieu {variable}. Hay chay lai Cell 22."
        )

graph_results = []

for name, data in [
    ("CIC-IDS2017", ids_flows),
    ("CIC-DDoS2019", ddos_flows)
]:
    result = analyze_graph(data, name)

    if result is not None:
        graph_results.append(result)

graph_summary_df = pd.DataFrame(graph_results)

print("\n=== GRAPH SUMMARY - BOTH DATASETS ===")
display(graph_summary_df)

graph_summary_df.to_csv(
    GRAPH_DIR / "graph_summary.csv",
    index=False
)

print("\nCELL 23 COMPLETED")



GRAPH ANALYSIS: CIC-IDS2017

=== LABEL DISTRIBUTION ===
Label
BENIGN    148241
Bot         1587
Name: count, dtype: int64

=== GRAPH SUMMARY ===
Flows: 149,828
Nodes: 4,667
Directed edges: 14,927
Edge reciprocity: 0.8848
Hub candidates: 323
P2P-like nodes: 609

=== TOP CENTRAL HUB CANDIDATES ===


,IP,In_Degree,In_Flows
1405,192.168.10.9,863,2909
1392,192.168.10.12,837,2795
1394,192.168.10.15,770,2697
1401,192.168.10.5,768,3717
1396,192.168.10.17,679,1939
1404,192.168.10.8,604,1870
1397,192.168.10.19,552,2051
1393,192.168.10.14,524,1784
1395,192.168.10.16,523,1760
1398,192.168.10.25,467,2133



=== TOP P2P-LIKE CANDIDATES ===


,IP,Out_Degree,In_Degree,Mutual_Peers,Reciprocal_Ratio
1405,192.168.10.9,1097,863,863,0.786691
1392,192.168.10.12,1025,837,837,0.816585
1394,192.168.10.15,991,770,770,0.776993
1401,192.168.10.5,936,768,768,0.820513
1396,192.168.10.17,859,679,679,0.790454
1404,192.168.10.8,762,604,604,0.792651
1397,192.168.10.19,710,552,552,0.777465
1393,192.168.10.14,647,524,524,0.809892
1395,192.168.10.16,645,523,523,0.810853
1398,192.168.10.25,607,467,467,0.769357



Saved: /content/drive/MyDrive/NIDS_Project/results/Graph/CIC-IDS2017

GRAPH ANALYSIS: CIC-DDoS2019

=== LABEL DISTRIBUTION ===
Label
Portmap    145266
BENIGN       4662
Name: count, dtype: int64

=== GRAPH SUMMARY ===
Flows: 149,928
Nodes: 231
Directed edges: 591
Edge reciprocity: 0.9171
Hub candidates: 4
P2P-like nodes: 14

=== TOP CENTRAL HUB CANDIDATES ===


,IP,In_Degree,In_Flows
98,192.168.50.6,94,316
100,192.168.50.8,91,281
101,192.168.50.9,82,169
155,4.2.2.4,5,1478



=== TOP P2P-LIKE CANDIDATES ===


,IP,Out_Degree,In_Degree,Mutual_Peers,Reciprocal_Ratio
98,192.168.50.6,115,94,94,0.817391
100,192.168.50.8,101,91,90,0.891089
101,192.168.50.9,92,82,82,0.891304
97,192.168.50.4,8,4,4,0.500000
31,172.217.10.74,4,4,4,1.000000
183,52.85.90.206,4,4,4,1.000000
36,172.217.11.14,3,3,3,1.000000
50,172.217.12.206,3,3,3,1.000000
82,172.217.9.238,3,3,3,1.000000
85,173.194.204.109,3,3,3,1.000000



Saved: /content/drive/MyDrive/NIDS_Project/results/Graph/CIC-DDoS2019

=== GRAPH SUMMARY - BOTH DATASETS ===


,Dataset,Flows,Nodes,Directed_Edges,Reciprocity,Hub_Candidates,P2P_Candidates
0,CIC-IDS2017,149828,4667,14927,0.88484,323,609
1,CIC-DDoS2019,149928,231,591,0.91709,4,14



CELL 23 COMPLETED


In [ ]:

# ============================================================
# CELL 24 | TEMPORAL WINDOW READINESS AUDIT
# CIC-IDS2017 + CIC-DDOS2019
# ============================================================

# Su dung lai: ROOT, pd
# Khong import thu vien moi

TEMPORAL_DIR = ROOT / "results" / "Temporal"
TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_DIR = ROOT / "results" / "Graph"

WINDOW_SIZE = 8
STRIDE = 8

audit_results = []
label_results = []

for dataset in ["CIC-IDS2017", "CIC-DDoS2019"]:

    print("\n" + "=" * 65)
    print("TEMPORAL READINESS:", dataset)
    print("=" * 65)

    path = GRAPH_DIR / f"{dataset}_raw_excerpt.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"Thieu raw excerpt: {path}"
        )

    df = pd.read_parquet(path)

    required = [
        "Source IP",
        "Timestamp",
        "Label"
    ]

    missing = set(required) - set(df.columns)

    if missing:
        raise ValueError(
            f"{dataset} thieu: {missing}"
        )

    # 1. Chuan hoa timestamp va metadata
    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        format="mixed",
        errors="coerce"
    )

    df["Label"] = (
        df["Label"]
        .fillna("__MISSING__")
        .astype(str)
        .str.strip()
    )

    df = df.dropna(
        subset=["Source IP", "Timestamp"]
    ).copy()

    df = df[
        df["Label"] != "__MISSING__"
    ].copy()

    # 2. Sap xep theo host va thoi gian
    df = df.sort_values(
        ["Source IP", "Timestamp"],
        kind="mergesort"
    )

    total_windows = 0
    host_count = 0
    window_label_counts = {}

    # 3. Dem so window co the tao theo tung host
    # Window gom 8 flow, stride 8 (khong overlap)
    # Nhan la nhan flow CUOI CUNG trong window
    for source_ip, group in df.groupby(
        "Source IP", sort=False
    ):

        if len(group) < WINDOW_SIZE:
            continue

        host_count += 1

        endpoint_labels = group.iloc[
            WINDOW_SIZE - 1::STRIDE
        ]["Label"]

        total_windows += len(endpoint_labels)

        counts = endpoint_labels.value_counts()

        for label, count in counts.items():
            window_label_counts[label] = (
                window_label_counts.get(label, 0)
                + int(count)
            )

    # 4. Thong ke
    print("Raw flows:", f"{len(df):,}")
    print("Unique hosts:", df["Source IP"].nunique())
    print("Hosts with >= 8 flows:", host_count)
    print("Potential windows:", f"{total_windows:,}")
    print("Raw classes:", df["Label"].nunique())
    print("Window classes:", len(window_label_counts))

    class_df = pd.DataFrame([
        {
            "Dataset": dataset,
            "Class": label,
            "Windows": count
        }
        for label, count in window_label_counts.items()
    ])

    if not class_df.empty:
        class_df = class_df.sort_values(
            "Windows",
            ascending=False
        )

        print("\nWINDOW LABEL DISTRIBUTION:")
        display(class_df)

    class_df.to_csv(
        TEMPORAL_DIR / f"{dataset}_window_labels.csv",
        index=False
    )

    audit_results.append({
        "Dataset": dataset,
        "Raw_Flows": len(df),
        "Unique_Hosts": df["Source IP"].nunique(),
        "Eligible_Hosts": host_count,
        "Potential_Windows": total_windows,
        "Raw_Classes": df["Label"].nunique(),
        "Window_Classes": len(window_label_counts)
    })

    del df

# 5. Tong hop
audit_df = pd.DataFrame(audit_results)

print("\n" + "=" * 65)
print("TEMPORAL READINESS SUMMARY")
print("=" * 65)

display(audit_df)

audit_df.to_csv(
    TEMPORAL_DIR / "temporal_readiness.csv",
    index=False
)

print("\nCELL 24 COMPLETED")



TEMPORAL READINESS: CIC-IDS2017
Raw flows: 150,000
Unique hosts: 3831
Hosts with >= 8 flows: 606
Potential windows: 17,530
Raw classes: 2
Window classes: 2

WINDOW LABEL DISTRIBUTION:


,Dataset,Class,Windows
0,CIC-IDS2017,BENIGN,17334
1,CIC-IDS2017,Bot,196



TEMPORAL READINESS: CIC-DDoS2019
Raw flows: 150,000
Unique hosts: 206
Hosts with >= 8 flows: 27
Potential windows: 18,682
Raw classes: 2
Window classes: 2

WINDOW LABEL DISTRIBUTION:


,Dataset,Class,Windows
1,CIC-DDoS2019,Portmap,18154
0,CIC-DDoS2019,BENIGN,528



TEMPORAL READINESS SUMMARY


,Dataset,Raw_Flows,Unique_Hosts,Eligible_Hosts,Potential_Windows,Raw_Classes,Window_Classes
0,CIC-IDS2017,150000,3831,606,17530,2,2
1,CIC-DDoS2019,150000,206,27,18682,2,2



CELL 24 COMPLETED


In [ ]:

# ============================================================
# CELL 25 | BUILD TEMPORAL RAW BLOCKS - BOTH DATASETS
# ============================================================

import zipfile
import subprocess
import shutil

TEMP_ROOT = ROOT / "datasets/prepared/temporal_blocks"
EDA_ROOT = ROOT / "results/EDA"
READY_DIR = ROOT / "datasets/prepared/ready"

TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# Moi block giu 5000 dong lien tiep trong CSV goc.
# Lay toi da 10 block o cac vi tri khac nhau moi file.
ROWS_PER_BLOCK = 5000
MAX_BLOCKS_PER_FILE = 10

DATASET_SLUG = (
    "rodrigorosasilva/"
    "cic-ddos2019-30gb-full-dataset-csv-files"
)

META = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]


# ------------------------------------------------------------
# 1. CHUAN HOA NHAN
# ------------------------------------------------------------

def normalize_temporal_label(label, dataset):
    s = str(label).strip()
    low = s.lower()

    if low in ("", "nan", "none", "__missing__"):
        return "__MISSING__"

    if low == "benign":
        return "BENIGN"

    if dataset == "CIC-DDoS2019":
        if low.replace("-", "").replace("_", "") == "udplag":
            return "UDPLag"

    if dataset == "CIC-IDS2017":
        if low.startswith("web attack"):
            if "sql injection" in low:
                return "WebAttack-SQLInjection"
            if "brute force" in low:
                return "WebAttack-BruteForce"
            if "xss" in low:
                return "WebAttack-XSS"

    return s


# ------------------------------------------------------------
# 2. DOC BLOCK LIEN TIEP TU CSV
# ------------------------------------------------------------

def extract_temporal_blocks(
    source_stream, dataset, file_id, total_rows
):
    output_dir = TEMP_ROOT / dataset
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"{file_id}.parquet"
    report_path = output_dir / f"{file_id}_manifest.json"

    if output_path.exists() and report_path.exists():
        print("[SKIP - DA LUU]", file_id)
        return

    # Lay danh sach feature tu baseline
    # CHI lay ten feature, KHONG dung scaler da fit
    package = joblib.load(
        READY_DIR / dataset / "preprocessor.joblib"
    )

    feature_cols = package["feature_names"]

    mapping = json.loads(
        (READY_DIR / dataset / "label_mapping.json")
        .read_text(encoding="utf-8")
    )

    allowed_labels = set(mapping.values())

    required = {"Source IP", "Destination IP", "Timestamp", "Label"}
    wanted = set(META) | set(feature_cols)

    n_chunks = max(
        1,
        (int(total_rows) + ROWS_PER_BLOCK - 1)
        // ROWS_PER_BLOCK
    )

    selected = set(
        np.linspace(
            0,
            n_chunks - 1,
            num=min(MAX_BLOCKS_PER_FILE, n_chunks),
            dtype=int
        ).tolist()
    )

    reader = pd.read_csv(
        source_stream,
        chunksize=ROWS_PER_BLOCK,
        dtype=str,
        usecols=lambda c: c.strip() in wanted,
        encoding="utf-8-sig",
        encoding_errors="replace",
        on_bad_lines="skip"
    )

    blocks = []
    used_chunks = []

    for chunk_idx, raw in enumerate(reader):

        if chunk_idx > max(selected):
            break

        if chunk_idx not in selected:
            continue

        raw.columns = raw.columns.str.strip()

        missing_meta = required - set(raw.columns)
        if missing_meta:
            print("[SKIP - THIEU META]", file_id, missing_meta)
            return

        columns = list(dict.fromkeys(
            [c for c in META if c in raw.columns]
            + feature_cols
        ))

        block = raw.reindex(columns=columns).copy()

        # Chuan hoa nhan theo baseline
        block["Label"] = block["Label"].map(
            lambda x: normalize_temporal_label(x, dataset)
        )

        # Chi giu cac nhan thuoc bai toan 12/18 lop
        block = block[
            block["Label"].isin(allowed_labels)
        ].copy()

        if block.empty:
            continue

        # Timestamp thuc te
        block["Timestamp"] = pd.to_datetime(
            block["Timestamp"],
            format="mixed",
            errors="coerce"
        )

        block = block.dropna(
            subset=["Source IP", "Destination IP", "Timestamp"]
        )

        block = block[
            block["Source IP"].astype(str).str.strip().ne("")
            & block["Destination IP"].astype(str).str.strip().ne("")
        ].copy()

        if block.empty:
            continue

        # Convert feature sang float32
        numeric = block[feature_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        numeric = numeric.replace(
            [np.inf, -np.inf],
            np.nan
        ).clip(-1e12, 1e12)

        block[feature_cols] = numeric.astype(np.float32)

        # Dinh danh doan lien tiep:
        # KHONG ghep window giua hai block khac nhau
        block["__capture"] = file_id
        block["__block_id"] = f"{file_id}@{chunk_idx:06d}"

        blocks.append(block)
        used_chunks.append(chunk_idx)

    if not blocks:
        print("[WARNING] Khong co flow hop le:", file_id)
        return

    result = pd.concat(blocks, ignore_index=True)

    # Luu nhat quan theo block + timestamp
    result = result.sort_values(
        ["__block_id", "Source IP", "Timestamp"],
        kind="mergesort"
    ).reset_index(drop=True)

    # Luu file tam roi doi ten
    temp_path = output_dir / f"{file_id}.tmp.parquet"
    result.to_parquet(temp_path, index=False)
    temp_path.replace(output_path)

    manifest = {
        "dataset": dataset,
        "file_id": file_id,
        "source_rows": int(total_rows),
        "selected_rows": int(len(result)),
        "selected_chunks": used_chunks,
        "n_features": len(feature_cols),
        "label_counts": {
            str(k): int(v)
            for k, v in result["Label"].value_counts().items()
        }
    }

    report_path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(
        f"[SAVED] {file_id} | "
        f"Rows={len(result):,} | "
        f"Classes={result['Label'].nunique()}"
    )


# ------------------------------------------------------------
# 3. CIC-IDS2017 - DOC TRUC TIEP ZIP TU GOOGLE DRIVE
# ------------------------------------------------------------

ids_zip = (
    ROOT / "datasets/CIC-IDS2017/CIC-IDS2017.zip"
)

assert ids_zip.exists(), f"Khong tim thay {ids_zip}"

ids_reports = sorted(
    (EDA_ROOT / "CIC-IDS2017").glob("*_eda.json")
)

with zipfile.ZipFile(ids_zip) as zf:

    member_lookup = {
        Path(name).name: name
        for name in zf.namelist()
        if name.lower().endswith(".csv")
    }

    for report_path in ids_reports:

        report = json.loads(
            report_path.read_text(encoding="utf-8")
        )

        filename = report["file"]
        file_id = report_path.name.removesuffix("_eda.json")

        if filename not in member_lookup:
            print("[SKIP] File ZIP khong co:", filename)
            continue

        with zf.open(member_lookup[filename]) as stream:
            extract_temporal_blocks(
                stream,
                "CIC-IDS2017",
                file_id,
                report["rows_read"]
            )

print("\n=== IDS2017 TEMPORAL EXTRACTION DONE ===")


# ------------------------------------------------------------
# 4. CIC-DDOS2019 - TAI TUNG CSV ZIP TU KAGGLE
# ------------------------------------------------------------

ddos_reports = sorted(
    (EDA_ROOT / "CIC-DDoS2019").glob("*_eda.json")
)

for report_path in ddos_reports:

    report = json.loads(
        report_path.read_text(encoding="utf-8")
    )

    file_id = report_path.name.removesuffix("_eda.json")

    output_path = (
        TEMP_ROOT / "CIC-DDoS2019" / f"{file_id}.parquet"
    )

    manifest_path = (
        TEMP_ROOT / "CIC-DDoS2019" /
        f"{file_id}_manifest.json"
    )

    # Khong tai lai file da xu ly
    if output_path.exists() and manifest_path.exists():
        print("[SKIP - DA LUU]", file_id)
        continue

    # Vi du: 03-11_Portmap -> 03-11/Portmap.csv
    rel_path = file_id.replace("_", "/", 1) + ".csv"

    stage_dir = (
        Path("/content/NIDS_Work/temporal_download")
        / file_id
    )

    stage_dir.mkdir(parents=True, exist_ok=True)

    print("\nDownloading:", rel_path)

    result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d", DATASET_SLUG,
            "-f", rel_path,
            "-p", str(stage_dir)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Khong tai duoc {rel_path}. "
            "Kiem tra Kaggle API Token."
        )

    # Doc CSV truc tiep tu ZIP, khong giai nen
    zip_files = sorted(stage_dir.rglob("*.zip"))

    if zip_files:

        with zipfile.ZipFile(zip_files[0]) as zf:

            members = [
                m for m in zf.namelist()
                if Path(m).name == Path(rel_path).name
            ]

            if not members:
                raise FileNotFoundError(
                    f"ZIP khong co {rel_path}"
                )

            with zf.open(members[0]) as stream:
                extract_temporal_blocks(
                    stream,
                    "CIC-DDoS2019",
                    file_id,
                    report["rows_read"]
                )

    else:
        csv_files = sorted(stage_dir.rglob("*.csv"))

        if not csv_files:
            raise FileNotFoundError(
                f"Khong tim thay CSV/ZIP cho {rel_path}"
            )

        extract_temporal_blocks(
            csv_files[0],
            "CIC-DDoS2019",
            file_id,
            report["rows_read"]
        )

    # Don file tai tam, chi sau khi da luu thanh cong
    if output_path.exists() and manifest_path.exists():
        shutil.rmtree(stage_dir)

print("\n=== DDOS2019 TEMPORAL EXTRACTION DONE ===")


# ------------------------------------------------------------
# 5. KIEM TRA PHU NHAN SAU KHI TRICH XUAT
# ------------------------------------------------------------

for dataset in ["CIC-IDS2017", "CIC-DDoS2019"]:

    folder = TEMP_ROOT / dataset
    paths = sorted(folder.glob("*.parquet"))

    print("\n" + "=" * 60)
    print("DATASET:", dataset)
    print("Temporal files:", len(paths))

    counts = pd.Series(dtype="int64")

    for path in paths:
        labels = pd.read_parquet(
            path, columns=["Label"]
        )["Label"].value_counts()

        counts = counts.add(labels, fill_value=0)

    counts = counts.fillna(0).astype(int)

    mapping = json.loads(
        (
            READY_DIR / dataset / "label_mapping.json"
        ).read_text(encoding="utf-8")
    )

    expected = set(mapping.values())
    found = set(counts.index)

    print("Expected classes:", len(expected))
    print("Found classes:", len(found))
    print("Missing classes:", sorted(expected - found))

    display(counts.sort_values(ascending=False))

print("\nCELL 25 COMPLETED")


[SAVED] Friday-WorkingHours-Afternoon-DDos_pcap_ISCX | Rows=45,745 | Classes=2
[SAVED] Friday-WorkingHours-Afternoon-PortScan_pcap_ISCX | Rows=46,467 | Classes=2
[SAVED] Friday-WorkingHours-Morning_pcap_ISCX | Rows=46,033 | Classes=2
[SAVED] Monday-WorkingHours_pcap_ISCX | Rows=49,918 | Classes=1
[SAVED] Thursday-WorkingHours-Afternoon-Infilteration_pcap_ISCX | Rows=48,598 | Classes=1
[SAVED] Thursday-WorkingHours-Morning-WebAttacks_pcap_ISCX | Rows=20,000 | Classes=2
[SAVED] Tuesday-WorkingHours_pcap_ISCX | Rows=45,909 | Classes=3
[SAVED] Wednesday-workingHours_pcap_ISCX | Rows=47,703 | Classes=3

=== IDS2017 TEMPORAL EXTRACTION DONE ===

Downloading: 01-12/DrDoS_DNS.csv
[SAVED] 01-12_DrDoS_DNS | Rows=49,413 | Classes=2

Downloading: 01-12/DrDoS_LDAP.csv
[SAVED] 01-12_DrDoS_LDAP | Rows=46,542 | Classes=2

Downloading: 01-12/DrDoS_MSSQL.csv
[SAVED] 01-12_DrDoS_MSSQL | Rows=49,498 | Classes=2

Downloading: 01-12/DrDoS_NTP.csv
[SAVED] 01-12_DrDoS_NTP | Rows=47,007 | Classes=2

Downloadin

,0
Label,
BENIGN,279446
PortScan,27972
DDoS,23612
DoS Hulk,16865
SSH-Patator,686
DoS GoldenEye,640
FTP-Patator,598
Bot,396
WebAttack-BruteForce,158



DATASET: CIC-DDoS2019
Temporal files: 18
Expected classes: 18
Found classes: 18
Missing classes: []


,0
Label,
Syn,124491
UDP,52187
NetBIOS,50877
DrDoS_NetBIOS,49950
DrDoS_MSSQL,49436
DrDoS_DNS,48221
LDAP,48154
DrDoS_UDP,46801
DrDoS_LDAP,46503



CELL 25 COMPLETED


In [ ]:

# ============================================================
# CELL 25.3 | RECOVERY COLAB SAU KHI GAP MAY
# ============================================================

from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import json

# 1. Ket noi lai Google Drive
drive.mount("/content/drive")

# 2. Khoi phuc ROOT
ROOT = Path("/content/drive/MyDrive/NIDS_Project")

# 3. Khoi phuc danh sach dataset
DATASETS = [
    "CIC-IDS2017",
    "CIC-DDoS2019"
]

# 4. Kiem tra file Temporal da luu
print("\n=== RECOVERY STATUS ===")

for dataset in DATASETS:

    folder = (
        ROOT / "datasets/prepared/temporal_blocks"
        / dataset
    )

    files = list(folder.glob("*.parquet"))

    print(f"\n{dataset}")
    print("Folder exists:", folder.exists())
    print("Temporal files:", len(files))

    mapping_path = (
        ROOT / "datasets/prepared/ready"
        / dataset / "label_mapping.json"
    )

    print("Label mapping:", mapping_path.exists())

print("\nROOT:", ROOT)
print("\nRECOVERY COMPLETED")



Mounted at /content/drive

=== RECOVERY STATUS ===

CIC-IDS2017
Folder exists: True
Temporal files: 23
Label mapping: True

CIC-DDoS2019
Folder exists: True
Temporal files: 18
Label mapping: True

ROOT: /content/drive/MyDrive/NIDS_Project

RECOVERY COMPLETED


In [ ]:

# ============================================================
# CELL 26 | TEMPORAL WINDOW COVERAGE AUDIT
# CIC-IDS2017 + CIC-DDOS2019
# ============================================================

# Su dung lai ROOT, pd, json
# Khong can install / import lai

TEMP_ROOT = ROOT / "datasets/prepared/temporal_blocks"
OUT_DIR = ROOT / "results/Temporal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 8
MAX_WINDOW_SECONDS = 300

DATASETS = [
    "CIC-IDS2017",
    "CIC-DDoS2019"
]

audit_all = []

for dataset in DATASETS:

    print("\n" + "=" * 70)
    print("TEMPORAL AUDIT:", dataset)
    print("=" * 70)

    folder = TEMP_ROOT / dataset
    files = sorted(folder.glob("*.parquet"))

    mapping_path = (
        ROOT / "datasets/prepared/ready"
        / dataset / "label_mapping.json"
    )

    mapping = json.loads(
        mapping_path.read_text(encoding="utf-8")
    )

    class_names = list(mapping.values())

    # Thong ke theo class
    stats = {
        label: {
            "Flow_Count": 0,
            "Windows_Stride8": 0,
            "Valid_Windows_Stride8": 0,
            "Valid_Windows_Stride1": 0,
            "ZeroSpan_Windows": 0,
            "Flow_Blocks": set(),
            "Valid_Blocks": set()
        }
        for label in class_names
    }

    total_groups = 0
    processed_files = 0

    for path in files:

        cols = [
            "Source IP",
            "Timestamp",
            "Label",
            "__block_id"
        ]

        df = pd.read_parquet(
            path,
            columns=cols
        )

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )

        df["Source IP"] = (
            df["Source IP"].astype("string").str.strip()
        )

        df["Label"] = (
            df["Label"].astype("string").str.strip()
        )

        df = df.dropna(subset=cols)

        df = df[
            df["Source IP"].ne("")
            & df["Label"].isin(class_names)
        ].copy()

        if df.empty:
            continue

        # Flow count va so block chua tung nhan
        for label, part in df.groupby("Label"):
            stats[label]["Flow_Count"] += len(part)
            stats[label]["Flow_Blocks"].update(
                part["__block_id"].unique().tolist()
            )

        # Khong ghep flow giua hai block hoac hai host
        df = df.sort_values(
            [
                "__block_id",
                "Source IP",
                "Timestamp"
            ],
            kind="mergesort"
        ).reset_index(drop=True)

        groups = df.groupby(
            ["__block_id", "Source IP"],
            sort=False
        )

        total_groups += groups.ngroups

        # Vi tri cua flow trong tung host/block
        position = groups.cumcount()

        # Timestamp cua flow dau tien trong window
        start_time = groups["Timestamp"].shift(
            WINDOW_SIZE - 1
        )

        span_seconds = (
            df["Timestamp"] - start_time
        ).dt.total_seconds()

        # Stride 1: co the overlap, chi de audit
        eligible_stride1 = (
            position >= WINDOW_SIZE - 1
        )

        # Stride 8: khong overlap trong moi host/block
        eligible_stride8 = (
            eligible_stride1
            & (
                (position - (WINDOW_SIZE - 1))
                % WINDOW_SIZE == 0
            )
        )

        # Dieu kien thoi gian tham do ban dau:
        # 0 < span <= 300 giay
        valid_time = (
            (span_seconds > 0)
            & (span_seconds <= MAX_WINDOW_SECONDS)
        )

        valid_stride1 = (
            eligible_stride1 & valid_time
        )

        valid_stride8 = (
            eligible_stride8 & valid_time
        )

        zero_span = (
            eligible_stride8
            & (span_seconds == 0)
        )

        masks = {
            "Windows_Stride8": eligible_stride8,
            "Valid_Windows_Stride8": valid_stride8,
            "Valid_Windows_Stride1": valid_stride1,
            "ZeroSpan_Windows": zero_span
        }

        for metric, mask in masks.items():

            counts = df.loc[
                mask, "Label"
            ].value_counts()

            for label, count in counts.items():
                stats[label][metric] += int(count)

        # So block tao duoc temporal window hop le
        valid_rows = df.loc[
            valid_stride8,
            ["Label", "__block_id"]
        ].drop_duplicates()

        for label, part in valid_rows.groupby("Label"):
            stats[label]["Valid_Blocks"].update(
                part["__block_id"].tolist()
            )

        processed_files += 1

    # ========================================================
    # TONG HOP THEO CLASS
    # ========================================================

    records = []

    for label in class_names:
        s = stats[label]

        records.append({
            "Dataset": dataset,
            "Class": label,
            "Flow_Count": s["Flow_Count"],
            "Flow_Blocks": len(s["Flow_Blocks"]),
            "Windows_Stride8": s["Windows_Stride8"],
            "Valid_Windows_Stride8":
                s["Valid_Windows_Stride8"],
            "Valid_Windows_Stride1":
                s["Valid_Windows_Stride1"],
            "ZeroSpan_Windows":
                s["ZeroSpan_Windows"],
            "Valid_Blocks": len(s["Valid_Blocks"])
        })

    result_df = pd.DataFrame(records)

    audit_all.append(result_df)

    print("Processed files:", processed_files)
    print("Host-block groups:", total_groups)

    print("\n=== WINDOW COVERAGE PER CLASS ===")

    display(
        result_df[
            [
                "Class",
                "Flow_Count",
                "Flow_Blocks",
                "Valid_Windows_Stride8",
                "Valid_Windows_Stride1",
                "Valid_Blocks"
            ]
        ].sort_values(
            "Valid_Windows_Stride8"
        )
    )

    # Canh bao cac lop kho tao tap danh gia doc lap
    problematic = result_df[
        (result_df["Valid_Windows_Stride8"] == 0)
        | (result_df["Valid_Blocks"] < 3)
    ]

    print("\n=== CLASSES REQUIRING ATTENTION ===")

    if problematic.empty:
        print("No immediate coverage warnings.")
    else:
        display(
            problematic[
                [
                    "Class",
                    "Flow_Count",
                    "Valid_Windows_Stride8",
                    "Valid_Blocks"
                ]
            ]
        )

    output_path = (
        OUT_DIR / f"{dataset}_window_audit.csv"
    )

    result_df.to_csv(
        output_path,
        index=False
    )

    print("Saved:", output_path)


# ============================================================
# FINAL AUDIT
# ============================================================

final_audit = pd.concat(
    audit_all,
    ignore_index=True
)

final_audit.to_csv(
    OUT_DIR / "all_datasets_window_audit.csv",
    index=False
)

summary = (
    final_audit.groupby("Dataset")
    .agg(
        Classes=("Class", "count"),
        Total_Flows=("Flow_Count", "sum"),
        Windows_Stride8=(
            "Valid_Windows_Stride8", "sum"
        ),
        Windows_Stride1=(
            "Valid_Windows_Stride1", "sum"
        ),
        Classes_With_Windows=(
            "Valid_Windows_Stride8",
            lambda s: int((s > 0).sum())
        ),
        Classes_With_3_Blocks=(
            "Valid_Blocks",
            lambda s: int((s >= 3).sum())
        )
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("TEMPORAL WINDOW AUDIT SUMMARY")
print("=" * 70)

display(summary)

print("\nCELL 26 COMPLETED")



TEMPORAL AUDIT: CIC-IDS2017
Processed files: 23
Host-block groups: 23117

=== WINDOW COVERAGE PER CLASS ===


,Class,Flow_Count,Flow_Blocks,Valid_Windows_Stride8,Valid_Windows_Stride1,Valid_Blocks
3,DoS GoldenEye,640,3,1,7,1
8,PortScan,27972,7,1,14,1
4,DoS Hulk,17005,5,2,14,2
2,DDoS,23612,7,2,14,2
10,WebAttack-BruteForce,158,1,2,28,1
7,FTP-Patator,598,1,4,35,1
9,SSH-Patator,686,5,7,54,3
1,Bot,396,8,16,150,5
5,DoS Slowhttptest,5499,2,17,127,2
11,WebAttack-XSS,652,4,17,136,4



=== CLASSES REQUIRING ATTENTION ===


,Class,Flow_Count,Valid_Windows_Stride8,Valid_Blocks
2,DDoS,23612,2,2
3,DoS GoldenEye,640,1,1
4,DoS Hulk,17005,2,2
5,DoS Slowhttptest,5499,17,2
7,FTP-Patator,598,4,1
8,PortScan,27972,1,1
10,WebAttack-BruteForce,158,2,1


Saved: /content/drive/MyDrive/NIDS_Project/results/Temporal/CIC-IDS2017_window_audit.csv

TEMPORAL AUDIT: CIC-DDoS2019
Processed files: 18
Host-block groups: 1312

=== WINDOW COVERAGE PER CLASS ===


,Class,Flow_Count,Flow_Blocks,Valid_Windows_Stride8,Valid_Windows_Stride1,Valid_Blocks
17,WebDDoS,76,2,9,75,1
0,BENIGN,16153,74,1648,12801,22
4,DrDoS_NTP,42413,10,5292,42307,10
12,Portmap,42985,10,5363,42902,10
16,UDPLag,44830,10,5600,44751,10
10,MSSQL,45778,10,5716,45703,10
14,TFTP,46065,10,5753,45982,10
7,DrDoS_SSDP,46346,10,5785,46265,10
6,DrDoS_SNMP,46371,10,5790,46295,10
2,DrDoS_LDAP,46503,10,5804,46416,10



=== CLASSES REQUIRING ATTENTION ===


,Class,Flow_Count,Valid_Windows_Stride8,Valid_Blocks
17,WebDDoS,76,9,1


Saved: /content/drive/MyDrive/NIDS_Project/results/Temporal/CIC-DDoS2019_window_audit.csv

TEMPORAL WINDOW AUDIT SUMMARY


,Dataset,Classes,Total_Flows,Windows_Stride8,Windows_Stride1,Classes_With_Windows,Classes_With_3_Blocks
0,CIC-DDoS2019,18,847637,105455,842802,18,17
1,CIC-IDS2017,12,425373,7981,60837,12,5



CELL 26 COMPLETED


In [ ]:

# ============================================================
# CELL 27 | TEMPORAL GROUPING STRATEGY AUDIT
# SOURCE-CENTRIC VS DESTINATION-CENTRIC
# ============================================================

# Su dung lai ROOT, pd, json
# Khong can import / install lai

TEMP_ROOT = ROOT / "datasets/prepared/temporal_blocks"
OUT_DIR = ROOT / "results/Temporal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 8
STRIDE = 8
MAX_WINDOW_SECONDS = 300

DATASETS = ["CIC-IDS2017", "CIC-DDoS2019"]

STRATEGIES = {
    "Source": ["__block_id", "Source IP"],
    "Destination": ["__block_id", "Destination IP"]
}

all_results = []

for dataset in DATASETS:

    print("\n" + "=" * 75)
    print("GROUPING AUDIT:", dataset)
    print("=" * 75)

    folder = TEMP_ROOT / dataset
    files = sorted(folder.glob("*.parquet"))

    mapping_path = (
        ROOT / "datasets/prepared/ready" /
        dataset / "label_mapping.json"
    )

    mapping = json.loads(
        mapping_path.read_text(encoding="utf-8")
    )

    classes = list(mapping.values())

    stats = {
        mode: {
            label: {
                "Valid_Windows": 0,
                "Homogeneous_Windows": 0,
                "Blocks": set(),
                "Captures": set()
            }
            for label in classes
        }
        for mode in STRATEGIES
    }

    for path in files:

        df = pd.read_parquet(
            path,
            columns=[
                "__block_id",
                "__capture",
                "Source IP",
                "Destination IP",
                "Timestamp",
                "Label"
            ]
        )

        df = df.dropna(
            subset=[
                "__block_id",
                "__capture",
                "Source IP",
                "Destination IP",
                "Timestamp",
                "Label"
            ]
        ).copy()

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            format="mixed",
            errors="coerce"
        )

        df = df.dropna(subset=["Timestamp"])

        df = df[
            df["Label"].isin(classes)
        ].copy()

        if df.empty:
            continue

        for mode, group_cols in STRATEGIES.items():

            work = df.sort_values(
                group_cols + ["Timestamp"],
                kind="mergesort"
            ).reset_index(drop=True)

            groups = work.groupby(
                group_cols, sort=False
            )

            position = groups.cumcount()

            start_time = groups["Timestamp"].shift(
                WINDOW_SIZE - 1
            )

            span = (
                work["Timestamp"] - start_time
            ).dt.total_seconds()

            # Window 8 flow, khong overlap
            eligible = (
                (position >= WINDOW_SIZE - 1)
                & (
                    (position - WINDOW_SIZE + 1)
                    % STRIDE == 0
                )
            )

            valid = (
                eligible
                & span.gt(0)
                & span.le(MAX_WINDOW_SECONDS)
            )

            # Kiem tra co thay doi nhan ben trong
            # window hay khong.
            # Chi dung nhan de KIEM TOAN,
            # KHONG dung nhan de tao window.
            changes = work["Label"].ne(
                groups["Label"].shift(1)
            )

            run_id = changes.cumsum()

            start_run = run_id.groupby(
                [work[c] for c in group_cols],
                sort=False
            ).shift(WINDOW_SIZE - 1)

            homogeneous = (
                valid
                & run_id.eq(start_run)
            )

            valid_rows = work.loc[
                valid,
                ["Label", "__block_id", "__capture"]
            ]

            pure_counts = work.loc[
                homogeneous, "Label"
            ].value_counts()

            for label, part in valid_rows.groupby("Label"):

                item = stats[mode][label]

                item["Valid_Windows"] += len(part)

                item["Homogeneous_Windows"] += int(
                    pure_counts.get(label, 0)
                )

                item["Blocks"].update(
                    part["__block_id"].unique()
                )

                item["Captures"].update(
                    part["__capture"].unique()
                )

            del work

        del df

    # ========================================================
    # TONG HOP THEO DATASET
    # ========================================================

    rows = []

    for mode in STRATEGIES:

        for label in classes:

            s = stats[mode][label]

            rows.append({
                "Dataset": dataset,
                "Strategy": mode,
                "Class": label,
                "Valid_Windows": s["Valid_Windows"],
                "Homogeneous_Windows":
                    s["Homogeneous_Windows"],
                "Valid_Blocks": len(s["Blocks"]),
                "Valid_Captures": len(s["Captures"])
            })

    result = pd.DataFrame(rows)

    all_results.append(result)

    print("\n=== SOURCE VS DESTINATION ===")

    pivot = result.pivot(
        index="Class",
        columns="Strategy",
        values="Valid_Windows"
    ).fillna(0).astype(int)

    pivot["Destination_Minus_Source"] = (
        pivot["Destination"] - pivot["Source"]
    )

    display(
        pivot.sort_values(
            "Destination_Minus_Source",
            ascending=False
        )
    )

    print("\n=== CLASS COVERAGE ===")

    for mode in STRATEGIES:

        subset = result[
            result["Strategy"] == mode
        ]

        print(f"\nStrategy: {mode}")

        display(
            subset[
                [
                    "Class",
                    "Valid_Windows",
                    "Homogeneous_Windows",
                    "Valid_Blocks",
                    "Valid_Captures"
                ]
            ].sort_values(
                "Valid_Windows"
            )
        )

    output = (
        OUT_DIR /
        f"{dataset}_grouping_comparison.csv"
    )

    result.to_csv(output, index=False)

    print("\nSaved:", output)

# ============================================================
# SUMMARY
# ============================================================

final = pd.concat(all_results, ignore_index=True)

summary = (
    final.groupby(["Dataset", "Strategy"])
    .agg(
        Valid_Windows=("Valid_Windows", "sum"),
        Homogeneous_Windows=(
            "Homogeneous_Windows", "sum"
        ),
        Classes_With_Windows=(
            "Valid_Windows",
            lambda s: int((s > 0).sum())
        ),
        Classes_With_3_Blocks=(
            "Valid_Blocks",
            lambda s: int((s >= 3).sum())
        ),
        Classes_With_3_Captures=(
            "Valid_Captures",
            lambda s: int((s >= 3).sum())
        )
    )
    .reset_index()
)

print("\n" + "=" * 75)
print("TEMPORAL GROUPING SUMMARY")
print("=" * 75)

display(summary)

final.to_csv(
    OUT_DIR / "grouping_comparison_all.csv",
    index=False
)

summary.to_csv(
    OUT_DIR / "grouping_summary.csv",
    index=False
)

print("\nCELL 27 COMPLETED")



GROUPING AUDIT: CIC-IDS2017

=== SOURCE VS DESTINATION ===


Strategy,Destination,Source,Destination_Minus_Source
Class,,,
BENIGN,10968,7891,3077
Bot,20,16,4
WebAttack-XSS,20,17,3
DoS Hulk,4,2,2
WebAttack-BruteForce,4,2,2
DoS slowloris,23,21,2
PortScan,3,1,2
DDoS,3,2,1
DoS GoldenEye,1,1,0



=== CLASS COVERAGE ===

Strategy: Source


,Class,Valid_Windows,Homogeneous_Windows,Valid_Blocks,Valid_Captures
3,DoS GoldenEye,1,1,1,1
8,PortScan,1,1,1,1
4,DoS Hulk,2,2,2,1
2,DDoS,2,2,2,1
10,WebAttack-BruteForce,2,2,1,1
7,FTP-Patator,4,4,1,1
9,SSH-Patator,7,7,3,1
1,Bot,16,6,5,1
5,DoS Slowhttptest,17,16,2,1
11,WebAttack-XSS,17,17,4,1



Strategy: Destination


,Class,Valid_Windows,Homogeneous_Windows,Valid_Blocks,Valid_Captures
15,DoS GoldenEye,1,1,1,1
14,DDoS,3,0,3,1
20,PortScan,3,1,3,1
19,FTP-Patator,4,4,1,1
22,WebAttack-BruteForce,4,2,1,1
16,DoS Hulk,4,2,4,1
21,SSH-Patator,7,3,4,1
17,DoS Slowhttptest,17,16,2,1
13,Bot,20,16,7,1
23,WebAttack-XSS,20,5,4,1


... (5 outputs left out)

,Class,Valid_Windows,Homogeneous_Windows,Valid_Blocks,Valid_Captures
35,WebDDoS,9,0,1,1
18,BENIGN,1409,1405,17,10
22,DrDoS_NTP,5292,5291,10,1
30,Portmap,5364,5361,10,1
34,UDPLag,5600,5568,10,1
28,MSSQL,5716,5716,10,2
32,TFTP,5752,5750,10,1
25,DrDoS_SSDP,5785,5785,10,1
24,DrDoS_SNMP,5790,5790,10,1
20,DrDoS_LDAP,5804,5804,10,1



Saved: /content/drive/MyDrive/NIDS_Project/results/Temporal/CIC-DDoS2019_grouping_comparison.csv

TEMPORAL GROUPING SUMMARY


,Dataset,Strategy,Valid_Windows,Homogeneous_Windows,Classes_With_Windows,Classes_With_3_Blocks,Classes_With_3_Captures
0,CIC-DDoS2019,Destination,105220,105162,18,17,2
1,CIC-DDoS2019,Source,105455,105372,18,17,2
2,CIC-IDS2017,Destination,11074,11027,12,8,1
3,CIC-IDS2017,Source,7981,7921,12,5,1



CELL 27 COMPLETED


In [ ]:

# ============================================================
# CELL 28 | TEMPORAL WINDOW TIME-SPAN DIAGNOSTICS
# ============================================================

# Tai su dung ROOT, pd, json tu Recovery Cell
# Khong tai dataset, khong train, khong install

TEMP_ROOT = ROOT / "datasets/prepared/temporal_blocks"
OUT_DIR = ROOT / "results/Temporal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 8
STRIDE = 8
MAX_SPAN = 300

DATASETS = ["CIC-IDS2017", "CIC-DDoS2019"]

STRATEGIES = {
    "Source": ["__block_id", "Source IP"],
    "Destination": ["__block_id", "Destination IP"],
    "BlockStream": ["__block_id"]
}

records = []

for dataset in DATASETS:

    print("\n" + "=" * 70)
    print("TEMPORAL TIME AUDIT:", dataset)
    print("=" * 70)

    folder = TEMP_ROOT / dataset
    files = sorted(folder.glob("*.parquet"))

    mapping_path = (
        ROOT / "datasets/prepared/ready" /
        dataset / "label_mapping.json"
    )

    mapping = json.loads(
        mapping_path.read_text(encoding="utf-8")
    )

    classes = list(mapping.values())

    for path in files:

        df = pd.read_parquet(
            path,
            columns=[
                "__block_id",
                "Source IP",
                "Destination IP",
                "Timestamp",
                "Label"
            ]
        )

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            format="mixed",
            errors="coerce"
        )

        df = df.dropna(
            subset=[
                "__block_id",
                "Source IP",
                "Destination IP",
                "Timestamp",
                "Label"
            ]
        )

        df = df[
            df["Label"].isin(classes)
        ].copy()

        if df.empty:
            continue

        for strategy, group_cols in STRATEGIES.items():

            work = df.sort_values(
                group_cols + ["Timestamp"],
                kind="mergesort"
            ).reset_index(drop=True)

            groups = work.groupby(
                group_cols,
                sort=False
            )

            pos = groups.cumcount()

            # Moi window co 8 flow khong overlap
            eligible = (
                (pos >= WINDOW_SIZE - 1)
                & (
                    (pos - WINDOW_SIZE + 1)
                    % STRIDE == 0
                )
            )

            start_time = groups["Timestamp"].shift(
                WINDOW_SIZE - 1
            )

            span = (
                work["Timestamp"] - start_time
            ).dt.total_seconds()

            windows = pd.DataFrame({
                "Class": work.loc[
                    eligible, "Label"
                ].to_numpy(),

                "Span_Seconds": span.loc[
                    eligible
                ].to_numpy()
            })

            for label in classes:

                durations = windows.loc[
                    windows["Class"] == label,
                    "Span_Seconds"
                ]

                records.append({
                    "Dataset": dataset,
                    "Strategy": strategy,
                    "Class": label,

                    "Candidate_Windows": int(
                        len(durations)
                    ),

                    "Zero_Span": int(
                        durations.eq(0).sum()
                    ),

                    "Positive_Within300": int(
                        (
                            durations.gt(0)
                            & durations.le(MAX_SPAN)
                        ).sum()
                    ),

                    "Nonnegative_Within300": int(
                        durations.between(
                            0, MAX_SPAN
                        ).sum()
                    ),

                    "Over300": int(
                        durations.gt(MAX_SPAN).sum()
                    ),

                    "Negative_Span": int(
                        durations.lt(0).sum()
                    )
                })

# ============================================================
# TONG HOP TUNG LOP
# ============================================================

raw_audit = pd.DataFrame(records)

metrics = [
    "Candidate_Windows",
    "Zero_Span",
    "Positive_Within300",
    "Nonnegative_Within300",
    "Over300",
    "Negative_Span"
]

audit = (
    raw_audit.groupby(
        ["Dataset", "Strategy", "Class"],
        as_index=False
    )[metrics]
    .sum()
)

audit.to_csv(
    OUT_DIR / "window_time_diagnostics.csv",
    index=False
)

for dataset in DATASETS:

    print("\n" + "=" * 70)
    print("DATASET:", dataset)
    print("=" * 70)

    subset = audit[
        audit["Dataset"] == dataset
    ]

    for strategy in STRATEGIES:

        part = subset[
            subset["Strategy"] == strategy
        ]

        print("\n=== STRATEGY:", strategy, "===")

        display(
            part[
                [
                    "Class",
                    "Candidate_Windows",
                    "Zero_Span",
                    "Positive_Within300",
                    "Nonnegative_Within300",
                    "Over300"
                ]
            ].sort_values(
                "Positive_Within300"
            ).reset_index(drop=True)
        )

# ============================================================
# TONG HOP CUOI CUNG
# ============================================================

summary = (
    audit.groupby(
        ["Dataset", "Strategy"],
        as_index=False
    )[metrics]
    .sum()
)

summary.to_csv(
    OUT_DIR / "window_time_summary.csv",
    index=False
)

print("\n" + "=" * 70)
print("TEMPORAL TIME DIAGNOSTIC SUMMARY")
print("=" * 70)

display(summary)

print("\nCELL 28 COMPLETED")



TEMPORAL TIME AUDIT: CIC-IDS2017

TEMPORAL TIME AUDIT: CIC-DDoS2019

DATASET: CIC-IDS2017

=== STRATEGY: Source ===


,Class,Candidate_Windows,Zero_Span,Positive_Within300,Nonnegative_Within300,Over300
0,DoS GoldenEye,79,76,1,77,2
1,PortScan,3493,3492,1,3493,0
2,DoS Hulk,2124,2121,2,2123,1
3,DDoS,2950,2948,2,2950,0
4,WebAttack-BruteForce,19,17,2,19,0
5,FTP-Patator,74,70,4,74,0
6,SSH-Patator,83,76,7,83,0
7,Bot,48,32,16,48,0
8,DoS Slowhttptest,688,671,17,688,0
9,WebAttack-XSS,79,62,17,79,0



=== STRATEGY: Destination ===


,Class,Candidate_Windows,Zero_Span,Positive_Within300,Nonnegative_Within300,Over300
0,DoS GoldenEye,79,76,1,77,2
1,DDoS,2951,2948,3,2951,0
2,PortScan,3494,3491,3,3494,0
3,FTP-Patator,74,70,4,74,0
4,WebAttack-BruteForce,20,16,4,20,0
5,DoS Hulk,2124,2119,4,2123,1
6,SSH-Patator,83,73,7,80,3
7,DoS Slowhttptest,688,671,17,688,0
8,Bot,43,23,20,43,0
9,WebAttack-XSS,82,62,20,82,0



=== STRATEGY: BlockStream ===


,Class,Candidate_Windows,Zero_Span,Positive_Within300,Nonnegative_Within300,Over300
0,WebAttack-BruteForce,20,20,0,20,0
1,Bot,54,53,1,54,0
2,DDoS,2952,2951,1,2952,0
3,PortScan,3498,3496,2,3498,0
4,WebAttack-XSS,81,79,2,81,0
5,DoS GoldenEye,80,76,4,80,0
6,FTP-Patator,75,70,5,75,0
7,DoS Hulk,2124,2119,5,2124,0
8,SSH-Patator,83,76,7,83,0
9,DoS Slowhttptest,688,670,18,688,0


... (5 outputs left out)

,Class,Candidate_Windows,Zero_Span,Positive_Within300,Nonnegative_Within300,Over300
0,WebDDoS,8,0,8,8,0
1,BENIGN,1998,0,1998,1998,0
2,DrDoS_NTP,5317,0,5317,5317,0
3,Portmap,5375,0,5375,5375,0
4,UDPLag,5607,0,5607,5607,0
5,MSSQL,5722,0,5722,5722,0
6,TFTP,5761,0,5761,5761,0
7,DrDoS_SSDP,5790,0,5790,5790,0
8,DrDoS_SNMP,5795,0,5795,5795,0
9,DrDoS_LDAP,5812,0,5812,5812,0



TEMPORAL TIME DIAGNOSTIC SUMMARY


,Dataset,Strategy,Candidate_Windows,Zero_Span,Positive_Within300,Nonnegative_Within300,Over300,Negative_Span
0,CIC-DDoS2019,BlockStream,105946,0,105946,105946,0,0
1,CIC-DDoS2019,Destination,105407,0,105220,105220,187,0
2,CIC-DDoS2019,Source,105541,0,105455,105455,86,0
3,CIC-IDS2017,BlockStream,53166,44882,8119,53001,165,0
4,CIC-IDS2017,Destination,43723,30605,11074,41679,2044,0
5,CIC-IDS2017,Source,47844,36150,7981,44131,3713,0



CELL 28 COMPLETED


In [ ]:

# ============================================================
# CELL 29 | TIMESTAMP PRECISION & DUPLICATION AUDIT
# ============================================================

import zipfile

TEMP_ROOT = ROOT / "datasets/prepared/temporal_blocks"
OUT_DIR = ROOT / "results/Temporal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CIC-IDS2017", "CIC-DDoS2019"]

# ============================================================
# 1. XEM TIMESTAMP NGUYEN BAN TRONG CSV IDS2017
# ============================================================

ids_zip = (
    ROOT / "datasets/CIC-IDS2017/CIC-IDS2017.zip"
)

print("=" * 65)
print("RAW IDS2017 TIMESTAMP INSPECTION")
print("=" * 65)

with zipfile.ZipFile(ids_zip) as zf:

    members = [
        name for name in zf.namelist()
        if name.lower().endswith(".csv")
    ]

    for keyword in [
        "Wednesday-workingHours",
        "Afternoon-PortScan",
        "Morning-WebAttacks"
    ]:

        matches = [
            name for name in members
            if keyword.lower() in name.lower()
        ]

        if not matches:
            print("Khong tim thay:", keyword)
            continue

        member = matches[0]

        with zf.open(member) as stream:

            sample = pd.read_csv(
                stream,
                nrows=2000,
                dtype=str,
                usecols=lambda c: c.strip() == "Timestamp",
                encoding="utf-8-sig",
                encoding_errors="replace"
            )

        sample.columns = sample.columns.str.strip()

        print("\nFILE:", member)
        print("Raw rows:", len(sample))
        print(
            "Unique timestamps:",
            sample["Timestamp"].nunique()
        )

        print("Timestamp examples:")
        print(
            sample["Timestamp"]
            .dropna()
            .drop_duplicates()
            .head(8)
            .tolist()
        )


# ============================================================
# 2. KIEM TRA TIMESTAMP TRONG TEMPORAL PARQUET
# ============================================================

records = []

for dataset in DATASETS:

    folder = TEMP_ROOT / dataset
    files = sorted(folder.glob("*.parquet"))

    print("\n" + "=" * 65)
    print("PARQUET AUDIT:", dataset)
    print("=" * 65)

    for path in files:

        df = pd.read_parquet(
            path,
            columns=[
                "__block_id",
                "Source IP",
                "Timestamp"
            ]
        )

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )

        df = df.dropna(
            subset=[
                "__block_id",
                "Source IP",
                "Timestamp"
            ]
        )

        if df.empty:
            continue

        ts = df["Timestamp"]

        # Do phan giai timestamp
        second_nonzero = ts.dt.second.ne(0)
        fractional_nonzero = (
            ts.dt.microsecond.ne(0)
            | ts.dt.nanosecond.ne(0)
        )

        # Kiem tra timestamp lien tiep trong cung host/block
        work = df.sort_values(
            ["__block_id", "Source IP", "Timestamp"],
            kind="mergesort"
        )

        groups = work.groupby(
            ["__block_id", "Source IP"],
            sort=False
        )

        previous = groups["Timestamp"].shift(1)

        comparable = previous.notna()

        same_timestamp = (
            work["Timestamp"].eq(previous)
            & comparable
        )

        comparable_count = int(comparable.sum())

        records.append({
            "Dataset": dataset,
            "File": path.name,
            "Rows": len(df),
            "Unique_Timestamps": ts.nunique(),
            "Unique_Ratio": ts.nunique() / len(df),
            "Nonzero_Second_Ratio":
                float(second_nonzero.mean()),
            "Fractional_Second_Ratio":
                float(fractional_nonzero.mean()),
            "Same_Timestamp_Adjacent_Ratio":
                (
                    int(same_timestamp.sum())
                    / comparable_count
                    if comparable_count else 0.0
                )
        })

# ============================================================
# 3. TONG HOP
# ============================================================

detail = pd.DataFrame(records)

detail.to_csv(
    OUT_DIR / "timestamp_precision_detail.csv",
    index=False
)

for dataset in DATASETS:

    subset = detail[
        detail["Dataset"] == dataset
    ].copy()

    print("\n" + "=" * 65)
    print("TIMESTAMP PRECISION:", dataset)
    print("=" * 65)

    print("Files:", len(subset))
    print("Rows:", int(subset["Rows"].sum()))

    display(
        subset[
            [
                "File",
                "Rows",
                "Unique_Ratio",
                "Nonzero_Second_Ratio",
                "Fractional_Second_Ratio",
                "Same_Timestamp_Adjacent_Ratio"
            ]
        ].sort_values(
            "Same_Timestamp_Adjacent_Ratio",
            ascending=False
        ).round(4)
    )

print("\nCELL 29 COMPLETED")


RAW IDS2017 TIMESTAMP INSPECTION

FILE: Wednesday-workingHours.pcap_ISCX.csv
Raw rows: 2000
Unique timestamps: 10
Timestamp examples:
['5/7/2017 8:42', '5/7/2017 8:43', '5/7/2017 8:44', '5/7/2017 8:45', '5/7/2017 8:46', '5/7/2017 8:47', '5/7/2017 8:48', '5/7/2017 8:49']

FILE: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Raw rows: 2000
Unique timestamps: 9
Timestamp examples:
['7/7/2017 1:00', '7/7/2017 1:01', '7/7/2017 1:02', '7/7/2017 1:03', '7/7/2017 1:04', '7/7/2017 1:05', '7/7/2017 1:06', '7/7/2017 1:07']

FILE: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Raw rows: 2000
Unique timestamps: 6
Timestamp examples:
['6/7/2017 8:59', '6/7/2017 9:00', '6/7/2017 9:01', '6/7/2017 9:02', '6/7/2017 9:03', '6/7/2017 9:04']

PARQUET AUDIT: CIC-IDS2017

PARQUET AUDIT: CIC-DDoS2019

TIMESTAMP PRECISION: CIC-IDS2017
Files: 23
Rows: 425373


,File,Rows,Unique_Ratio,Nonzero_Second_Ratio,Fractional_Second_Ratio,Same_Timestamp_Adjacent_Ratio
18,RECOVERY_Wednesday-workingHours_pcap_ISCX_0000...,5000,0.0050,0.000,0.0,0.9950
0,Friday-WorkingHours-Afternoon-DDos_pcap_ISCX.p...,45745,0.0006,0.000,0.0,0.9920
1,Friday-WorkingHours-Afternoon-PortScan_pcap_IS...,46467,0.0007,0.000,0.0,0.9879
19,Thursday-WorkingHours-Afternoon-Infilteration_...,48598,0.0013,0.000,0.0,0.9724
20,Thursday-WorkingHours-Morning-WebAttacks_pcap_...,20000,0.0012,0.000,0.0,0.9706
2,Friday-WorkingHours-Morning_pcap_ISCX.parquet,46033,0.0015,0.000,0.0,0.9645
6,RECOVERY_Thursday-WorkingHours-Morning-WebAtta...,5000,0.0014,0.000,0.0,0.9605
7,RECOVERY_Thursday-WorkingHours-Morning-WebAtta...,5000,0.0014,0.000,0.0,0.9593
5,RECOVERY_Thursday-WorkingHours-Morning-WebAtta...,5000,0.0016,0.000,0.0,0.9543
13,RECOVERY_Wednesday-workingHours_pcap_ISCX_0000...,5000,0.0168,0.000,0.0,0.9455



TIMESTAMP PRECISION: CIC-DDoS2019
Files: 18
Rows: 847637


,File,Rows,Unique_Ratio,Nonzero_Second_Ratio,Fractional_Second_Ratio,Same_Timestamp_Adjacent_Ratio
32,01-12_TFTP.parquet,47827,0.9969,0.9991,1.0,0.0031
26,01-12_DrDoS_NTP.parquet,47007,0.9996,0.9970,1.0,0.0004
37,03-11_Portmap.parquet,46694,0.9996,0.9989,1.0,0.0004
30,01-12_DrDoS_UDP.parquet,46802,0.9999,1.0000,1.0,0.0001
35,03-11_MSSQL.parquet,45786,0.9999,1.0000,1.0,0.0001
38,03-11_Syn.parquet,45541,0.9279,0.9963,1.0,0.0000
36,03-11_NetBIOS.parquet,45899,1.0000,0.9804,1.0,0.0000
23,01-12_DrDoS_DNS.parquet,49413,1.0000,0.9985,1.0,0.0000
24,01-12_DrDoS_LDAP.parquet,46542,1.0000,0.9611,1.0,0.0000
25,01-12_DrDoS_MSSQL.parquet,49498,1.0000,1.0000,1.0,0.0000



CELL 29 COMPLETED


In [ ]:

# ============================================================
# CELL 30 | BUILD TEMPORAL TRAIN / VAL / TEST
# DATASET: CIC-DDoS2019
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

DATASET = "CIC-DDoS2019"

INPUT_DIR = (
    ROOT / "datasets/prepared/temporal_blocks"
    / DATASET
)

READY_DIR = (
    ROOT / "datasets/prepared/ready"
    / DATASET
)

OUTPUT_DIR = (
    ROOT / "datasets/prepared/temporal_ready"
    / DATASET
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 8
STRIDE = 8
MAX_WINDOW_SECONDS = 300

# Thuc nghiem temporal ban dau: 17 lop
EXCLUDED_CLASSES = {"WebDDoS"}

# ============================================================
# 1. LAY FEATURE NAMES VA LABEL MAPPING
# ============================================================

baseline_package = joblib.load(
    READY_DIR / "preprocessor.joblib"
)

feature_cols = list(
    baseline_package["feature_names"]
)

mapping = json.loads(
    (READY_DIR / "label_mapping.json")
    .read_text(encoding="utf-8")
)

all_classes = [
    mapping[str(i)]
    for i in range(len(mapping))
]

class_names = [
    name for name in all_classes
    if name not in EXCLUDED_CLASSES
]

label_to_id = {
    name: i
    for i, name in enumerate(class_names)
}

print("Features:", len(feature_cols))
print("Temporal classes:", len(class_names))
print("Excluded:", sorted(EXCLUDED_CLASSES))

files = sorted(INPUT_DIR.glob("*.parquet"))

if not files:
    raise FileNotFoundError(
        f"Khong co Temporal Parquet: {INPUT_DIR}"
    )

# ============================================================
# 2. SPLIT THEO BLOCK TRONG TUNG CAPTURE
# ============================================================

# Moi block chi thuoc MOT tap.
# Sap xep theo timestamp, sau do chia 70/15/15.

block_records = []

for path in files:

    meta = pd.read_parquet(
        path,
        columns=[
            "__capture",
            "__block_id",
            "Timestamp"
        ]
    )

    meta["Timestamp"] = pd.to_datetime(
        meta["Timestamp"],
        errors="coerce"
    )

    first_times = (
        meta.dropna(subset=["Timestamp"])
        .groupby(
            ["__capture", "__block_id"],
            as_index=False
        )["Timestamp"]
        .min()
    )

    block_records.append(first_times)

block_df = pd.concat(
    block_records,
    ignore_index=True
).drop_duplicates(
    ["__capture", "__block_id"]
)

block_split = {}
split_rows = []

for capture, part in block_df.groupby("__capture"):

    part = part.sort_values(
        ["Timestamp", "__block_id"]
    ).reset_index(drop=True)

    n = len(part)

    if n < 3:
        raise ValueError(
            f"Capture {capture} chi co {n} blocks. "
            "Khong du chia 3 tap."
        )

    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))

    if n_train + n_val >= n:
        n_train = n - 2
        n_val = 1

    for i, row in part.iterrows():

        if i < n_train:
            split = "train"
        elif i < n_train + n_val:
            split = "val"
        else:
            split = "test"

        key = (
            row["__capture"],
            row["__block_id"]
        )

        block_split[key] = split

        split_rows.append({
            "Capture": key[0],
            "Block": key[1],
            "Split": split,
            "First_Timestamp": row["Timestamp"]
        })

split_df = pd.DataFrame(split_rows)

split_df.to_csv(
    OUTPUT_DIR / "block_split_manifest.csv",
    index=False
)

print("\n=== BLOCK SPLIT ===")
display(
    split_df.groupby(
        ["Capture", "Split"]
    ).size().unstack(fill_value=0)
)

# ============================================================
# 3. TAO WINDOWS TRONG TUNG BLOCK + SOURCE IP
# ============================================================

X_parts = {
    "train": [],
    "val": [],
    "test": []
}

y_parts = {
    "train": [],
    "val": [],
    "test": []
}

window_metadata = []

for file_idx, path in enumerate(files, start=1):

    columns = list(dict.fromkeys(
        [
            "__capture",
            "__block_id",
            "Source IP",
            "Timestamp",
            "Label"
        ] + feature_cols
    ))

    df = pd.read_parquet(
        path,
        columns=columns
    )

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "__capture",
            "__block_id",
            "Source IP",
            "Timestamp",
            "Label"
        ]
    ).copy()

    df = df.sort_values(
        [
            "__block_id",
            "Source IP",
            "Timestamp"
        ],
        kind="mergesort"
    )

    file_windows = 0

    for (capture, block_id, source_ip), group in (
        df.groupby(
            [
                "__capture",
                "__block_id",
                "Source IP"
            ],
            sort=False
        )
    ):

        if len(group) < WINDOW_SIZE:
            continue

        split = block_split[(capture, block_id)]

        # Mang timestamps
        timestamps = (
            group["Timestamp"]
            .to_numpy(dtype="datetime64[ns]")
        )

        # Mang features: (Flows, Features)
        features = group[feature_cols].to_numpy(
            dtype=np.float32
        )

        features[~np.isfinite(features)] = np.nan

        labels = group["Label"].astype(str).to_numpy()

        # Tao cac window khong chong lan
        n_windows = len(group) // WINDOW_SIZE

        if n_windows == 0:
            continue

        index_matrix = np.arange(
            n_windows * WINDOW_SIZE
        ).reshape(n_windows, WINDOW_SIZE)

        start_times = timestamps[
            index_matrix[:, 0]
        ]

        end_times = timestamps[
            index_matrix[:, -1]
        ]

        spans = (
            (end_times - start_times)
            / np.timedelta64(1, "s")
        )

        # Giua 0 va 300 giay
        valid_time = (
            (spans > 0)
            & (spans <= MAX_WINDOW_SECONDS)
        )

        window_labels = labels[index_matrix]

        # Khong su dung WebDDoS trong pilot 17 lop
        allowed = np.isin(
            window_labels,
            class_names
        ).all(axis=1)

        keep = valid_time & allowed

        if not keep.any():
            continue

        indices = index_matrix[keep]

        # Input shape: (N, 8, 66)
        X_windows = features[indices]

        # Nhan cua flow CUOI trong moi window
        endpoint_labels = labels[
            indices[:, -1]
        ]

        y_windows = np.array(
            [
                label_to_id[label]
                for label in endpoint_labels
            ],
            dtype=np.int64
        )

        X_parts[split].append(X_windows)
        y_parts[split].append(y_windows)

        for j, label in enumerate(endpoint_labels):

            window_metadata.append({
                "Split": split,
                "Capture": capture,
                "Block": block_id,
                "Source_IP": source_ip,
                "Start_Time": start_times[keep][j],
                "End_Time": end_times[keep][j],
                "Label": label,
                "Homogeneous": bool(
                    np.all(
                        window_labels[keep][j]
                        == label
                    )
                )
            })

        file_windows += len(y_windows)

    print(
        f"[{file_idx}/{len(files)}] "
        f"{path.name}: {file_windows:,} windows"
    )

    del df

# ============================================================
# 4. GHEP WINDOWS VA KIEM TRA
# ============================================================

X_data = {}
y_data = {}

for split in ["train", "val", "test"]:

    if not X_parts[split]:
        raise ValueError(
            f"Khong tao duoc windows cho {split}"
        )

    X_data[split] = np.concatenate(
        X_parts[split],
        axis=0
    )

    y_data[split] = np.concatenate(
        y_parts[split],
        axis=0
    )

    print(
        split,
        "X:", X_data[split].shape,
        "y:", y_data[split].shape
    )

# Xac nhan khong co block chung giua cac split
metadata_df = pd.DataFrame(window_metadata)

blocks_by_split = {
    split: set(
        metadata_df.loc[
            metadata_df["Split"] == split,
            "Block"
        ]
    )
    for split in ["train", "val", "test"]
}

assert blocks_by_split["train"].isdisjoint(
    blocks_by_split["val"]
)

assert blocks_by_split["train"].isdisjoint(
    blocks_by_split["test"]
)

assert blocks_by_split["val"].isdisjoint(
    blocks_by_split["test"]
)

print("\nBlock overlap check: PASS")

# ============================================================
# 5. FIT PREPROCESSOR CHI TREN TRAIN
# ============================================================

n_features = len(feature_cols)

X_train_flat = X_data["train"].reshape(
    -1, n_features
)

imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(
    X_train_flat
)

scaler.fit(X_train_imputed)

del X_train_imputed, X_train_flat

# Transform ca 3 tap voi cung preprocessor
for split in ["train", "val", "test"]:

    original_shape = X_data[split].shape

    flat = X_data[split].reshape(
        -1, n_features
    )

    flat = imputer.transform(flat)
    flat = scaler.transform(flat)

    X_data[split] = flat.reshape(
        original_shape
    ).astype(np.float32)

    if not np.isfinite(X_data[split]).all():
        raise ValueError(
            f"{split} con NaN/Inf sau transform"
        )

    del flat

# ============================================================
# 6. LUU DU LIEU VA PREPROCESSOR
# ============================================================

for split in ["train", "val", "test"]:

    np.savez_compressed(
        OUTPUT_DIR / f"{split}.npz",
        X=X_data[split],
        y=y_data[split]
    )

    print(
        "[SAVED]",
        OUTPUT_DIR / f"{split}.npz"
    )

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "feature_names": feature_cols,
        "class_names": class_names,
        "window_size": WINDOW_SIZE,
        "stride": STRIDE,
        "max_window_seconds": MAX_WINDOW_SECONDS,
        "grouping": "Source IP",
        "excluded_classes": sorted(EXCLUDED_CLASSES)
    },
    OUTPUT_DIR / "temporal_preprocessor.joblib"
)

metadata_df.to_csv(
    OUTPUT_DIR / "window_metadata.csv",
    index=False
)

(
    OUTPUT_DIR / "label_mapping.json"
).write_text(
    json.dumps(
        {
            str(i): name
            for i, name in enumerate(class_names)
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 7. FINAL CLASS COVERAGE
# ============================================================

coverage = []

for i, label in enumerate(class_names):

    coverage.append({
        "Class": label,
        "Train": int(
            (y_data["train"] == i).sum()
        ),
        "Val": int(
            (y_data["val"] == i).sum()
        ),
        "Test": int(
            (y_data["test"] == i).sum()
        )
    })

coverage_df = pd.DataFrame(coverage)

print("\n=== TEMPORAL CLASS COVERAGE ===")
display(coverage_df)

missing_in_any = coverage_df[
    (coverage_df[["Train", "Val", "Test"]] == 0)
    .any(axis=1)
]

if not missing_in_any.empty:
    print("\nWARNING: Classes missing in a split!")
    display(missing_in_any)
else:
    print("\nAll 17 classes have Train/Val/Test windows.")

coverage_df.to_csv(
    OUTPUT_DIR / "class_coverage.csv",
    index=False
)

print("\nCELL 30 COMPLETED")


Features: 66
Temporal classes: 17
Excluded: ['WebDDoS']

=== BLOCK SPLIT ===


Split,test,train,val
Capture,,,
01-12_DrDoS_DNS,2,7,1
01-12_DrDoS_LDAP,2,7,1
01-12_DrDoS_MSSQL,2,7,1
01-12_DrDoS_NTP,2,7,1
01-12_DrDoS_NetBIOS,2,7,1
01-12_DrDoS_SNMP,2,7,1
01-12_DrDoS_SSDP,2,7,1
01-12_DrDoS_UDP,2,7,1
01-12_Syn,2,7,1


[1/18] 01-12_DrDoS_DNS.parquet: 6,142 windows
[2/18] 01-12_DrDoS_LDAP.parquet: 5,806 windows
[3/18] 01-12_DrDoS_MSSQL.parquet: 6,178 windows
[4/18] 01-12_DrDoS_NTP.parquet: 5,791 windows
[5/18] 01-12_DrDoS_NetBIOS.parquet: 6,240 windows
[6/18] 01-12_DrDoS_SNMP.parquet: 5,790 windows
[7/18] 01-12_DrDoS_SSDP.parquet: 5,788 windows
[8/18] 01-12_DrDoS_UDP.parquet: 5,841 windows
[9/18] 01-12_Syn.parquet: 5,952 windows
[10/18] 01-12_TFTP.parquet: 5,931 windows
[11/18] 01-12_UDPLag.parquet: 5,644 windows
[12/18] 03-11_LDAP.parquet: 6,017 windows
[13/18] 03-11_MSSQL.parquet: 5,716 windows
[14/18] 03-11_NetBIOS.parquet: 5,730 windows
[15/18] 03-11_Portmap.parquet: 5,761 windows
[16/18] 03-11_Syn.parquet: 5,555 windows
[17/18] 03-11_UDP.parquet: 5,893 windows
[18/18] 03-11_UDPLag.parquet: 5,639 windows
train X: (78442, 8, 66) y: (78442,)
val X: (11236, 8, 66) y: (11236,)
test X: (15736, 8, 66) y: (15736,)

Block overlap check: PASS
[SAVED] /content/drive/MyDrive/NIDS_Project/datasets/prepared/te

,Class,Train,Val,Test
0,BENIGN,1016,1,631
1,DrDoS_DNS,4296,624,1100
2,DrDoS_LDAP,4364,624,816
3,DrDoS_MSSQL,4366,624,1183
4,DrDoS_NTP,3827,624,841
5,DrDoS_NetBIOS,4367,624,1247
6,DrDoS_SNMP,4369,625,796
7,DrDoS_SSDP,4365,625,795
8,DrDoS_UDP,4368,624,849
9,LDAP,4366,622,1026



All 17 classes have Train/Val/Test windows.

CELL 30 COMPLETED


In [ ]:

# ============================================================
# CELL 30.1 | AUDIT BENIGN DISTRIBUTION BY BLOCK
# ============================================================

DATASET = "CIC-DDoS2019"

OUTPUT_DIR = (
    ROOT / "datasets/prepared/temporal_ready"
    / DATASET
)

metadata = pd.read_csv(
    OUTPUT_DIR / "window_metadata.csv"
)

# 1. Phan bo nhan theo split
print("=== LABEL DISTRIBUTION BY SPLIT ===")

label_distribution = pd.crosstab(
    metadata["Label"],
    metadata["Split"]
)

display(label_distribution)

# 2. BENIGN theo capture va split
print("\n=== BENIGN BY CAPTURE ===")

benign = metadata[
    metadata["Label"] == "BENIGN"
].copy()

benign_capture = pd.crosstab(
    benign["Capture"],
    benign["Split"]
)

display(benign_capture)

# 3. Thong ke BENIGN theo tung block
print("\n=== BENIGN WINDOWS BY BLOCK ===")

block_stats = (
    metadata.groupby(
        ["Capture", "Block", "Split"]
    )
    .agg(
        Total_Windows=("Label", "size"),
        Benign_Windows=(
            "Label",
            lambda s: int((s == "BENIGN").sum())
        )
    )
    .reset_index()
)

block_stats["Benign_Ratio"] = (
    block_stats["Benign_Windows"]
    / block_stats["Total_Windows"]
)

display(
    block_stats[
        block_stats["Benign_Windows"] > 0
    ].sort_values(
        "Benign_Windows",
        ascending=False
    ).head(30)
)

# 4. Tim block Train co BENIGN
print("\n=== TRAIN BLOCKS AVAILABLE FOR REBALANCING ===")

train_candidates = block_stats[
    (block_stats["Split"] == "train")
    & (block_stats["Benign_Windows"] > 0)
].sort_values(
    "Benign_Windows",
    ascending=False
)

display(train_candidates.head(30))

# 5. Luu bao cao
block_stats.to_csv(
    OUTPUT_DIR / "block_label_audit.csv",
    index=False
)

print("\nCELL 30.1 COMPLETED")


=== LABEL DISTRIBUTION BY SPLIT ===


Split,test,train,val
Label,,,
BENIGN,631,1016,1
DrDoS_DNS,1100,4296,624
DrDoS_LDAP,816,4364,624
DrDoS_MSSQL,1183,4366,624
DrDoS_NTP,841,3827,624
DrDoS_NetBIOS,1247,4367,624
DrDoS_SNMP,796,4369,625
DrDoS_SSDP,795,4365,625
DrDoS_UDP,849,4368,624



=== BENIGN BY CAPTURE ===


Split,test,train,val
Capture,,,
01-12_DrDoS_DNS,64,58,0
01-12_DrDoS_LDAP,0,2,0
01-12_DrDoS_MSSQL,3,2,0
01-12_DrDoS_NTP,27,472,0
01-12_DrDoS_NetBIOS,0,2,0
01-12_DrDoS_SSDP,0,3,0
01-12_TFTP,178,0,0
01-12_UDPLag,0,76,0
03-11_LDAP,1,2,1



=== BENIGN WINDOWS BY BLOCK ===


,Capture,Block,Split,Total_Windows,Benign_Windows,Benign_Ratio
30,01-12_DrDoS_NTP,01-12_DrDoS_NTP@000000,train,560,464,0.828571
140,03-11_Portmap,03-11_Portmap@000000,train,556,398,0.715827
151,03-11_Syn,03-11_Syn@000096,test,504,321,0.636905
99,01-12_TFTP,01-12_TFTP@004021,test,311,178,0.572347
101,01-12_UDPLag,01-12_UDPLag@000008,train,573,76,0.132635
9,01-12_DrDoS_DNS,01-12_DrDoS_DNS@001014,test,540,64,0.118519
0,01-12_DrDoS_DNS,01-12_DrDoS_DNS@000000,train,609,58,0.095238
159,03-11_Syn,03-11_Syn@000864,test,56,37,0.660714
39,01-12_DrDoS_NTP,01-12_DrDoS_NTP@000243,test,244,27,0.110656
33,01-12_DrDoS_NTP,01-12_DrDoS_NTP@000081,train,623,6,0.009631



=== TRAIN BLOCKS AVAILABLE FOR REBALANCING ===


,Capture,Block,Split,Total_Windows,Benign_Windows,Benign_Ratio
30,01-12_DrDoS_NTP,01-12_DrDoS_NTP@000000,train,560,464,0.828571
140,03-11_Portmap,03-11_Portmap@000000,train,556,398,0.715827
101,01-12_UDPLag,01-12_UDPLag@000008,train,573,76,0.132635
0,01-12_DrDoS_DNS,01-12_DrDoS_DNS@000000,train,609,58,0.095238
33,01-12_DrDoS_NTP,01-12_DrDoS_NTP@000081,train,623,6,0.009631
60,01-12_DrDoS_SSDP,01-12_DrDoS_SSDP@000000,train,624,3,0.004808
114,03-11_LDAP,03-11_LDAP@000187,train,624,2,0.003205
45,01-12_DrDoS_NetBIOS,01-12_DrDoS_NetBIOS@000454,train,624,2,0.003205
21,01-12_DrDoS_MSSQL,01-12_DrDoS_MSSQL@000100,train,622,2,0.003215
12,01-12_DrDoS_LDAP,01-12_DrDoS_LDAP@000096,train,623,1,0.001605



CELL 30.1 COMPLETED


In [ ]:

# ============================================================
# CELL 30.2 | REBALANCE BLOCK SPLIT + REBUILD TEMPORAL DATA
# Khong tai lai dataset, khong huan luyen model
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

DATASET = "CIC-DDoS2019"

RAW_DIR = (
    ROOT / "datasets/prepared/temporal_blocks" / DATASET
)
BASE_DIR = (
    ROOT / "datasets/prepared/ready" / DATASET
)
OLD_DIR = (
    ROOT / "datasets/prepared/temporal_ready" / DATASET
)
NEW_DIR = (
    ROOT / "datasets/prepared/temporal_ready" /
    "CIC-DDoS2019_v2"
)
NEW_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 8
MAX_SECONDS = 300
SPLITS = ["train", "val", "test"]

# ------------------------------------------------------------
# 1. Khoi phuc feature names va 17 nhan
# ------------------------------------------------------------

package = joblib.load(BASE_DIR / "preprocessor.joblib")
feature_cols = list(package["feature_names"])

mapping = json.loads(
    (BASE_DIR / "label_mapping.json")
    .read_text(encoding="utf-8")
)

classes = [
    mapping[str(i)]
    for i in range(len(mapping))
    if mapping[str(i)] != "WebDDoS"
]

label_to_id = {
    name: i for i, name in enumerate(classes)
}

assert len(feature_cols) == 66
assert len(classes) == 17

# ------------------------------------------------------------
# 2. Hoan doi hai block Train / Validation
# ------------------------------------------------------------

manifest = pd.read_csv(
    OLD_DIR / "block_split_manifest.csv"
)

assert not manifest.duplicated(
    ["Capture", "Block"]
).any(), "Manifest co block trung lap"

capture = "03-11_Portmap"
train_block = "03-11_Portmap@000000"

mask_train = (
    (manifest["Capture"] == capture)
    & (manifest["Block"] == train_block)
    & (manifest["Split"] == "train")
)

mask_val = (
    (manifest["Capture"] == capture)
    & (manifest["Split"] == "val")
)

assert mask_train.sum() == 1, (
    "Khong tim thay train block can hoan doi"
)
assert mask_val.sum() == 1, (
    "Capture Portmap khong co dung 1 val block"
)

val_block = manifest.loc[
    mask_val, "Block"
].iloc[0]

old_test_blocks = set(
    zip(
        manifest.loc[
            manifest["Split"] == "test", "Capture"
        ],
        manifest.loc[
            manifest["Split"] == "test", "Block"
        ]
    )
)

manifest.loc[mask_train, "Split"] = "val"
manifest.loc[mask_val, "Split"] = "train"

new_test_blocks = set(
    zip(
        manifest.loc[
            manifest["Split"] == "test", "Capture"
        ],
        manifest.loc[
            manifest["Split"] == "test", "Block"
        ]
    )
)

assert old_test_blocks == new_test_blocks

split_map = {
    (row.Capture, row.Block): row.Split
    for row in manifest.itertuples(index=False)
}

print("=== BLOCK SWAP ===")
print("Train -> Val:", train_block)
print("Val -> Train:", val_block)
print("Test blocks: UNCHANGED")

# ------------------------------------------------------------
# 3. Du bao coverage tu metadata da luu
# ------------------------------------------------------------

old_meta = pd.read_csv(
    OLD_DIR / "window_metadata.csv",
    usecols=["Capture", "Block", "Label"]
)

old_meta["Split"] = [
    split_map[(cap, block)]
    for cap, block in zip(
        old_meta["Capture"],
        old_meta["Block"]
    )
]

expected = pd.crosstab(
    old_meta["Label"],
    old_meta["Split"]
).reindex(
    index=classes,
    columns=SPLITS,
    fill_value=0
)

print("\n=== EXPECTED CLASS COVERAGE ===")
display(expected)

assert (expected > 0).all().all(), (
    "Co lop bi mat khoi Train/Val/Test"
)
assert expected.loc["BENIGN", "train"] >= 100
assert expected.loc["BENIGN", "val"] >= 100
assert expected.loc["BENIGN", "test"] >= 100

# Luu manifest moi
manifest.to_csv(
    NEW_DIR / "block_split_manifest.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Tao lai raw windows tu Temporal Parquet
# ------------------------------------------------------------

X_parts = {s: [] for s in SPLITS}
y_parts = {s: [] for s in SPLITS}
meta_parts = []

files = sorted(RAW_DIR.glob("*.parquet"))

columns = list(dict.fromkeys(
    [
        "__capture",
        "__block_id",
        "Source IP",
        "Timestamp",
        "Label"
    ] + feature_cols
))

for file_number, path in enumerate(files, start=1):

    df = pd.read_parquet(
        path,
        columns=columns
    )

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "__capture",
            "__block_id",
            "Source IP",
            "Timestamp",
            "Label"
        ]
    ).copy()

    df = df.sort_values(
        ["__block_id", "Source IP", "Timestamp"],
        kind="mergesort"
    )

    created = 0

    grouped = df.groupby(
        ["__capture", "__block_id", "Source IP"],
        sort=False
    )

    for (cap, block, source_ip), group in grouped:

        if len(group) < WINDOW:
            continue

        split = split_map[(cap, block)]

        timestamps = group["Timestamp"].to_numpy(
            dtype="datetime64[ns]"
        )

        labels = group["Label"].astype(str).to_numpy()

        features = group[feature_cols].to_numpy(
            dtype=np.float32
        )

        features[~np.isfinite(features)] = np.nan

        # Window khong chong lap: stride = 8
        n = len(group) // WINDOW

        indices = np.arange(
            n * WINDOW
        ).reshape(n, WINDOW)

        starts = timestamps[indices[:, 0]]
        ends = timestamps[indices[:, -1]]

        span = (
            (ends - starts)
            / np.timedelta64(1, "s")
        )

        window_labels = labels[indices]

        # Loai window co WebDDoS
        allowed = np.isin(
            window_labels, classes
        ).all(axis=1)

        valid_time = (
            (span > 0)
            & (span <= MAX_SECONDS)
        )

        keep = allowed & valid_time

        if not keep.any():
            continue

        selected = indices[keep]

        X_win = features[selected]
        endpoint_labels = labels[selected[:, -1]]

        y_win = np.array(
            [
                label_to_id[name]
                for name in endpoint_labels
            ],
            dtype=np.int64
        )

        X_parts[split].append(X_win)
        y_parts[split].append(y_win)

        homogeneous = (
            window_labels[keep]
            == endpoint_labels[:, None]
        ).all(axis=1)

        meta_parts.append(
            pd.DataFrame({
                "Capture": cap,
                "Block": block,
                "Source_IP": source_ip,
                "Start_Time": starts[keep],
                "End_Time": ends[keep],
                "Label": endpoint_labels,
                "Homogeneous": homogeneous,
                "Split": split
            })
        )

        created += len(y_win)

    print(
        f"[{file_number}/{len(files)}] "
        f"{path.name}: {created:,} windows"
    )

    del df

# ------------------------------------------------------------
# 5. Ghep arrays
# ------------------------------------------------------------

X_raw = {}
y_data = {}

for split in SPLITS:

    if not X_parts[split]:
        raise ValueError(
            f"Khong co windows cho split {split}"
        )

    X_raw[split] = np.concatenate(
        X_parts[split],
        axis=0
    )

    y_data[split] = np.concatenate(
        y_parts[split],
        axis=0
    )

    print(
        f"{split}: "
        f"X={X_raw[split].shape}, "
        f"y={y_data[split].shape}"
    )

del X_parts, y_parts

# ------------------------------------------------------------
# 6. Kiem tra coverage truoc khi luu
# ------------------------------------------------------------

actual = pd.DataFrame({
    split: [
        int((y_data[split] == i).sum())
        for i in range(len(classes))
    ]
    for split in SPLITS
}, index=classes)

print("\n=== ACTUAL CLASS COVERAGE ===")
display(actual)

assert np.array_equal(
    actual.to_numpy(),
    expected.to_numpy()
), (
    "Coverage khong khop du bao. "
    "Can kiem tra truoc khi luu."
)

assert (actual > 0).all().all()

print("\nClass coverage: PASS")

# ------------------------------------------------------------
# 7. Fit preprocessing CHI tren Train moi
# ------------------------------------------------------------

n_features = len(feature_cols)

train_flat = X_raw["train"].reshape(
    -1, n_features
)

imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

scaler = StandardScaler()

train_imputed = imputer.fit_transform(
    train_flat
)

scaler.fit(train_imputed)

del train_imputed, train_flat

print("Train-only preprocessing: FITTED")

# ------------------------------------------------------------
# 8. Transform va luu tung split
# ------------------------------------------------------------

for split in SPLITS:

    shape = X_raw[split].shape

    flat = X_raw[split].reshape(
        -1, n_features
    )

    transformed = scaler.transform(
        imputer.transform(flat)
    )

    transformed = transformed.reshape(
        shape
    ).astype(np.float32)

    assert np.isfinite(transformed).all(), (
        f"{split} van con NaN/Inf"
    )

    output_path = NEW_DIR / f"{split}.npz"

    np.savez_compressed(
        output_path,
        X=transformed,
        y=y_data[split]
    )

    print("[SAVED]", output_path)

    del flat, transformed

# ------------------------------------------------------------
# 9. Luu metadata va preprocessor
# ------------------------------------------------------------

metadata = pd.concat(
    meta_parts,
    ignore_index=True
)

metadata.to_csv(
    NEW_DIR / "window_metadata.csv",
    index=False
)

actual.rename_axis("Class").to_csv(
    NEW_DIR / "class_coverage.csv"
)

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "feature_names": feature_cols,
        "class_names": classes,
        "window_size": WINDOW,
        "stride": WINDOW,
        "max_window_seconds": MAX_SECONDS,
        "grouping": "Source IP",
        "excluded_classes": ["WebDDoS"],
        "split_method": "stratified_block_holdout_v2"
    },
    NEW_DIR / "temporal_preprocessor.joblib"
)

(NEW_DIR / "label_mapping.json").write_text(
    json.dumps(
        {
            str(i): name
            for i, name in enumerate(classes)
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. Kiem tra cuoi
# ------------------------------------------------------------

for split in SPLITS:

    p = NEW_DIR / f"{split}.npz"

    with np.load(p) as arr:
        print(
            split,
            "X:", arr["X"].shape,
            "y:", arr["y"].shape
        )

print("\nBENIGN:")
print(actual.loc["BENIGN"].to_dict())

print("\nOutput:", NEW_DIR)
print("CELL 30.2 COMPLETED")


=== BLOCK SWAP ===
Train -> Val: 03-11_Portmap@000000
Val -> Train: 03-11_Portmap@000029
Test blocks: UNCHANGED

=== EXPECTED CLASS COVERAGE ===


Split,train,val,test
Label,,,
BENIGN,618,399,631
DrDoS_DNS,4296,624,1100
DrDoS_LDAP,4364,624,816
DrDoS_MSSQL,4366,624,1183
DrDoS_NTP,3827,624,841
DrDoS_NetBIOS,4367,624,1247
DrDoS_SNMP,4369,625,796
DrDoS_SSDP,4365,625,795
DrDoS_UDP,4368,624,849


[1/18] 01-12_DrDoS_DNS.parquet: 6,142 windows
[2/18] 01-12_DrDoS_LDAP.parquet: 5,806 windows
[3/18] 01-12_DrDoS_MSSQL.parquet: 6,178 windows
[4/18] 01-12_DrDoS_NTP.parquet: 5,791 windows
[5/18] 01-12_DrDoS_NetBIOS.parquet: 6,240 windows
[6/18] 01-12_DrDoS_SNMP.parquet: 5,790 windows
[7/18] 01-12_DrDoS_SSDP.parquet: 5,788 windows
[8/18] 01-12_DrDoS_UDP.parquet: 5,841 windows
[9/18] 01-12_Syn.parquet: 5,952 windows
[10/18] 01-12_TFTP.parquet: 5,931 windows
[11/18] 01-12_UDPLag.parquet: 5,644 windows
[12/18] 03-11_LDAP.parquet: 6,017 windows
[13/18] 03-11_MSSQL.parquet: 5,716 windows
[14/18] 03-11_NetBIOS.parquet: 5,730 windows
[15/18] 03-11_Portmap.parquet: 5,761 windows
[16/18] 03-11_Syn.parquet: 5,555 windows
[17/18] 03-11_UDP.parquet: 5,893 windows
[18/18] 03-11_UDPLag.parquet: 5,639 windows
train: X=(78511, 8, 66), y=(78511,)
val: X=(11167, 8, 66), y=(11167,)
test: X=(15736, 8, 66), y=(15736,)

=== ACTUAL CLASS COVERAGE ===


,train,val,test
BENIGN,618,399,631
DrDoS_DNS,4296,624,1100
DrDoS_LDAP,4364,624,816
DrDoS_MSSQL,4366,624,1183
DrDoS_NTP,3827,624,841
DrDoS_NetBIOS,4367,624,1247
DrDoS_SNMP,4369,625,796
DrDoS_SSDP,4365,625,795
DrDoS_UDP,4368,624,849
LDAP,4366,622,1026



Class coverage: PASS
Train-only preprocessing: FITTED
[SAVED] /content/drive/MyDrive/NIDS_Project/datasets/prepared/temporal_ready/CIC-DDoS2019_v2/train.npz
[SAVED] /content/drive/MyDrive/NIDS_Project/datasets/prepared/temporal_ready/CIC-DDoS2019_v2/val.npz
[SAVED] /content/drive/MyDrive/NIDS_Project/datasets/prepared/temporal_ready/CIC-DDoS2019_v2/test.npz
train X: (78511, 8, 66) y: (78511,)
val X: (11167, 8, 66) y: (11167,)
test X: (15736, 8, 66) y: (15736,)

BENIGN:
{'train': 618, 'val': 399, 'test': 631}

Output: /content/drive/MyDrive/NIDS_Project/datasets/prepared/temporal_ready/CIC-DDoS2019_v2
CELL 30.2 COMPLETED


In [ ]:

# ============================================================
# CELL 31 - TRAIN TEMPORAL CNN-LSTM (CPU)
# CIC-DDoS2019 V2 | 8 flows | 66 features | 17 classes
# ============================================================

if "torch" not in globals():
    import torch

if "precision_recall_fscore_support" not in globals():
    from sklearn.metrics import precision_recall_fscore_support

if "perf_counter" not in globals():
    from time import perf_counter

# 1. Cấu hình
DATA_DIR = ROOT / "datasets/prepared/temporal_ready/CIC-DDoS2019_v2"
MODEL_DIR = ROOT / "models/CIC-DDoS2019/Temporal/CNN_LSTM_v2_CPU"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = MODEL_DIR / "best.pt"
LAST_PATH = MODEL_DIR / "last.pt"
HISTORY_PATH = MODEL_DIR / "training_history.csv"

SEED = 42
BATCH_SIZE = 256
MAX_EPOCHS = 30
PATIENCE = 6
LR = 0.001
N_CLASSES = 17

device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Device:", device)
print("CPU threads:", torch.get_num_threads())

# 2. Load Train và Validation
def load_data(split):
    with np.load(DATA_DIR / f"{split}.npz") as data:
        return data["X"].astype(np.float32), data["y"].astype(np.int64)

X_train, y_train = load_data("train")
X_val, y_val = load_data("val")

mapping = json.loads((DATA_DIR / "label_mapping.json").read_text())
class_names = [mapping[str(i)] for i in range(N_CLASSES)]

assert X_train.shape[1:] == X_val.shape[1:] == (8, 66)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Classes:", N_CLASSES)

# 3. DataLoader
generator = torch.Generator().manual_seed(SEED)

def make_loader(X, y, shuffle=False):
    dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(X), torch.from_numpy(y))
    return torch.utils.data.DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        generator=generator if shuffle else None, num_workers=0)

train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val)

# 4. Kiến trúc Temporal CNN-LSTM
class TemporalCNNLSTM(torch.nn.Module):
    def __init__(self, n_classes=17):
        super().__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv1d(1, 32, 5, padding=2),
            torch.nn.BatchNorm1d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(2),
            torch.nn.Conv1d(32, 64, 3, padding=1),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(2),
            torch.nn.AdaptiveAvgPool1d(4),
            torch.nn.Flatten()
        )
        self.flow_embedding = torch.nn.Sequential(
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2)
        )
        self.lstm = torch.nn.LSTM(128, 128, batch_first=True)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(64, n_classes)
        )

    def forward(self, x):
        b, t, f = x.shape
        x = self.cnn(x.reshape(b * t, 1, f))
        x = self.flow_embedding(x).reshape(b, t, 128)
        x, _ = self.lstm(x)
        return self.classifier(x[:, -1, :])

model = TemporalCNNLSTM(N_CLASSES).to(device)

# 5. Class Weight, Loss, Optimizer
counts = np.bincount(y_train, minlength=N_CLASSES)
assert (counts > 0).all()

weights = np.sqrt(counts.max() / counts)
weights = np.clip(weights / weights.mean(), 0.3, 4.0)
class_weights = torch.tensor(weights, dtype=torch.float32)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5)

print("Parameters:", sum(p.numel() for p in model.parameters()))

# 6. Hàm Train / Validation dùng chung
def run_epoch(loader, training=False):
    model.train(training)
    loss_sum, weight_sum = 0.0, 0.0
    targets, predictions = [], []

    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            w = class_weights[yb].sum().item()
            loss_sum += loss.item() * w
            weight_sum += w
            targets.extend(yb.tolist())
            predictions.extend(logits.detach().argmax(1).tolist())

    p, r, f1, _ = precision_recall_fscore_support(
        targets, predictions, labels=np.arange(N_CLASSES),
        average="macro", zero_division=0)

    return {
        "loss": loss_sum / weight_sum,
        "accuracy": float(np.mean(np.array(targets) == predictions)),
        "precision": float(p), "recall": float(r), "f1": float(f1)
    }

# 7. Lưu và khôi phục checkpoint
def save_checkpoint(data, path):
    temp = path.with_suffix(".tmp")
    torch.save(data, temp)
    temp.replace(path)

config = {
    "dataset": DATA_DIR.name, "window": 8, "features": 66,
    "classes": class_names, "batch_size": BATCH_SIZE,
    "lr": LR, "seed": SEED, "patience": PATIENCE
}

start_epoch, best_f1, best_epoch, wait = 1, -1.0, 0, 0
history = []

if LAST_PATH.exists():
    ckpt = torch.load(LAST_PATH, map_location="cpu", weights_only=True)
    assert ckpt["config"] == config, "Checkpoint không khớp cấu hình"
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    generator.set_state(ckpt["generator_state"])
    torch.set_rng_state(ckpt["rng_state"])

    start_epoch = ckpt["epoch"] + 1
    best_f1, best_epoch = ckpt["best_f1"], ckpt["best_epoch"]
    wait, history = ckpt["wait"], ckpt["history"]
    print("Resume from epoch:", start_epoch)

# 8. Training Loop
print("\n========== START TRAINING ==========")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    if wait >= PATIENCE:
        print("Early stopping.")
        break

    t0 = perf_counter()
    train = run_epoch(train_loader, training=True)
    val = run_epoch(val_loader)
    scheduler.step(val["f1"])

    improved = val["f1"] > best_f1 + 1e-4

    if improved:
        best_f1, best_epoch, wait = val["f1"], epoch, 0
        save_checkpoint({
            "model_state_dict": model.state_dict(),
            "epoch": epoch, "config": config,
            "class_names": class_names, "val_metrics": val
        }, BEST_PATH)
    else:
        wait += 1

    history.append({
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train.items()},
        **{f"val_{k}": v for k, v in val.items()},
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": perf_counter() - t0
    })

    save_checkpoint({
        "epoch": epoch, "model_state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "generator_state": generator.get_state(),
        "rng_state": torch.get_rng_state(),
        "best_f1": best_f1, "best_epoch": best_epoch,
        "wait": wait, "history": history, "config": config
    }, LAST_PATH)

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss={train['loss']:.4f} | F1={train['f1']:.4f} | "
        f"Val Loss={val['loss']:.4f} | Acc={val['accuracy']:.4f} | "
        f"F1={val['f1']:.4f} | "
        f"{perf_counter()-t0:.1f}s"
        + (" [NEW BEST]" if improved else ""), flush=True
    )

# 9. Tổng kết
print("\n========== TRAINING SUMMARY ==========")
print("Best epoch:", best_epoch)
print("Best Validation Macro-F1:", round(best_f1, 6))
print("Best model:", BEST_PATH)
print("Last checkpoint:", LAST_PATH)

display(pd.DataFrame(history).tail(10).round(4))
print("\nCELL 31 COMPLETED")



Device: cpu
CPU threads: 1
Train: (78511, 8, 66)
Validation: (11167, 8, 66)
Classes: 17
Parameters: 180945

========== START TRAINING ==========
Epoch 01/30 | Train Loss=1.2888 | F1=0.4669 | Val Loss=0.9384 | Acc=0.6226 | F1=0.4934 | 81.3s [NEW BEST]
Epoch 02/30 | Train Loss=0.9018 | F1=0.5744 | Val Loss=0.8737 | Acc=0.5889 | F1=0.5187 | 79.7s [NEW BEST]
Epoch 03/30 | Train Loss=0.8605 | F1=0.5935 | Val Loss=1.0190 | Acc=0.6111 | F1=0.4844 | 80.8s
Epoch 04/30 | Train Loss=0.8391 | F1=0.6002 | Val Loss=0.9198 | Acc=0.5849 | F1=0.4835 | 79.7s
Epoch 05/30 | Train Loss=0.8224 | F1=0.6060 | Val Loss=0.8751 | Acc=0.6195 | F1=0.5248 | 81.3s [NEW BEST]
Epoch 06/30 | Train Loss=0.8132 | F1=0.6145 | Val Loss=0.8666 | Acc=0.6182 | F1=0.5215 | 81.2s
Epoch 07/30 | Train Loss=0.8049 | F1=0.6189 | Val Loss=0.8762 | Acc=0.6153 | F1=0.5273 | 82.3s [NEW BEST]
Epoch 08/30 | Train Loss=0.7999 | F1=0.6219 | Val Loss=0.8693 | Acc=0.6234 | F1=0.5396 | 79.5s [NEW BEST]
Epoch 09/30 | Train Loss=0.7958 | F1=0.6

,epoch,train_loss,train_accuracy,train_precision,train_recall,train_f1,val_loss,val_accuracy,val_precision,val_recall,val_f1,lr,seconds
18,19,0.7600,0.6667,0.6810,0.6535,0.6446,0.9627,0.6290,0.5994,0.5671,0.5465,0.0002,80.2763
19,20,0.7583,0.6671,0.6820,0.6538,0.6452,1.0259,0.6140,0.5890,0.5513,0.5321,0.0002,81.7388
20,21,0.7574,0.6670,0.6828,0.6537,0.6449,0.9422,0.6330,0.6001,0.5709,0.5495,0.0002,80.9994
21,22,0.7567,0.6669,0.6820,0.6533,0.6447,0.9607,0.6286,0.6043,0.5666,0.5528,0.0002,81.0472
22,23,0.7587,0.6666,0.6822,0.6534,0.6449,0.9564,0.6275,0.5969,0.5653,0.5484,0.0002,81.2393
23,24,0.7576,0.6672,0.6829,0.6540,0.6452,0.9598,0.6286,0.6002,0.5669,0.5481,0.0002,80.3828
24,25,0.7549,0.6680,0.6846,0.6545,0.6459,0.9545,0.6306,0.6009,0.5687,0.5483,0.0001,81.2858
25,26,0.7533,0.6693,0.6866,0.6564,0.6481,1.4669,0.5353,0.5368,0.4689,0.4350,0.0001,80.7620
26,27,0.7521,0.6706,0.6873,0.6574,0.6489,1.3132,0.5659,0.5583,0.5008,0.4767,0.0001,81.2028
27,28,0.7514,0.6697,0.6860,0.6566,0.6485,0.9595,0.6318,0.6010,0.5700,0.5522,0.0001,81.7618



CELL 31 COMPLETED


In [ ]:

# ============================================================
# CELL 32 - TRAIN TEMPORAL CNN-BiLSTM (CPU)
# Tai su dung toan bo thu vien va ham cua Cell 31
# ============================================================

# 1. Duong dan luu model rieng
BI_DIR = ROOT / "models/CIC-DDoS2019/Temporal/CNN_BiLSTM_v2_CPU"
BI_DIR.mkdir(parents=True, exist_ok=True)

BI_BEST = BI_DIR / "best.pt"
BI_LAST = BI_DIR / "last.pt"
BI_HISTORY = BI_DIR / "training_history.csv"

torch.manual_seed(SEED)
np.random.seed(SEED)

# 2. Tai su dung du lieu va khoi tao lai DataLoader
generator = torch.Generator().manual_seed(SEED)
train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val)

# 3. Ke thua CNN-LSTM, thay LSTM bang BiLSTM
class TemporalCNNBiLSTM(TemporalCNNLSTM):
    def __init__(self, n_classes=17):
        super().__init__(n_classes)

        self.lstm = torch.nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.classifier[0] = torch.nn.Linear(256, 64)

    def forward(self, x):
        b, t, f = x.shape
        x = self.cnn(x.reshape(b * t, 1, f))
        x = self.flow_embedding(x).reshape(b, t, 128)

        # Lay hidden state cuoi cua ca hai chieu
        _, (h, _) = self.lstm(x)
        x = torch.cat([h[-2], h[-1]], dim=1)

        return self.classifier(x)

model = TemporalCNNBiLSTM(N_CLASSES).to(device)

print("Model: Temporal CNN-BiLSTM")
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

# 4. Loss, Optimizer va Scheduler
# Tai su dung criterion va class_weights cua Cell 31
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5,
    patience=2, min_lr=1e-5
)

config_bi = {
    **config,
    "architecture": "TemporalCNNBiLSTM",
    "bidirectional": True,
    "max_epochs": MAX_EPOCHS
}

# 5. Resume checkpoint neu da train truoc do
start_bi, best_bi, epoch_bi, wait_bi = 1, -1.0, 0, 0
history_bi = []

if BI_LAST.exists():
    ckpt = torch.load(
        BI_LAST, map_location="cpu", weights_only=True
    )

    assert ckpt["config"] == config_bi, "Config khong khop"

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    generator.set_state(ckpt["generator_state"])
    torch.set_rng_state(ckpt["rng_state"])

    start_bi = ckpt["epoch"] + 1
    best_bi = ckpt["best_f1"]
    epoch_bi = ckpt["best_epoch"]
    wait_bi = ckpt["wait"]
    history_bi = ckpt["history"]

    print("Resume from epoch:", start_bi)

# 6. Training Loop
print("\n========== TRAIN CNN-BiLSTM ==========")

for epoch in range(start_bi, MAX_EPOCHS + 1):

    if wait_bi >= PATIENCE:
        print("Early stopping.")
        break

    t0 = perf_counter()

    # Tai su dung ham train/validation cua Cell 31
    train = run_epoch(train_loader, training=True)
    val = run_epoch(val_loader)

    scheduler.step(val["f1"])

    improved = val["f1"] > best_bi + 1e-4

    if improved:
        best_bi, epoch_bi, wait_bi = val["f1"], epoch, 0

        save_checkpoint({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "config": config_bi,
            "class_names": class_names,
            "val_metrics": val
        }, BI_BEST)
    else:
        wait_bi += 1

    history_bi.append({
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train.items()},
        **{f"val_{k}": v for k, v in val.items()},
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": perf_counter() - t0
    })

    save_checkpoint({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "generator_state": generator.get_state(),
        "rng_state": torch.get_rng_state(),
        "best_f1": best_bi,
        "best_epoch": epoch_bi,
        "wait": wait_bi,
        "history": history_bi,
        "config": config_bi
    }, BI_LAST)

    pd.DataFrame(history_bi).to_csv(BI_HISTORY, index=False)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss={train['loss']:.4f} | "
        f"F1={train['f1']:.4f} | "
        f"Val Loss={val['loss']:.4f} | "
        f"Acc={val['accuracy']:.4f} | "
        f"F1={val['f1']:.4f} | "
        f"{perf_counter()-t0:.1f}s"
        + (" [NEW BEST]" if improved else ""),
        flush=True
    )

# 7. Tong ket
print("\n========== TRAINING SUMMARY ==========")
print("Best epoch:", epoch_bi)
print("Best Validation Macro-F1:", round(best_bi, 6))
print("Best model:", BI_BEST)

display(pd.DataFrame(history_bi).tail(10).round(4))

print("\nCELL 32 COMPLETED")


Model: Temporal CNN-BiLSTM
Device: cpu
Parameters: 321233

========== TRAIN CNN-BiLSTM ==========
Epoch 01/30 | Train Loss=1.2365 | F1=0.4833 | Val Loss=0.8958 | Acc=0.6341 | F1=0.5076 | 112.3s [NEW BEST]
Epoch 02/30 | Train Loss=0.8900 | F1=0.5754 | Val Loss=0.8511 | Acc=0.6316 | F1=0.5313 | 90.6s [NEW BEST]
Epoch 03/30 | Train Loss=0.8566 | F1=0.5918 | Val Loss=1.0470 | Acc=0.6068 | F1=0.4775 | 91.0s
Epoch 04/30 | Train Loss=0.8347 | F1=0.6004 | Val Loss=0.9018 | Acc=0.5850 | F1=0.4952 | 90.0s
Epoch 05/30 | Train Loss=0.8258 | F1=0.6071 | Val Loss=0.8616 | Acc=0.6049 | F1=0.5175 | 89.5s
Epoch 06/30 | Train Loss=0.8038 | F1=0.6183 | Val Loss=0.8732 | Acc=0.6146 | F1=0.5247 | 90.0s
Epoch 07/30 | Train Loss=0.7985 | F1=0.6215 | Val Loss=0.8629 | Acc=0.6226 | F1=0.5332 | 90.4s [NEW BEST]
Epoch 08/30 | Train Loss=0.7969 | F1=0.6220 | Val Loss=0.8689 | Acc=0.6225 | F1=0.5448 | 90.1s [NEW BEST]
Epoch 09/30 | Train Loss=0.7901 | F1=0.6250 | Val Loss=0.8830 | Acc=0.6222 | F1=0.5358 | 90.2s
Ep

,epoch,train_loss,train_accuracy,train_precision,train_recall,train_f1,val_loss,val_accuracy,val_precision,val_recall,val_f1,lr,seconds
20,21,0.7625,0.6646,0.6795,0.6511,0.6430,0.9097,0.6280,0.5957,0.5660,0.5439,0.0002,89.6884
21,22,0.7640,0.6637,0.6776,0.6498,0.6413,0.8940,0.6319,0.6032,0.5700,0.5540,0.0002,89.8250
22,23,0.7622,0.6653,0.6794,0.6514,0.6430,0.9040,0.6178,0.6142,0.5563,0.5464,0.0002,90.1817
23,24,0.7626,0.6648,0.6783,0.6514,0.6434,0.9106,0.6394,0.6143,0.5780,0.5593,0.0002,89.5178
24,25,0.7594,0.6647,0.6775,0.6510,0.6430,0.9013,0.6351,0.5988,0.5734,0.5509,0.0002,89.3968
25,26,0.7602,0.6661,0.6782,0.6528,0.6449,1.2046,0.5706,0.5441,0.5053,0.4735,0.0002,90.3493
26,27,0.7568,0.6668,0.6809,0.6534,0.6446,1.1882,0.5741,0.5592,0.5097,0.4842,0.0001,90.1999
27,28,0.7532,0.6705,0.6849,0.6573,0.6493,0.8989,0.6314,0.6072,0.5696,0.5522,0.0001,89.7724
28,29,0.7516,0.6692,0.6833,0.6562,0.6482,0.9522,0.6097,0.5847,0.5467,0.5174,0.0001,90.0988
29,30,0.7517,0.6712,0.6867,0.6580,0.6499,0.9132,0.6280,0.5998,0.5665,0.5483,0.0001,91.9994



CELL 32 COMPLETED


In [ ]:

# ============================================================
# CELL 33 - MODEL EVALUATION & SELECTION
# Khong train lai, tai su dung Cell 31 va 32
# ============================================================

RESULT_DIR = ROOT / "results/Temporal/Final"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BENIGN_ID = class_names.index("BENIGN")

# 1. Load hai best checkpoint
models = {
    "CNN-LSTM": TemporalCNNLSTM(N_CLASSES),
    "CNN-BiLSTM": TemporalCNNBiLSTM(N_CLASSES)
}

paths = {
    "CNN-LSTM": MODEL_DIR / "best.pt",
    "CNN-BiLSTM": BI_DIR / "best.pt"
}

for name, net in models.items():
    ckpt = torch.load(paths[name], map_location="cpu",
                      weights_only=True)

    assert ckpt["class_names"] == class_names
    net.load_state_dict(ckpt["model_state_dict"])
    net.eval()

# 2. Ham danh gia da lop va nhi phan
@torch.inference_mode()
def evaluate(net, X, y):
    predictions = []

    for xb, _ in make_loader(X, y):
        predictions.extend(net(xb).argmax(1).tolist())

    pred = np.array(predictions)

    p, r, f1, _ = precision_recall_fscore_support(
        y, pred, labels=np.arange(N_CLASSES),
        average="macro", zero_division=0
    )

    # BENIGN = 0, Attack = 1
    actual_attack = y != BENIGN_ID
    predicted_attack = pred != BENIGN_ID

    tp = int((actual_attack & predicted_attack).sum())
    tn = int((~actual_attack & ~predicted_attack).sum())
    fp = int((~actual_attack & predicted_attack).sum())
    fn = int((actual_attack & ~predicted_attack).sum())

    metrics = {
        "Accuracy": float((y == pred).mean()),
        "Precision_Macro": float(p),
        "Recall_Macro": float(r),
        "Macro_F1": float(f1),
        "Binary_Accuracy": (tp + tn) / len(y),
        "Attack_Precision": tp / max(tp + fp, 1),
        "Attack_Recall": tp / max(tp + fn, 1),
        "Binary_F1": 2 * tp / max(2 * tp + fp + fn, 1),
        "FPR": fp / max(fp + tn, 1),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }

    return metrics, pred

# 3. Do inference latency, batch size = 1
@torch.inference_mode()
def benchmark(net, X, n=100):
    samples = torch.from_numpy(X[:n])

    # Warmup
    for _ in range(10):
        net(samples[:1])

    times = []
    for sample in samples:
        t0 = perf_counter()
        net(sample.unsqueeze(0))
        times.append((perf_counter() - t0) * 1000)

    return {
        "Latency_P50_ms": float(np.median(times)),
        "Latency_P95_ms": float(np.percentile(times, 95))
    }

# 4. So sanh tren Validation
rows = []

for name, net in models.items():
    metrics, _ = evaluate(net, X_val, y_val)
    latency = benchmark(net, X_val)

    rows.append({
        "Model": name,
        **metrics,
        **latency,
        "Parameters": sum(p.numel() for p in net.parameters())
    })

comparison = pd.DataFrame(rows).set_index("Model")

print("=== VALIDATION MODEL COMPARISON ===")
display(comparison.T.round(4))

comparison.to_csv(RESULT_DIR / "validation_comparison.csv")

# 5. Chon mo hinh theo muc tieu MVP
eligible = comparison[
    (comparison["Attack_Recall"] >= 0.95)
    & (comparison["FPR"] <= 0.05)
].sort_values("Macro_F1", ascending=False)

selected = None

if len(eligible) == 0:
    print("\nWARNING: Chua model nao dat muc tieu MVP.")
    print("Can kiem tra nguong canh bao tren Validation.")
    print("Chua danh gia Test.")

else:
    selected = eligible.index[0]

    # Neu F1 chenh lech duoi 2 diem %, uu tien CNN-LSTM
    if "CNN-LSTM" in eligible.index:
        gap = (
            eligible.iloc[0]["Macro_F1"]
            - eligible.loc["CNN-LSTM", "Macro_F1"]
        )
        if gap < 0.02:
            selected = "CNN-LSTM"

    print("\nSELECTED MODEL:", selected)

# 6. Chi danh gia Test sau khi chon model
if selected is not None:
    X_test, y_test = load_data("test")

    test_metrics, test_pred = evaluate(
        models[selected], X_test, y_test
    )

    print("\n=== FINAL TEST METRICS ===")
    test_df = pd.DataFrame(
        [test_metrics], index=[selected]
    )
    display(test_df.T.round(4))

    # Confusion Matrix
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
    np.add.at(cm, (y_test, test_pred), 1)

    cm_df = pd.DataFrame(
        cm, index=class_names, columns=class_names
    )

    print("\n=== CONFUSION MATRIX ===")
    display(cm_df)

    # Danh gia tung lop
    p, r, f1, support = precision_recall_fscore_support(
        y_test, test_pred, labels=np.arange(N_CLASSES),
        zero_division=0
    )

    report = pd.DataFrame({
        "Class": class_names,
        "Precision": p, "Recall": r,
        "F1": f1, "Support": support
    })

    display(report.round(4))

    test_df.to_csv(RESULT_DIR / "test_metrics.csv")
    cm_df.to_csv(RESULT_DIR / "confusion_matrix.csv")
    report.to_csv(RESULT_DIR / "classification_report.csv",
                  index=False)

    (RESULT_DIR / "selected_model.json").write_text(
        json.dumps({
            "model": selected,
            "checkpoint": str(paths[selected]),
            "dataset": "CIC-DDoS2019_v2",
            "input_shape": [8, 66],
            "classes": class_names
        }, indent=2)
    )

print("\nCELL 33 COMPLETED")


=== VALIDATION MODEL COMPARISON ===


Model,CNN-LSTM,CNN-BiLSTM
Accuracy,0.6286,0.6394
Precision_Macro,0.6043,0.6143
Recall_Macro,0.5666,0.5780
Macro_F1,0.5528,0.5593
Binary_Accuracy,0.9966,0.9965
Attack_Precision,0.9993,0.9992
Attack_Recall,0.9971,0.9972
Binary_F1,0.9982,0.9982
FPR,0.0175,0.0226
TP,10737.0000,10738.0000



SELECTED MODEL: CNN-LSTM

=== FINAL TEST METRICS ===


,CNN-LSTM
Accuracy,0.5536
Precision_Macro,0.5190
Recall_Macro,0.5509
Macro_F1,0.5243
Binary_Accuracy,0.9969
Attack_Precision,0.9986
Attack_Recall,0.9981
Binary_F1,0.9984
FPR,0.0333
TP,15077.0000



=== CONFUSION MATRIX ===


,BENIGN,DrDoS_DNS,DrDoS_LDAP,DrDoS_MSSQL,DrDoS_NTP,DrDoS_NetBIOS,DrDoS_SNMP,DrDoS_SSDP,DrDoS_UDP,LDAP,MSSQL,NetBIOS,Portmap,Syn,TFTP,UDP,UDPLag
BENIGN,610,3,0,0,14,0,0,0,0,0,0,0,0,1,0,0,3
DrDoS_DNS,2,542,326,0,26,0,61,5,0,103,0,1,0,0,0,0,34
DrDoS_LDAP,0,58,497,0,0,0,28,0,0,233,0,0,0,0,0,0,0
DrDoS_MSSQL,0,1,0,656,0,0,0,0,0,0,526,0,0,0,0,0,0
DrDoS_NTP,0,182,0,0,658,0,0,0,0,0,1,0,0,0,0,0,0
DrDoS_NetBIOS,0,0,0,0,0,0,0,0,0,0,0,556,691,0,0,0,0
DrDoS_SNMP,0,3,53,0,0,0,648,0,0,91,0,0,0,0,0,0,1
DrDoS_SSDP,0,0,0,0,0,0,0,196,346,0,0,3,2,0,0,248,0
DrDoS_UDP,0,0,0,0,0,0,0,223,271,0,0,2,1,0,0,347,5
LDAP,0,17,436,0,0,0,33,0,0,540,0,0,0,0,0,0,0


,Class,Precision,Recall,F1,Support
0,BENIGN,0.9561,0.9667,0.9614,631
1,DrDoS_DNS,0.6056,0.4927,0.5434,1100
2,DrDoS_LDAP,0.3788,0.6091,0.4671,816
3,DrDoS_MSSQL,0.6833,0.5545,0.6122,1183
4,DrDoS_NTP,0.7985,0.7824,0.7904,841
5,DrDoS_NetBIOS,0.0000,0.0000,0.0000,1247
6,DrDoS_SNMP,0.8372,0.8141,0.8255,796
7,DrDoS_SSDP,0.3544,0.2465,0.2908,795
8,DrDoS_UDP,0.3094,0.3192,0.3142,849
9,LDAP,0.5584,0.5263,0.5419,1026



CELL 33 COMPLETED


In [ ]:

# ============================================================
# CELL 34 - FINAL PIPELINE EVALUATION
# Gaussian + Temporal CNN-LSTM
# ============================================================

if "LedoitWolf" not in globals():
    from sklearn.covariance import LedoitWolf

if "joblib" not in globals():
    import joblib

PIPELINE_DIR = ROOT / "results/Temporal/Pipeline"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Su dung CNN-LSTM da duoc chon
net = models["CNN-LSTM"]
net.eval()

# 2. Fit Gaussian chi tren BENIGN cua Train V2
# Su dung 66 features cua flow cuoi
X_benign = X_train[y_train == BENIGN_ID, -1, :]

gaussian = LedoitWolf().fit(X_benign)

# Chon nguong bang BENIGN Validation, khong dung Test
val_scores = gaussian.mahalanobis(X_val[:, -1, :])

threshold = float(np.quantile(
    val_scores[y_val == BENIGN_ID], 0.99
))

print("Gaussian training samples:", len(X_benign))
print("Gaussian threshold:", round(threshold, 4))

# 3. Ham danh gia phat hien nhi phan
def binary_metrics(actual, predicted):
    tp = int((actual & predicted).sum())
    tn = int((~actual & ~predicted).sum())
    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    return {
        "Accuracy": (tp + tn) / len(actual),
        "Precision": tp / max(tp + fp, 1),
        "Recall": tp / max(tp + fn, 1),
        "F1": 2 * tp / max(2 * tp + fp + fn, 1),
        "FPR": fp / max(fp + tn, 1),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }

# 4. So sanh cac pipeline tren cung du lieu
def compare_pipelines(X, y, predictions):
    actual = y != BENIGN_ID
    cnn_attack = predictions != BENIGN_ID

    scores = gaussian.mahalanobis(X[:, -1, :])
    gaussian_attack = scores > threshold

    pipelines = {
        "Gaussian Only": gaussian_attack,
        "CNN-LSTM Only": cnn_attack,
        "Gaussian AND CNN-LSTM": gaussian_attack & cnn_attack,
        "Gaussian OR CNN-LSTM": gaussian_attack | cnn_attack
    }

    return pd.DataFrame([
        {"Pipeline": name, **binary_metrics(actual, pred)}
        for name, pred in pipelines.items()
    ])

# 5. Validation
_, val_pred = evaluate(net, X_val, y_val)

val_results = compare_pipelines(X_val, y_val, val_pred)

print("\n=== VALIDATION PIPELINE COMPARISON ===")
display(val_results.round(4))

# Kien truc mac dinh: CNN-LSTM quyet dinh canh bao,
# Gaussian cung cap diem bat thuong bo sung.
# Khong dung Test de dieu chinh nguong.

# 6. Test - tai su dung test_pred tu Cell 33
test_results = compare_pipelines(X_test, y_test, test_pred)

print("\n=== TEST PIPELINE COMPARISON ===")
display(test_results.round(4))

# 7. Luu ket qua
val_results.to_csv(
    PIPELINE_DIR / "validation_comparison.csv", index=False
)

test_results.to_csv(
    PIPELINE_DIR / "test_comparison.csv", index=False
)

joblib.dump({
    "model": gaussian,
    "threshold": threshold,
    "input": "last_flow_scaled",
    "n_features": 66,
    "dataset": "CIC-DDoS2019_v2"
}, PIPELINE_DIR / "gaussian_temporal.joblib")

print("\nGaussian model saved.")
print("CELL 34 COMPLETED")


Gaussian training samples: 618
Gaussian threshold: 185.0203

=== VALIDATION PIPELINE COMPARISON ===


,Pipeline,Accuracy,Precision,Recall,F1,FPR,TP,TN,FP,FN
0,Gaussian Only,0.0356,0.4286,0.0003,0.0006,0.0100,3,395,4,10765
1,CNN-LSTM Only,0.9966,0.9993,0.9971,0.9982,0.0175,10737,392,7,31
2,Gaussian AND CNN-LSTM,0.0360,1.0000,0.0003,0.0006,0.0000,3,399,0,10765
3,Gaussian OR CNN-LSTM,0.9962,0.9990,0.9971,0.9980,0.0276,10737,388,11,31



=== TEST PIPELINE COMPARISON ===


,Pipeline,Accuracy,Precision,Recall,F1,FPR,TP,TN,FP,FN
0,Gaussian Only,0.0403,0.5484,0.0011,0.0022,0.0222,17,617,14,15088
1,CNN-LSTM Only,0.9969,0.9986,0.9981,0.9984,0.0333,15077,610,21,28
2,Gaussian AND CNN-LSTM,0.0412,1.0000,0.0011,0.0022,0.0000,17,631,0,15088
3,Gaussian OR CNN-LSTM,0.9960,0.9977,0.9981,0.9979,0.0555,15077,596,35,28



Gaussian model saved.
CELL 34 COMPLETED


In [ ]:

# ============================================================
# CELL 35 - EXPORT MODEL & INFERENCE TEST
# ============================================================

if "shutil" not in globals():
    import shutil

EXPORT_DIR = ROOT / "deployment/NIDS_Model_v1"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load checkpoint CNN-LSTM tốt nhất
checkpoint = torch.load(
    MODEL_DIR / "best.pt",
    map_location="cpu",
    weights_only=True
)

net = TemporalCNNLSTM(N_CLASSES)
net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

# 2. Export trọng số và TorchScript
shutil.copy2(MODEL_DIR / "best.pt", EXPORT_DIR / "best.pt")

example = torch.from_numpy(X_val[:1])

with torch.no_grad():
    scripted = torch.jit.trace(net, example)

scripted.save(str(EXPORT_DIR / "cnn_lstm.pt"))

# 3. Copy preprocessing và Gaussian
for filename in ["temporal_preprocessor.joblib",
                 "label_mapping.json"]:
    shutil.copy2(DATA_DIR / filename, EXPORT_DIR / filename)

shutil.copy2(
    PIPELINE_DIR / "gaussian_temporal.joblib",
    EXPORT_DIR / "gaussian_temporal.joblib"
)

preprocessor = joblib.load(
    EXPORT_DIR / "temporal_preprocessor.joblib"
)

# 4. Lưu cấu hình triển khai
deploy_config = {
    "model": "TemporalCNNLSTM",
    "dataset": "CIC-DDoS2019_v2",
    "window_size": 8,
    "stride": 8,
    "n_features": 66,
    "n_classes": 17,
    "max_window_seconds": 300,
    "grouping": "Source IP",
    "feature_names": list(preprocessor["feature_names"]),
    "class_names": class_names,
    "benign_id": BENIGN_ID,
    "target": "last_flow_label",
    "decision": "CNN-LSTM argmax",
    "gaussian_mode": "auxiliary_score",
    "gaussian_threshold": float(threshold),
    "pytorch_version": str(torch.__version__),
    "device": "cpu",
    "best_epoch": int(checkpoint["epoch"])
}

(EXPORT_DIR / "config.json").write_text(
    json.dumps(deploy_config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 5. Lưu kết quả đánh giá
for filename in [
    "validation_comparison.csv",
    "test_metrics.csv",
    "confusion_matrix.csv",
    "classification_report.csv"
]:
    shutil.copy2(RESULT_DIR / filename, EXPORT_DIR / filename)

shutil.copy2(
    PIPELINE_DIR / "test_comparison.csv",
    EXPORT_DIR / "pipeline_comparison.csv"
)

# 6. Load lại TorchScript và kiểm tra kết quả
loaded_model = torch.jit.load(
    str(EXPORT_DIR / "cnn_lstm.pt"),
    map_location="cpu"
).eval()

samples = torch.from_numpy(X_val[:32])

with torch.inference_mode():
    original = net(samples)
    exported = loaded_model(samples)

    assert torch.allclose(
        original, exported, atol=1e-5, rtol=1e-4
    ), "Exported model output mismatch"

    probabilities = torch.softmax(exported, dim=1)
    confidence, predictions = probabilities.max(dim=1)

print("=== INFERENCE TEST ===")
print("Model output consistency: PASS")
print("Input:", samples.shape)
print("Output:", exported.shape)

display(pd.DataFrame({
    "Actual": [class_names[i] for i in y_val[:32]],
    "Predicted": [class_names[i] for i in predictions.tolist()],
    "Detection": [
        "BENIGN" if i == BENIGN_ID else "ATTACK"
        for i in predictions.tolist()
    ],
    "Confidence": confidence.numpy().round(4)
}).head(10))

# 7. Đóng gói ZIP
zip_path = shutil.make_archive(
    str(EXPORT_DIR), "zip", root_dir=EXPORT_DIR
)

print("\n=== DEPLOYMENT PACKAGE ===")

for file in sorted(EXPORT_DIR.iterdir()):
    if file.is_file():
        print(file.name, f"| {file.stat().st_size / 1024:.2f} KB")

print("\nZIP:", zip_path)
print("CELL 35 COMPLETED")


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


=== INFERENCE TEST ===
Model output consistency: PASS
Input: torch.Size([32, 8, 66])
Output: torch.Size([32, 17])


/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


,Actual,Predicted,Detection,Confidence
0,DrDoS_DNS,DrDoS_DNS,ATTACK,0.9999
1,DrDoS_DNS,LDAP,ATTACK,0.7451
2,DrDoS_DNS,LDAP,ATTACK,0.5008
3,DrDoS_DNS,DrDoS_DNS,ATTACK,0.9999
4,DrDoS_DNS,LDAP,ATTACK,0.4340
5,DrDoS_DNS,DrDoS_DNS,ATTACK,0.9998
6,DrDoS_DNS,DrDoS_DNS,ATTACK,0.9998
7,DrDoS_DNS,DrDoS_LDAP,ATTACK,0.4449
8,DrDoS_DNS,DrDoS_DNS,ATTACK,0.6959
9,DrDoS_DNS,DrDoS_DNS,ATTACK,0.7195



=== DEPLOYMENT PACKAGE ===
best.pt | 714.88 KB
classification_report.csv | 1.15 KB
cnn_lstm.pt | 750.47 KB
config.json | 2.24 KB
confusion_matrix.csv | 0.93 KB
gaussian_temporal.joblib | 34.99 KB
label_mapping.json | 0.31 KB
pipeline_comparison.csv | 0.54 KB
temporal_preprocessor.joblib | 4.27 KB
test_metrics.csv | 0.31 KB
validation_comparison.csv | 0.63 KB

ZIP: /content/drive/MyDrive/NIDS_Project/deployment/NIDS_Model_v1.zip
CELL 35 COMPLETED


In [ ]:

# CELL 36.1 - RECOVERY & MODEL AUDIT

from google.colab import drive
from pathlib import Path
from importlib.metadata import version

import torch
import numpy as np
import pandas as pd
import joblib
import json

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/NIDS_Project")

DATA_DIR = ROOT / "datasets/prepared/ready/CIC-IDS2017"
MODEL_DIR = ROOT / "models/CIC-IDS2017/DL"
EXPORT_DIR = ROOT / "deployment/CIC-IDS2017"

assert ROOT.exists(), "Khong tim thay NIDS_Project"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")

# Kiem tra cac checkpoint da huan luyen
print("=== CHECKPOINTS ===")

for path in sorted(MODEL_DIR.glob("*.pt")):
    print(path.name, "|", round(path.stat().st_size / 1024, 2), "KB")

# Kiem tra ket qua danh gia
print("\n=== MODEL COMPARISON ===")

for folder in [ROOT / "models", ROOT / "results"]:
    for path in folder.rglob("dl_models_summary.csv"):
        print("\nFile:", path)
        display(pd.read_csv(path))

# Kiem tra preprocessor
print("\n=== PREPROCESSOR ===")

preprocessor = joblib.load(DATA_DIR / "preprocessor.joblib")

print("Keys:", list(preprocessor.keys()))
print("Number of features:", len(preprocessor["feature_names"]))

# Kiem tra label mapping
mapping = json.loads(
    (DATA_DIR / "label_mapping.json").read_text()
)

print("\n=== LABEL MAPPING ===")
print(mapping)

# Kiem tra du lieu Validation
with np.load(DATA_DIR / "val.npz") as data:
    print("\n=== VALIDATION DATA ===")
    print("Keys:", data.files)
    print("X shape:", data["X"].shape)
    print("y shape:", data["y"].shape)

# Kiem tra cau truc checkpoint
print("\n=== CHECKPOINT STRUCTURE ===")

for path in sorted(MODEL_DIR.glob("*.pt")):
    name = path.stem.lower().replace("-", "").replace("_", "")

    if "cnnlstm" not in name and "cnnbilstm" not in name:
        continue

    print("\nModel:", path.name)

    try:
        ckpt = torch.load(
            path, map_location="cpu", weights_only=True
        )

        print("Keys:", list(ckpt.keys()))

        state = ckpt.get("model_state_dict",
                         ckpt.get("state_dict", ckpt))

        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                print(key, tuple(value.shape))

    except Exception as e:
        print("Load error:", str(e))

# Kiem tra phien ban thu vien
print("\n=== ENVIRONMENT ===")

for lib in ["torch", "numpy", "scikit-learn",
            "scipy", "pandas", "joblib"]:
    print(f"{lib}=={version(lib)}")

print("\nCELL 36.1 COMPLETED")


Mounted at /content/drive
=== CHECKPOINTS ===
BiLSTM_best.pt | 144.57 KB
CNN-BiLSTM_best.pt | 262.64 KB
CNN-LSTM_best.pt | 144.5 KB
CNN1D_best.pt | 35.68 KB
LSTM_best.pt | 73.42 KB

=== MODEL COMPARISON ===

File: /content/drive/MyDrive/NIDS_Project/results/dl_models_summary.csv


,Dataset,Model,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy,Best_Epoch
0,CIC-IDS2017,CNN1D,0.849138,0.855036,0.909894,0.887052,12
1,CIC-IDS2017,LSTM,0.837153,0.840775,0.923087,0.887809,17
2,CIC-IDS2017,BiLSTM,0.810069,0.814100,0.897889,0.858528,15
3,CIC-IDS2017,CNN-LSTM,0.862683,0.860558,0.944855,0.893863,7
4,CIC-IDS2017,CNN-BiLSTM,0.867516,0.872032,0.954749,0.905239,15
5,CIC-DDoS2019,CNN1D,0.421669,0.418515,0.717989,0.462041,7
6,CIC-DDoS2019,LSTM,0.425810,0.423225,0.715372,0.465197,17
7,CIC-DDoS2019,BiLSTM,0.433861,0.426553,0.722797,0.476992,15
8,CIC-DDoS2019,CNN-LSTM,0.481173,0.482639,0.735029,0.528511,15
9,CIC-DDoS2019,CNN-BiLSTM,0.493094,0.493523,0.740263,0.529573,14



=== PREPROCESSOR ===
Keys: ['feature_names', 'imputer', 'scaler', 'label_encoder', 'dataset']
Number of features: 70

=== LABEL MAPPING ===
{'0': 'BENIGN', '1': 'Bot', '2': 'DDoS', '3': 'DoS GoldenEye', '4': 'DoS Hulk', '5': 'DoS Slowhttptest', '6': 'DoS slowloris', '7': 'FTP-Patator', '8': 'PortScan', '9': 'SSH-Patator', '10': 'WebAttack-BruteForce', '11': 'WebAttack-XSS'}

=== VALIDATION DATA ===
Keys: ['X', 'y']
X shape: (7580, 70)
y shape: (7580,)

=== CHECKPOINT STRUCTURE ===

Model: CNN-BiLSTM_best.pt
Keys: ['model_state', 'architecture', 'dataset', 'n_features', 'n_classes', 'class_names', 'best_epoch', 'best_val_macro_f1']

Model: CNN-LSTM_best.pt
Keys: ['model_state', 'architecture', 'dataset', 'n_features', 'n_classes', 'class_names', 'best_epoch', 'best_val_macro_f1']

=== ENVIRONMENT ===
torch==2.14.1+cpu
numpy==2.4.6
scikit-learn==1.6.1
scipy==1.16.3
pandas==2.2.3
joblib==1.6.0

CELL 36.1 COMPLETED


In [ ]:

# ============================================================
# CELL 36.2 - INSPECT CNN-LSTM (CIC-IDS2017)
# ============================================================

BEST_PATH = MODEL_DIR / "CNN-LSTM_best.pt"
GAUSSIAN_PATH = (
    ROOT / "models/CIC-IDS2017/Gaussian/gaussian_ledoitwolf.joblib"
)

assert BEST_PATH.exists(), "Khong tim thay checkpoint CNN-LSTM"

# 1. Load checkpoint
checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=True
)

print("=== MODEL INFORMATION ===")

for key, value in checkpoint.items():
    if key != "model_state":
        print(f"{key}: {value}")

# 2. Kiem tra cau truc cac layer
state_dict = checkpoint["model_state"]

print("\n=== MODEL LAYERS ===")

for name, weight in state_dict.items():
    print(name, tuple(weight.shape))

# 3. Kiem tra bo tien xu ly
feature_names = list(preprocessor["feature_names"])
imputer = preprocessor["imputer"]
scaler = preprocessor["scaler"]

print("\n=== PREPROCESSOR ===")
print("Number of features:", len(feature_names))
print("Imputer:", type(imputer).__name__)
print("Scaler:", type(scaler).__name__)

assert len(feature_names) == checkpoint["n_features"]

# 4. Danh sach dac trung dau vao
print("\n=== FEATURE NAMES ===")

for i, name in enumerate(feature_names):
    print(f"{i + 1:02d}. {name}")

# 5. Kiem tra label mapping
class_names = [
    mapping[str(i)] for i in range(len(mapping))
]

assert class_names == list(checkpoint["class_names"])
assert len(class_names) == checkpoint["n_classes"]

print("\n=== CLASS NAMES ===")

for i, name in enumerate(class_names):
    print(i, name)

# 6. Kiem tra Gaussian model
print("\n=== GAUSSIAN MODEL ===")

if GAUSSIAN_PATH.exists():
    gaussian_artifact = joblib.load(GAUSSIAN_PATH)

    print("Path:", GAUSSIAN_PATH)
    print("Size:", round(GAUSSIAN_PATH.stat().st_size / 1024, 2), "KB")
    print("Type:", type(gaussian_artifact).__name__)

    if isinstance(gaussian_artifact, dict):
        print("Keys:", list(gaussian_artifact.keys()))

        if "threshold" in gaussian_artifact:
            print("Threshold:", gaussian_artifact["threshold"])

else:
    print("WARNING: Gaussian artifact not found")

# 7. Doc ket qua danh gia CNN-LSTM
summary = pd.read_csv(
    ROOT / "results/dl_models_summary.csv"
)

cnn_lstm_result = summary[
    (summary["Dataset"] == "CIC-IDS2017")
    & (summary["Model"] == "CNN-LSTM")
]

print("\n=== CNN-LSTM EVALUATION ===")
display(cnn_lstm_result.reset_index(drop=True))

# 8. Tong ket
print("\n=== EXPORT READINESS ===")
print("Checkpoint:", BEST_PATH.exists())
print("Preprocessor:", (DATA_DIR / "preprocessor.joblib").exists())
print("Label mapping:", (DATA_DIR / "label_mapping.json").exists())
print("Gaussian:", GAUSSIAN_PATH.exists())
print("Validation:", (DATA_DIR / "val.npz").exists())
print("Test:", (DATA_DIR / "test.npz").exists())

print("\nCELL 36.2 COMPLETED")


=== MODEL INFORMATION ===
architecture: CNN-LSTM
dataset: CIC-IDS2017
n_features: 70
n_classes: 12
class_names: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'PortScan', 'SSH-Patator', 'WebAttack-BruteForce', 'WebAttack-XSS']
best_epoch: 7
best_val_macro_f1: 0.8626832794161796

=== MODEL LAYERS ===
cnn.0.weight (32, 1, 5)
cnn.0.bias (32,)
cnn.1.weight (32,)
cnn.1.bias (32,)
cnn.1.running_mean (32,)
cnn.1.running_var (32,)
cnn.1.num_batches_tracked ()
cnn.4.weight (48, 32, 3)
cnn.4.bias (48,)
cnn.5.weight (48,)
cnn.5.bias (48,)
cnn.5.running_mean (48,)
cnn.5.running_var (48,)
cnn.5.num_batches_tracked ()
lstm.weight_ih_l0 (256, 48)
lstm.weight_hh_l0 (256, 64)
lstm.bias_ih_l0 (256,)
lstm.bias_hh_l0 (256,)
classifier.1.weight (12, 64)
classifier.1.bias (12,)

=== PREPROCESSOR ===
Number of features: 70
Imputer: SimpleImputer
Scaler: StandardScaler

=== FEATURE NAMES ===
01. Destination Port
02. Protocol
03. Flow Duration
04. Tot

,Dataset,Model,Val_Macro_F1,Test_Macro_F1,Test_Accuracy,Test_Balanced_Accuracy,Best_Epoch
0,CIC-IDS2017,CNN-LSTM,0.862683,0.860558,0.944855,0.893863,7



=== EXPORT READINESS ===
Checkpoint: True
Preprocessor: True
Label mapping: True
Gaussian: True
Validation: True
Test: True

CELL 36.2 COMPLETED


In [ ]:

# ============================================================
# CELL 36.3 - EXPORT CNN-LSTM CIC-IDS2017
# ============================================================

from torch import nn

if "shutil" not in globals():
    import shutil

# 1. Khoi phuc dung kien truc tu Cell 17
class FlowFeatureCNNLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm1d(48),
            nn.ReLU()
        )

        self.lstm = nn.LSTM(
            input_size=48,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        z = self.cnn(x.unsqueeze(1))
        z = z.transpose(1, 2)
        z, _ = self.lstm(z)
        return self.classifier(z[:, -1, :])


# 2. Load checkpoint tot nhat
net = FlowFeatureCNNLSTM(checkpoint["n_classes"])

net.load_state_dict(
    checkpoint["model_state"], strict=True
)

net.eval()

print("Model:", checkpoint["architecture"])
print("Best epoch:", checkpoint["best_epoch"])
print("Parameters:", sum(p.numel() for p in net.parameters()))


# 3. Load du lieu Validation da tien xu ly
with np.load(DATA_DIR / "val.npz") as data:
    X_val = data["X"].astype(np.float32)
    y_val = data["y"].astype(np.int64)

assert X_val.shape[1] == 70
assert np.isfinite(X_val).all()

samples = torch.from_numpy(X_val[:32])


# 4. Export TorchScript
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

with torch.no_grad():
    scripted = torch.jit.trace(net, samples[:1])

scripted.save(str(EXPORT_DIR / "cnn_lstm.pt"))

shutil.copy2(BEST_PATH, EXPORT_DIR / "best.pt")


# 5. Copy preprocessor, label mapping, Gaussian
for filename in [
    "preprocessor.joblib",
    "label_mapping.json"
]:
    shutil.copy2(
        DATA_DIR / filename,
        EXPORT_DIR / filename
    )

shutil.copy2(
    GAUSSIAN_PATH,
    EXPORT_DIR / "gaussian.joblib"
)


# 6. Luu cau hinh trien khai
deploy_config = {
    "model": "FlowFeatureCNNLSTM",
    "architecture": "CNN-LSTM",
    "dataset": "CIC-IDS2017",
    "input_type": "single_flow",
    "input_shape": [70],
    "n_features": 70,
    "n_classes": 12,
    "feature_names": feature_names,
    "class_names": class_names,
    "benign_id": class_names.index("BENIGN"),
    "window_size": 1,
    "stride": 1,
    "target": "current_flow_label",
    "decision": "CNN-LSTM argmax",
    "gaussian_mode": "auxiliary_score",
    "gaussian_threshold": float(
        gaussian_artifact["threshold"]
    ),
    "device": "cpu",
    "best_epoch": int(checkpoint["best_epoch"]),
    "pytorch_version": str(torch.__version__)
}

(EXPORT_DIR / "config.json").write_text(
    json.dumps(deploy_config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# 7. Copy ket qua danh gia da co
OLD_RESULTS = ROOT / "results/CIC-IDS2017/DL"

result_files = {
    "CNN-LSTM_metrics.json": "test_metrics.json",
    "CNN-LSTM_classification_report.csv": "classification_report.csv",
    "CNN-LSTM_confusion_matrix.csv": "confusion_matrix.csv",
    "CNN-LSTM_history.csv": "training_history.csv"
}

for source, target in result_files.items():
    shutil.copy2(
        OLD_RESULTS / source,
        EXPORT_DIR / target
    )

summary[summary["Dataset"] == "CIC-IDS2017"].to_csv(
    EXPORT_DIR / "model_comparison.csv",
    index=False
)


# 8. Load lai model va kiem tra inference
loaded_model = torch.jit.load(
    str(EXPORT_DIR / "cnn_lstm.pt"),
    map_location="cpu"
).eval()

with torch.inference_mode():
    # Kiem tra ca batch size 1 va batch size 32
    for batch in [samples[:1], samples]:
        original = net(batch)
        exported = loaded_model(batch)

        assert torch.allclose(
            original, exported,
            atol=1e-5, rtol=1e-4
        ), "Model output mismatch"

    probabilities = torch.softmax(exported, dim=1)
    confidence, predictions = probabilities.max(dim=1)

print("\n=== INFERENCE TEST ===")
print("Model output consistency: PASS")
print("Input:", samples.shape)
print("Output:", exported.shape)

display(pd.DataFrame({
    "Actual": [class_names[i] for i in y_val[:32]],
    "Predicted": [class_names[i] for i in predictions.tolist()],
    "Detection": [
        "BENIGN" if i == deploy_config["benign_id"]
        else "ATTACK"
        for i in predictions.tolist()
    ],
    "Confidence": confidence.numpy().round(4)
}).head(10))


# 9. Luu phien ban thu vien
environment = "\n".join(
    f"{lib}=={version(lib)}"
    for lib in [
        "torch", "numpy", "scikit-learn",
        "scipy", "pandas", "joblib"
    ]
)

(EXPORT_DIR / "environment.txt").write_text(environment)


# 10. Dong goi ZIP
zip_path = shutil.make_archive(
    str(EXPORT_DIR), "zip", root_dir=EXPORT_DIR
)

print("\n=== DEPLOYMENT PACKAGE ===")

for file in sorted(EXPORT_DIR.iterdir()):
    if file.is_file():
        print(file.name, f"| {file.stat().st_size / 1024:.2f} KB")

print("\nZIP:", zip_path)
print("CELL 36.3 COMPLETED")


Model: CNN-LSTM
Best epoch: 7
Parameters: 34972


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(



=== INFERENCE TEST ===
Model output consistency: PASS
Input: torch.Size([32, 70])
Output: torch.Size([32, 12])


,Actual,Predicted,Detection,Confidence
0,BENIGN,BENIGN,BENIGN,0.9951
1,DDoS,DDoS,ATTACK,0.9941
2,DoS Hulk,BENIGN,BENIGN,0.9965
3,DoS Slowhttptest,DoS Slowhttptest,ATTACK,0.9914
4,BENIGN,BENIGN,BENIGN,0.9794
5,DoS Slowhttptest,DoS Slowhttptest,ATTACK,0.9951
6,Bot,Bot,ATTACK,0.7020
7,DDoS,DDoS,ATTACK,0.9924
8,DoS GoldenEye,DoS GoldenEye,ATTACK,0.9978
9,BENIGN,BENIGN,BENIGN,0.9982



=== DEPLOYMENT PACKAGE ===
best.pt | 144.50 KB
classification_report.csv | 1.11 KB
cnn_lstm.pt | 165.44 KB
config.json | 2.32 KB
confusion_matrix.csv | 0.58 KB
environment.txt | 0.09 KB
gaussian.joblib | 39.25 KB
label_mapping.json | 0.26 KB
model_comparison.csv | 0.56 KB
preprocessor.joblib | 4.60 KB
test_metrics.json | 0.23 KB
training_history.csv | 0.46 KB

ZIP: /content/drive/MyDrive/NIDS_Project/deployment/CIC-IDS2017.zip
CELL 36.3 COMPLETED
